# DM-AHR-ViT — standalone training notebook

**One file. No repository, no scripts, no external Python files.** The project's entire
implementation is embedded in the third code cell and materialised into a temporary package
inside your Drive folder. Every phase below calls the library functions directly; nothing is
run through a subprocess and nothing is cloned from GitHub.

It trains the models and stops. No evaluation, no XAI, no statistics, no report, no external
real-hair validation.

Produced checkpoints (all under `ARTIFACT_ROOT/checkpoints/`):

- `dmahr/gen_iter*.pth` — repaired base diffusion generator, 50,000 iterations
- `dmahr_v2/final_gen.pth` — fine-tuned generator, +20,000 pasted-real-strand iterations
- `vit/<arm>_seed<seed>/best.pt` — ViT-B/16 for arms A, A*, B and Cv2

Full run is many GPU-hours. A100 (Colab Pro) recommended; T4 works but is much slower. Drive
backing makes every phase resumable.


## 1. Config


In [ ]:
FAST_MODE = False
DIFF_BASE_ITERS = 50000
DIFF_FINETUNE_ITERS = 20000
CV2_STEPS = 50
CV2_START_T = 600
CV2_SHARDS = 4
SEEDS_FULL = [42, 123, 2026, 7, 11, 17, 23, 31]
SEEDS_PRE = [42, 123, 2026]
SEEDS = SEEDS_PRE if FAST_MODE else SEEDS_FULL
ARMS = ["A", "Astar", "B", "Cv2"]
print("FAST_MODE", FAST_MODE, "| seeds", SEEDS,
      "| diffusion", DIFF_BASE_ITERS, "+", DIFF_FINETUNE_ITERS)


## 2. Drive-backed paths


In [ ]:
import os, sys, time, json, hashlib, shutil
from pathlib import Path

IN_COLAB = Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/dm_ahr_vit")
else:
    BASE = Path.cwd() / "_dm_ahr_run"
BASE.mkdir(parents=True, exist_ok=True)
DATA_ROOT = BASE / "data"
ARTIFACT_ROOT = BASE / "artifacts"
for p in (DATA_ROOT, ARTIFACT_ROOT):
    p.mkdir(parents=True, exist_ok=True)
os.environ["DATA_ROOT"] = str(DATA_ROOT)
os.environ["ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("BASE:", BASE)


In [ ]:
# The whole implementation, embedded. Written once into BASE/inline and imported directly.
EMBEDDED = {
    "dm_ahr_vit/__init__.py": "\"\"\"DM-AHR-ViT: diffusion hair removal vs. DullRazor for dermoscopic classification.\"\"\"\n\n__version__ = \"0.1.0\"\n\n",
    "dm_ahr_vit/config.py": "\"\"\"Configuration loading and hashing.\n\nConfigs are JSON so the exact bytes can be hashed and committed. Every phase\nrecords the hash of the config it used so results are traceable.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nfrom .paths import REPO_ROOT\nfrom .paths import artifact_root\n\nCLASSES = (\"akiec\", \"bcc\", \"bkl\", \"df\", \"mel\", \"nv\", \"vasc\")\nCLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}\n\n# Expected C0 class counts from the HAM10000 data descriptor. These are only\n# used to compare against recomputed values, never as data.\nEXPECTED_C0_COUNTS = {\n    \"akiec\": 327,\n    \"bcc\": 514,\n    \"bkl\": 1099,\n    \"df\": 115,\n    \"mel\": 1113,\n    \"nv\": 6705,\n    \"vasc\": 142,\n}\nEXPECTED_C0_TOTAL = 10015\n\nDEFAULT_PROTOCOL = \"configs/protocol.json\"\n\n\ndef load_config(path: str | Path) -> dict[str, Any]:\n    path = Path(path)\n    if not path.is_absolute():\n        path = REPO_ROOT / path\n    with path.open(\"rb\") as fh:\n        return json.loads(fh.read().decode(\"utf-8\"))\n\n\ndef config_hash(cfg: dict[str, Any]) -> str:\n    blob = json.dumps(cfg, sort_keys=True, separators=(\",\", \":\")).encode(\"utf-8\")\n    return hashlib.sha256(blob).hexdigest()\n\n\ndef config_hash_file(path: str | Path) -> str:\n    p = Path(path)\n    if not p.is_absolute():\n        p = REPO_ROOT / p\n    return hashlib.sha256(p.read_bytes()).hexdigest()\n\n\ndef save_json(obj: Any, path: str | Path) -> Path:\n    p = Path(path)\n    p.parent.mkdir(parents=True, exist_ok=True)\n    tmp = p.with_suffix(p.suffix + \".tmp\")\n    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True, default=str) + \"\\n\")\n    tmp.replace(p)\n    return p\n\n\ndef frozen_path(name: str) -> Path:\n    return artifact_root() / \"logs\" / \"frozen\" / f\"{name}.json\"\n\n\ndef load_frozen(name: str, default_config: str) -> dict[str, Any]:\n    \"\"\"Prefer a hash-frozen config produced at runtime over the repo default.\"\"\"\n    p = frozen_path(name)\n    if p.exists():\n        return json.loads(p.read_text())\n    return load_config(default_config)\n\n\ndef save_frozen(name: str, cfg: dict[str, Any]) -> Path:\n    return save_json(cfg, frozen_path(name))\n\n\ndef load_hair_screen_config() -> dict[str, Any]:\n    \"\"\"The frozen hair-screening config shipped with the code (`configs/hair_screen.json`).\n\n    Its embedded hash is recomputed so a changed file cannot pass as the frozen one.\n    \"\"\"\n    cfg = load_config(\"configs/hair_screen.json\")\n    body = {k: v for k, v in cfg.items() if k != \"config_hash\"}\n    if config_hash(body) != cfg.get(\"config_hash\"):\n        raise ValueError(\"configs/hair_screen.json does not match its recorded config_hash\")\n    return cfg\n\n\ndef sha256_file(path: str | Path, chunk: int = 1 << 20) -> str:\n    h = hashlib.sha256()\n    with open(path, \"rb\") as fh:\n        for block in iter(lambda: fh.read(chunk), b\"\"):\n            h.update(block)\n    return h.hexdigest()\n\n",
    "dm_ahr_vit/data/__init__.py": "\"\"\"Data package: manifests, splits, datasets, masks, synthetic hair.\"\"\"\n",
    "dm_ahr_vit/data/composite_metrics.py": "\"\"\"Per-item metrics for hair removal on composites with known clean targets.\n\nAll arrays are at the same resolution (the diffusion model's 128 px). `alpha` is the soft\nstrand coverage of the composite, `lesion` a boolean lesion mask. Lesion damage is measured\nonly on lesion pixels the hair did not touch, where any change from the input is damage.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport cv2\nimport numpy as np\n\nCHANGE_LEVELS = 10\nHAIR_ALPHA = 0.02\nMIN_LESION_PIXELS = 50\n\n\ndef item_metrics(out: np.ndarray, target: np.ndarray, comp: np.ndarray, alpha: np.ndarray,\n                 lesion: np.ndarray) -> dict:\n    out, target, comp = (np.asarray(x, np.float64) for x in (out, target, comp))\n    mse = float(np.mean((out - target) ** 2))\n    psnr = float(10.0 * np.log10(255.0**2 / mse)) if mse > 0 else float(\"inf\")\n    w = alpha[..., None].astype(np.float64)\n    denom = float(w.sum() * 3 + 1e-9)\n    e_floor = float((np.abs(comp - target) * w).sum() / denom)\n    e_out = float((np.abs(out - target) * w).sum() / denom)\n    removal = 1.0 - e_out / max(e_floor, 1e-9)\n\n    hair_region = cv2.dilate((alpha > HAIR_ALPHA).astype(np.uint8),\n                             cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))) > 0\n    clear_lesion = lesion & ~hair_region\n    clear_skin = ~lesion & ~hair_region\n    changed = np.abs(out - comp).max(axis=2) > CHANGE_LEVELS\n    err = np.abs(out - target).mean(axis=2)\n    nan = float(\"nan\")\n    return {\n        \"psnr\": psnr,\n        \"hair_removal\": removal,\n        \"lesion_pixels_clear_of_hair\": int(clear_lesion.sum()),\n        \"lesion_mae\": float(err[clear_lesion].mean()) if clear_lesion.sum() >= MIN_LESION_PIXELS else nan,\n        \"lesion_changed\": float(changed[clear_lesion].mean()) if clear_lesion.sum() >= MIN_LESION_PIXELS else nan,\n        \"skin_mae\": float(err[clear_skin].mean()) if clear_skin.sum() >= MIN_LESION_PIXELS else nan,\n        \"skin_changed\": float(changed[clear_skin].mean()) if clear_skin.sum() >= MIN_LESION_PIXELS else nan,\n    }\n\n\ndef bootstrap_mean_ci(x: np.ndarray, reps: int = 2000, seed: int = 0) -> tuple[float, float, float]:\n    x = np.asarray(x, float)\n    x = x[np.isfinite(x)]\n    if x.size == 0:\n        return float(\"nan\"), float(\"nan\"), float(\"nan\")\n    rng = np.random.default_rng(seed)\n    means = x[rng.integers(0, x.size, (reps, x.size))].mean(axis=1)\n    lo, hi = np.percentile(means, [2.5, 97.5])\n    return float(x.mean()), float(lo), float(hi)\n",
    "dm_ahr_vit/data/dataset.py": "\"\"\"Arm datasets over the frozen splits, with the matched ViT recipe.\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\nfrom torchvision import transforms\n\nfrom ..config import CLASS_TO_IDX\n\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\nIMAGENET_STD = (0.229, 0.224, 0.225)\n\n\ndef build_transforms(cfg: dict, train: bool):\n    \"\"\"Eval: square resize + ImageNet normalize. Train: the recipe in vit.json.\"\"\"\n    size = int(cfg[\"image_size\"])\n    aug = cfg.get(\"augmentation\", {})\n    norm = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)\n    if train:\n        return transforms.Compose([\n            transforms.RandomResizedCrop(\n                size,\n                scale=tuple(aug.get(\"random_resized_crop_scale\", (0.75, 1.0))),\n                ratio=tuple(aug.get(\"random_resized_crop_ratio\", (3.0 / 4.0, 4.0 / 3.0))),\n            ),\n            transforms.RandomHorizontalFlip(float(aug.get(\"horizontal_flip\", 0.5))),\n            transforms.RandomVerticalFlip(float(aug.get(\"vertical_flip\", 0.5))),\n            transforms.RandomRotation(float(aug.get(\"rotation_degrees\", 15))),\n            transforms.ColorJitter(\n                *[float(v) for v in aug.get(\"color_jitter\", [0.15, 0.15, 0.1, 0.02])]\n            ),\n            transforms.ToTensor(),\n            norm,\n        ])\n    # Eval: resize the shorter side to int(size * 1.14) = 255, then centre-crop.\n    # This is the pipeline the recorded runs used; squashing to a square or\n    # resizing to 256 changes predictions on these 600x450 images.\n    from ..xai.geometry import eval_ops  # one definition of the model's frame (see xai/geometry)\n\n    return transforms.Compose([*eval_ops(size), transforms.ToTensor(), norm])\n\n\nclass ArmDataset(Dataset):\n    \"\"\"One arm of the study over one split.\n\n    Arm A reads the original ISIC JPEGs; every processed arm (B, C, Astar,\n    Creleased, ...) reads ``processed_root/<arm>/<image_id>.png``.\n    \"\"\"\n\n    def __init__(self, arm: str, split: str, splits_df: pd.DataFrame,\n                 raw_images_dir, processed_root, transform, image_ids=None):\n        df = splits_df[splits_df[\"split\"] == split]\n        if image_ids is not None:\n            df = df[df[\"image_id\"].isin(set(image_ids))]\n        self.records = df.reset_index(drop=True)\n        self.arm = arm\n        self.raw_images_dir = Path(raw_images_dir)\n        self.processed_root = Path(processed_root)\n        self.transform = transform\n\n    def __len__(self) -> int:\n        return len(self.records)\n\n    def _path(self, image_id: str) -> Path:\n        if self.arm == \"A\":\n            return self.raw_images_dir / f\"{image_id}.jpg\"\n        return self.processed_root / self.arm / f\"{image_id}.png\"\n\n    def __getitem__(self, idx: int) -> dict:\n        r = self.records.iloc[idx]\n        with Image.open(self._path(r[\"image_id\"])) as fh:\n            img = fh.convert(\"RGB\")\n            x = self.transform(img)\n        return {\n            \"image\": x,\n            \"label\": int(CLASS_TO_IDX[r[\"dx\"]]),\n            \"image_id\": str(r[\"image_id\"]),\n            \"group_id\": str(r[\"group_id\"]),\n            \"dx\": str(r[\"dx\"]),\n        }\n\n\ndef collate(batch: list) -> dict:\n    return {\n        \"image\": torch.stack([b[\"image\"] for b in batch]),\n        \"label\": torch.tensor([b[\"label\"] for b in batch], dtype=torch.long),\n        \"image_id\": [b[\"image_id\"] for b in batch],\n        \"group_id\": [b[\"group_id\"] for b in batch],\n        \"dx\": [b[\"dx\"] for b in batch],\n    }\n",
    "dm_ahr_vit/data/hair_composite.py": "\"\"\"Training pairs made by compositing REAL hair onto clean dermoscopic images.\n\nA clean-ish target image (very low detector hair coverage) gets real hair strands, harvested\nby `hair/harvest.py` from a different hairy image, pasted on at full resolution. The pair\n(target, composite) is then resized to the diffusion model's native size with the same\nbicubic resize the rest of the pipeline uses. Targets and sources always come from the same\nsplit, and only the train and validation splits are accepted: the test split cannot be used.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\n\nfrom .synthetic_hair import render_hair\n\nCLEAN_MAX_COVERAGE = 0.0167\nHAIR_MIN_COVERAGE = 0.03\n# Layers with very little harvested hair were where review found ink, bubble rims and lesion dots.\nMIN_LAYER_FRACTION = 0.004\nALLOWED_SPLITS = (\"train\", \"val\")\n\n\ndef select_pools(screening: pd.DataFrame, split: str) -> tuple[list[str], list[str]]:\n    \"\"\"Target ids (clean-ish) and candidate source ids (clearly hairy) for one split.\"\"\"\n    if split not in ALLOWED_SPLITS:\n        raise ValueError(f\"split {split!r} is not allowed; only {ALLOWED_SPLITS}\")\n    df = screening[screening[\"split\"] == split]\n    targets = df.loc[df[\"hair_coverage\"] <= CLEAN_MAX_COVERAGE, \"image_id\"]\n    sources = df.loc[df[\"hair_coverage\"] >= HAIR_MIN_COVERAGE, \"image_id\"]\n    return sorted(targets.tolist()), sorted(sources.tolist())\n\n\nclass HairCompositeDataset(Dataset):\n    \"\"\"(composite, clean) pairs; items match `PairedDataset`: HR = clean, SR = composite.\"\"\"\n\n    def __init__(self, split: str, targets: list[str], sources: list[str],\n                 images_dir: Path, layers_dir: Path, size: int = 128,\n                 length: int | None = None, fixed: bool = False, seed: int = 0,\n                 simple_prob: float = 0.3, max_shift: int = 120, max_sources: int = 3,\n                 groups: dict[str, str] | None = None):\n        if split not in ALLOWED_SPLITS:\n            raise ValueError(f\"split {split!r} is not allowed; only {ALLOWED_SPLITS}\")\n        self.split = split\n        self.images_dir = Path(images_dir)\n        self.layers_dir = Path(layers_dir)\n        self.targets = list(targets)\n        self.sources = [s for s in sources if (self.layers_dir / f\"{s}.png\").exists()]\n        if not self.targets or not self.sources:\n            raise ValueError(\"no targets or no harvested sources for this split\")\n        self.size, self.fixed, self.seed = int(size), bool(fixed), int(seed)\n        self.simple_prob, self.max_shift = float(simple_prob), int(max_shift)\n        self.max_sources = max(1, int(max_sources))\n        self.length = int(length) if length else len(self.targets)\n        self.groups = groups or {}\n        self._rng_cache: np.random.Generator | None = None\n\n    def __len__(self) -> int:\n        return self.length\n\n    def _rng(self, idx: int) -> np.random.Generator:\n        if self.fixed:\n            return np.random.default_rng([self.seed, int(idx)])\n        if self._rng_cache is None:\n            self._rng_cache = np.random.default_rng(\n                [self.seed, os.getpid(), int(torch.initial_seed() % (2**31))])\n        return self._rng_cache\n\n    def compose(self, idx: int) -> tuple[np.ndarray, np.ndarray, np.ndarray, dict]:\n        \"\"\"Full-resolution target, composite, soft alpha, and what was used.\"\"\"\n        rng = self._rng(idx)\n        tid = self.targets[int(idx) % len(self.targets)]\n        k = int(rng.integers(1, self.max_sources + 1))\n        sids = [self.sources[int(rng.integers(len(self.sources)))] for _ in range(k)]\n        target = np.asarray(Image.open(self.images_dir / f\"{tid}.jpg\").convert(\"RGB\"))\n        h, w = target.shape[:2]\n        comp = target.astype(np.float32)\n        alpha_tot = np.zeros((h, w), np.float32)\n        for sid in sids:\n            src = np.asarray(Image.open(self.images_dir / f\"{sid}.jpg\").convert(\"RGB\"))\n            mask = cv2.imread(str(self.layers_dir / f\"{sid}.png\"), cv2.IMREAD_GRAYSCALE)\n            if src.shape[:2] != (h, w):\n                src = cv2.resize(src, (w, h), interpolation=cv2.INTER_LINEAR)\n            if mask.shape != (h, w):\n                mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)\n            if rng.random() < 0.5:\n                src, mask = src[:, ::-1], mask[:, ::-1]\n            if rng.random() < 0.5:\n                src, mask = src[::-1], mask[::-1]\n            dy = int(rng.integers(-self.max_shift, self.max_shift + 1))\n            dx = int(rng.integers(-self.max_shift, self.max_shift + 1))\n            src = np.roll(src, (dy, dx), axis=(0, 1))\n            mask = np.roll(mask, (dy, dx), axis=(0, 1))\n            m = (mask > 0).astype(np.uint8)\n            m = cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))\n            alpha = np.clip(cv2.GaussianBlur(m.astype(np.float32), (0, 0), 0.7), 0.0, 1.0)\n            a3 = alpha[..., None]\n            comp = comp * (1 - a3) + src.astype(np.float32) * a3\n            alpha_tot = 1.0 - (1.0 - alpha_tot) * (1.0 - alpha)\n        comp = np.clip(np.rint(comp), 0, 255).astype(np.uint8)\n        simple = False\n        if self.simple_prob > 0 and rng.random() < self.simple_prob:\n            hairy, _, _ = render_hair(Image.fromarray(comp), tid, int(rng.integers(2**31)))\n            comp, simple = np.asarray(hairy), True\n        return target, comp, alpha_tot, {\"target\": tid, \"sources\": sids, \"simple\": simple}\n\n    def _resize(self, arr: np.ndarray) -> torch.Tensor:\n        im = Image.fromarray(arr).resize((self.size, self.size), Image.Resampling.BICUBIC)\n        x = np.asarray(im, dtype=np.float32)\n        return torch.from_numpy(x / 127.5 - 1.0).permute(2, 0, 1)\n\n    def __getitem__(self, idx: int):\n        target, comp, _, _ = self.compose(idx)\n        return {\"HR\": self._resize(target), \"SR\": self._resize(comp)}\n\n    def manifest(self) -> dict:\n        ids = sorted(set(self.targets) | set(self.sources))\n        grp = {self.groups[i] for i in ids if i in self.groups}\n        return {\"n\": int(self.length), \"target_ids\": sorted(self.targets),\n                \"source_ids\": sorted(self.sources), \"splits\": [self.split],\n                \"groups\": sorted(grp)}\n\n\ndef build_datasets(screening_csv: Path, images_dir: Path, layers_dir: Path,\n                   size: int = 128, val_length: int = 192, seed: int = 0,\n                   min_layer_fraction: float = MIN_LAYER_FRACTION):\n    \"\"\"Train and validation composite datasets from the screening table.\"\"\"\n    scr = pd.read_csv(screening_csv)\n    layer_stats = pd.read_csv(Path(layers_dir) / \"summary.csv\")\n    usable = set(layer_stats.loc[layer_stats[\"fraction\"] >= min_layer_fraction, \"image_id\"])\n    groups = dict(zip(scr[\"image_id\"], scr[\"group_id\"]))\n    tr_t, tr_s = select_pools(scr, \"train\")\n    va_t, va_s = select_pools(scr, \"val\")\n    tr_s, va_s = [i for i in tr_s if i in usable], [i for i in va_s if i in usable]\n    train = HairCompositeDataset(\"train\", tr_t, tr_s, images_dir, layers_dir, size,\n                                 fixed=False, seed=seed, groups=groups)\n    val = HairCompositeDataset(\"val\", va_t, va_s, images_dir, layers_dir, size,\n                               length=min(val_length, len(va_t)), fixed=True,\n                               seed=seed + 1, groups=groups)\n    return train, val\n",
    "dm_ahr_vit/data/ham10000.py": "\"\"\"HAM10000 metadata (Harvard Dataverse) and cohort helpers.\"\"\"\n\nfrom __future__ import annotations\n\nimport zipfile\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom ..config import CLASSES, EXPECTED_C0_COUNTS, EXPECTED_C0_TOTAL\n\nDATAVERSE_DOI = \"10.7910/DVN/DBW86T\"\nDATAVERSE_ACCESS = \"https://dataverse.harvard.edu/api/access/datafile/{fid}\"\nMETADATA_FILE_ID = 3282766\nMASKS_FILE_ID = 3282767\n\n\ndef read_metadata(path) -> pd.DataFrame:\n    df = pd.read_csv(path, sep=\"\\t\")\n    df[\"image_id\"] = df[\"image_id\"].astype(str)\n    df[\"lesion_id\"] = df[\"lesion_id\"].astype(str)\n    return df\n\n\ndef class_counts(df: pd.DataFrame) -> dict:\n    counts = df[\"dx\"].value_counts().to_dict()\n    return {c: int(counts.get(c, 0)) for c in CLASSES}\n\n\ndef expected_comparison(metadata: pd.DataFrame) -> dict:\n    counts = class_counts(metadata)\n    return {\n        \"recomputed\": counts,\n        \"expected\": EXPECTED_C0_COUNTS,\n        \"total\": int(sum(counts.values())),\n        \"expected_total\": EXPECTED_C0_TOTAL,\n        \"matches\": all(counts[c] == EXPECTED_C0_COUNTS[c] for c in CLASSES),\n    }\n\n\ndef extract_masks(zip_path: Path, out_dir: Path) -> dict:\n    out_dir.mkdir(parents=True, exist_ok=True)\n    n = 0\n    with zipfile.ZipFile(zip_path) as zf:\n        for name in zf.namelist():\n            if name.endswith(\"/\"):\n                continue\n            target = out_dir / Path(name).name\n            if not target.exists():\n                target.write_bytes(zf.read(name))\n            n += 1\n    return {\"extracted\": n, \"out_dir\": str(out_dir)}\n",
    "dm_ahr_vit/data/isic.py": "\"\"\"ISIC Archive collection download.\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport requests\n\n\ndef download_collection(collection_id: int, images_dir: Path, records_path: Path,\n                        workers: int = 16, limit: int | None = None) -> dict:\n    \"\"\"Download one ISIC collection, recording per-file status to records.jsonl.\"\"\"\n    images_dir.mkdir(parents=True, exist_ok=True)\n    records_path.parent.mkdir(parents=True, exist_ok=True)\n    api = f\"https://api.isic-archive.com/api/v2/images?collection_id={collection_id}\"\n    images, page = [], 0\n    while True:\n        r = requests.get(api, params={\"limit\": 100, \"offset\": page * 100}, timeout=60)\n        r.raise_for_status()\n        batch = r.json().get(\"results\", [])\n        if not batch:\n            break\n        images.extend(batch)\n        page += 1\n        if limit and len(images) >= limit:\n            images = images[:limit]\n            break\n\n    def fetch(img):\n        iid = img[\"isic_id\"]\n        out = images_dir / f\"{iid}.jpg\"\n        url = f\"https://isic-archive.s3.amazonaws.com/images/{iid}.jpg\"\n        rec = {\"isic_id\": iid, \"url\": url, \"bytes\": None, \"sha256\": None,\n               \"status\": \"failed\", \"attribution\": img.get(\"attribution\"),\n               \"license\": img.get(\"license\"),\n               \"expected_size\": img.get(\"metadata\", {}).get(\"acquisition\", {}).get(\"pixels\")}\n        try:\n            if out.exists():\n                blob = out.read_bytes()\n            else:\n                resp = requests.get(url, timeout=120)\n                resp.raise_for_status()\n                blob = resp.content\n                out.write_bytes(blob)\n            rec.update({\"bytes\": len(blob), \"sha256\": hashlib.sha256(blob).hexdigest(),\n                        \"status\": \"downloaded\"})\n        except Exception as exc:\n            rec[\"status\"] = f\"failed: {exc}\"\n        return rec\n\n    with ThreadPoolExecutor(max_workers=workers) as ex:\n        records = list(ex.map(fetch, images))\n    with records_path.open(\"w\") as fh:\n        for rec in records:\n            fh.write(json.dumps(rec) + \"\\n\")\n    return {\"collection\": collection_id, \"n\": len(records),\n            \"downloaded\": sum(r[\"status\"] == \"downloaded\" for r in records)}\n",
    "dm_ahr_vit/data/manifest.py": "\"\"\"File manifests: decode audit, dHash near-duplicate candidates, confirmation.\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\n\n\ndef _describe(path: Path, image_id: str) -> dict:\n    row = {\n        \"image_id\": image_id, \"decode_status\": \"fail_missing\",\n        \"width\": None, \"height\": None, \"mode\": None, \"format\": None,\n        \"sha256\": None, \"bytes\": None,\n    }\n    if not path.exists():\n        return row\n    row[\"bytes\"] = path.stat().st_size\n    try:\n        with Image.open(path) as fh:\n            fh.load()\n            w, h = fh.size\n            row.update({\"decode_status\": \"ok\", \"width\": w, \"height\": h,\n                        \"mode\": fh.mode, \"format\": fh.format})\n    except Exception:\n        row[\"decode_status\"] = \"fail_decode\"\n        return row\n    row[\"sha256\"] = hashlib.sha256(path.read_bytes()).hexdigest()\n    return row\n\n\ndef dhash(img: Image.Image, size: int = 8) -> int:\n    \"\"\"64-bit difference hash on a (size+1)x(size) grayscale grid.\"\"\"\n    g = np.asarray(img.convert(\"L\").resize((size + 1, size), Image.Resampling.BILINEAR),\n                   dtype=np.int16)\n    bits = g[:, 1:] > g[:, :-1]\n    v = 0\n    for b in bits.ravel():\n        v = (v << 1) | int(b)\n    return v\n\n\ndef hamming_pairs(hashes: np.ndarray, threshold: int, ids: list) -> list:\n    n = len(hashes)\n    out = []\n    for i in range(n):\n        for j in range(i + 1, n):\n            d = bin(int(hashes[i]) ^ int(hashes[j])).count(\"1\")\n            if d <= threshold:\n                out.append({\"id_a\": ids[i], \"id_b\": ids[j], \"distance\": d})\n    return out\n\n\ndef confirm_near_duplicates(candidates: pd.DataFrame, man: pd.DataFrame,\n                            max_mean_abs_diff: float = 4.0,\n                            min_correlation: float = 0.985) -> pd.DataFrame:\n    \"\"\"Pixel-level confirmation of dHash candidates: mean abs diff and correlation.\"\"\"\n    import cv2\n\n    paths = dict(zip(man[\"image_id\"], man[\"path\"]))\n    rows = []\n    for r in candidates.itertuples(index=False):\n        try:\n            a = cv2.imread(str(Path(paths[r.id_a])), cv2.IMREAD_GRAYSCALE)\n            b = cv2.imread(str(Path(paths[r.id_b])), cv2.IMREAD_GRAYSCALE)\n            if a is None or b is None or a.shape != b.shape:\n                rows.append({\"id_a\": r.id_a, \"id_b\": r.id_b, \"dhash_distance\": r.distance,\n                             \"mean_abs_diff\": float(\"inf\"), \"correlation\": 0.0,\n                             \"confirmed\": False})\n                continue\n            mad = float(np.abs(a.astype(np.float64) - b.astype(np.float64)).mean())\n            va, vb = a.astype(np.float64).ravel(), b.astype(np.float64).ravel()\n            corr = float(np.corrcoef(va, vb)[0, 1]) if va.std() > 0 and vb.std() > 0 else 0.0\n            rows.append({\"id_a\": r.id_a, \"id_b\": r.id_b, \"dhash_distance\": r.distance,\n                         \"mean_abs_diff\": mad, \"correlation\": corr,\n                         \"confirmed\": bool(mad <= max_mean_abs_diff and corr >= min_correlation)})\n        except Exception:\n            rows.append({\"id_a\": r.id_a, \"id_b\": r.id_b, \"dhash_distance\": r.distance,\n                         \"mean_abs_diff\": float(\"inf\"), \"correlation\": 0.0, \"confirmed\": False})\n    return pd.DataFrame(rows)\n\n\ndef build_manifest(records_path: Path, images_dir: Path, metadata: pd.DataFrame,\n                   out_dir: Path):\n    \"\"\"Per-image manifest from the download records + HAM metadata.\"\"\"\n    records = [json.loads(l) for l in Path(records_path).read_text().splitlines() if l.strip()]\n    meta = metadata.set_index(\"image_id\")\n    rows, excl = [], []\n    for rec in records:\n        iid = rec[\"isic_id\"]\n        path = images_dir / f\"{iid}.jpg\"\n        d = _describe(path, iid)\n        d.update({\n            \"path\": str(path),\n            \"in_c0\": iid in meta.index,\n            \"cohort\": \"C0\" if iid in meta.index else \"C0-extra\",\n            \"license\": rec.get(\"license\"),\n            \"attribution\": rec.get(\"attribution\"),\n            \"expected_bytes\": rec.get(\"expected_size\"),\n            \"download_status\": rec.get(\"status\"),\n        })\n        if iid in meta.index:\n            d[\"dx\"] = str(meta.loc[iid, \"dx\"])\n            d[\"lesion_id\"] = str(meta.loc[iid, \"lesion_id\"])\n        else:\n            d[\"dx\"] = None\n            d[\"lesion_id\"] = None\n            excl.append({\"image_id\": iid, \"cohort\": \"C0-extra\",\n                         \"reason\": \"not in HAM10000 metadata\", \"path\": str(path)})\n        if d[\"decode_status\"] != \"ok\":\n            excl.append({\"image_id\": iid, \"cohort\": d[\"cohort\"],\n                         \"reason\": d[\"decode_status\"], \"path\": str(path)})\n        if d[\"decode_status\"] == \"ok\":\n            with Image.open(path) as fh:\n                d[\"dhash\"] = dhash(fh)\n        else:\n            d[\"dhash\"] = None\n        rows.append(d)\n    df = pd.DataFrame(rows)\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    df.to_csv(out_dir / \"data_manifest.csv\", index=False)\n    pd.DataFrame(excl).to_csv(out_dir / \"exclusions.csv\", index=False)\n    return df, pd.DataFrame(excl), {\"n\": len(df)}\n",
    "dm_ahr_vit/data/masks.py": "\"\"\"Lesion-mask loading (X1 tier) and the mask manifest builder.\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\n\nfrom ..paths import data_root\n\nX1_PROVENANCE = (\n    \"HAM10000_segmentations_lesion_tschandl (Harvard Dataverse doi:10.7910/DVN/DBW86T)\"\n)\n\n\ndef resolve(path) -> Path:\n    \"\"\"Re-root container-recorded ``/__modal/...`` paths onto the local data root.\"\"\"\n    p = Path(path)\n    if p.exists():\n        return p\n    marker = \"/data/\"\n    if marker in str(p):\n        return data_root() / str(p).split(marker, 1)[1]\n    return p\n\n\ndef load_mask_224(path, size: int = 224) -> np.ndarray:\n    \"\"\"Grayscale mask -> binary {0,1} array resized with NEAREST.\"\"\"\n    with Image.open(resolve(path)) as fh:\n        m = fh.convert(\"L\").resize((size, size), Image.Resampling.NEAREST)\n    return (np.array(m) > 127).astype(np.uint8)\n\n\ndef load_mask_registered(path, image_size_wh=None, size: int = 224) -> np.ndarray:\n    \"\"\"Lesion mask in the model's frame; delegates to `xai.geometry.mask_to_model_frame`.\"\"\"\n    from ..xai.geometry import mask_to_model_frame\n\n    with Image.open(resolve(path)) as fh:\n        return mask_to_model_frame(fh, image_size_wh, size)\n\n\ndef build_x1_manifest(masks_dir: Path, split_df: pd.DataFrame, man: pd.DataFrame,\n                      out_path: Path) -> pd.DataFrame:\n    \"\"\"One row per split image: mask presence, geometry checks, foreground fraction.\"\"\"\n    sizes = dict(zip(man[\"image_id\"], zip(man[\"width\"], man[\"height\"])))\n    dx = dict(zip(split_df[\"image_id\"], split_df[\"dx\"]))\n    group = dict(zip(split_df[\"image_id\"], split_df[\"group_id\"]))\n    split = dict(zip(split_df[\"image_id\"], split_df[\"split\"]))\n    rows = []\n    for image_id in split_df[\"image_id\"]:\n        p = masks_dir / f\"{image_id}_segmentation.png\"\n        row = {\n            \"image_id\": image_id,\n            \"tier\": \"X1\",\n            \"split\": split[image_id],\n            \"dx\": dx[image_id],\n            \"group_id\": group[image_id],\n            \"has_mask\": bool(p.exists()),\n            \"mask_path\": str(p),\n            \"provenance\": X1_PROVENANCE,\n            \"expert_traced_claimed\": False,\n        }\n        status = \"ok\"\n        if p.exists():\n            with Image.open(p) as fh:\n                arr = np.array(fh.convert(\"L\"))\n            w, h = arr.shape[1], arr.shape[0]\n            binary = set(np.unique(arr)) <= {0, 255}\n            non_empty = bool((arr > 127).any())\n            fg = float((arr > 127).mean())\n            iw, ih = sizes.get(image_id, (None, None))\n            size_matches = bool(iw == w and ih == h)\n            row.update({\n                \"width\": w, \"height\": h, \"binary\": bool(binary),\n                \"non_empty\": non_empty, \"foreground_fraction\": fg,\n                \"size_matches\": size_matches,\n            })\n            if not (binary and non_empty and size_matches):\n                status = \"degraded\"\n        else:\n            row.update({\"width\": None, \"height\": None, \"binary\": None,\n                        \"non_empty\": None, \"foreground_fraction\": None,\n                        \"size_matches\": None})\n            status = \"missing_mask\"\n        row[\"status\"] = status\n        rows.append(row)\n    df = pd.DataFrame(rows)\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    df.to_csv(out_path, index=False)\n    return df\n",
    "dm_ahr_vit/data/splits.py": "\"\"\"Lesion-grouped splits: union of lesion_id and confirmed near-duplicates.\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom ..config import CLASSES\n\n\nclass UnionFind:\n    def __init__(self):\n        self.parent: dict = {}\n\n    def find(self, x):\n        if x not in self.parent:\n            self.parent[x] = x\n            return x\n        root = x\n        while self.parent[root] != root:\n            root = self.parent[root]\n        while self.parent[x] != root:  # path compression\n            self.parent[x], x = root, self.parent[x]\n        return root\n\n    def union(self, a, b):\n        ra, rb = self.find(a), self.find(b)\n        if ra != rb:\n            self.parent[rb] = ra\n\n\ndef build_groups(c0: pd.DataFrame, near: pd.DataFrame) -> pd.DataFrame:\n    \"\"\"Merge images sharing a lesion_id or a confirmed near-duplicate link.\"\"\"\n    uf = UnionFind()\n    for lesion in c0[\"lesion_id\"].unique():\n        members = c0.loc[c0[\"lesion_id\"] == lesion, \"image_id\"].tolist()\n        for m in members[1:]:\n            uf.union(members[0], m)\n    if len(near):\n        for r in near.itertuples(index=False):\n            uf.union(r.id_a, r.id_b)\n    df = c0.copy()\n    first_seen: dict = {}\n    group_ids = []\n    for iid in df[\"image_id\"]:\n        root = uf.find(iid)\n        if root not in first_seen:\n            first_seen[root] = f\"G{len(first_seen):05d}\"\n        group_ids.append(first_seen[root])\n    df[\"group_id\"] = group_ids\n    return df\n\n\ndef freeze_splits(grouped: pd.DataFrame, seed: int, fractions: dict,\n                  out_dir, protocol_hash: str) -> dict:\n    \"\"\"Stratified group split, frozen to CSV plus its SHA-256.\"\"\"\n    rng = np.random.default_rng(seed)\n    order = [\"train\", \"val\", \"test\"]\n    fr = [fractions[s] for s in order]\n    assign: dict = {}\n    for dx in CLASSES:\n        groups = sorted(grouped.loc[grouped[\"dx\"] == dx, \"group_id\"].unique())\n        rng.shuffle(groups)\n        n = len(groups)\n        n_tr = int(round(fr[0] * n))\n        n_va = int(round(fr[1] * n))\n        for g in groups[:n_tr]:\n            assign[g] = \"train\"\n        for g in groups[n_tr:n_tr + n_va]:\n            assign[g] = \"val\"\n        for g in groups[n_tr + n_va:]:\n            assign[g] = \"test\"\n    df = grouped.copy()\n    df[\"split\"] = df[\"group_id\"].map(assign)\n    df = df[[\"image_id\", \"lesion_id\", \"group_id\", \"dx\", \"split\", \"sha256\"]]\n    df = df.sort_values(\"image_id\").reset_index(drop=True)\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    df.to_csv(out_dir / \"splits.csv\", index=False)\n    digest = hashlib.sha256((out_dir / \"splits.csv\").read_bytes()).hexdigest()\n    (out_dir / \"splits.sha256\").write_text(digest + \"\\n\")\n    counts = {s: df[df[\"split\"] == s][\"dx\"].value_counts().to_dict() for s in order}\n    return {\n        \"counts\": counts,\n        \"n_groups\": int(df[\"group_id\"].nunique()),\n        \"splits_sha256\": digest,\n        \"protocol_hash\": protocol_hash,\n    }\n\n\ndef verify_splits(df: pd.DataFrame) -> dict:\n    \"\"\"No group may appear in more than one split.\"\"\"\n    problems = []\n    splits = [\"train\", \"val\", \"test\"]\n    for i, a in enumerate(splits):\n        for b in splits[i + 1:]:\n            ga = set(df.loc[df[\"split\"] == a, \"group_id\"])\n            gb = set(df.loc[df[\"split\"] == b, \"group_id\"])\n            if ga & gb:\n                problems.append(f\"group overlap between {a} and {b}: {len(ga & gb)} groups\")\n    if df[\"image_id\"].duplicated().any():\n        problems.append(\"duplicate image_id in split frame\")\n    sizes = {s: int((df[\"split\"] == s).sum()) for s in splits}\n    return {\"ok\": not problems, \"problems\": problems, \"sizes\": sizes}\n",
    "dm_ahr_vit/data/synthetic_hair.py": "\"\"\"Seeded synthetic hair rendering for paired hair-removal evaluation.\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\n\nimport numpy as np\nfrom PIL import Image\n\n\ndef _rng(image_id: str, seed: int) -> np.random.Generator:\n    h = hashlib.sha256(f\"{image_id}:{seed}\".encode()).digest()\n    return np.random.default_rng(int.from_bytes(h[:8], \"little\"))\n\n\ndef render_hair(img: Image.Image, image_id: str, seed: int = 0):\n    \"\"\"Draw random dark curved hairs; return (hairy image, binary mask, params).\"\"\"\n    rng = _rng(image_id, seed)\n    w, h = img.size\n    n_hairs = int(rng.integers(8, 20))\n    blur_sigma = float(rng.uniform(0.5, 1.5))\n    mask = np.zeros((h, w), dtype=np.uint8)\n    for _ in range(n_hairs):\n        x0, y0 = rng.uniform(0, w), rng.uniform(0, h)\n        angle = rng.uniform(0, 2 * np.pi)\n        length = rng.uniform(0.4, 1.2) * max(w, h)\n        curve = rng.uniform(-0.6, 0.6)\n        thickness = int(rng.integers(1, 4))\n        steps = 48\n        xs = np.linspace(0, length, steps)\n        px = x0 + xs * np.cos(angle)\n        py = y0 + xs * np.sin(angle)\n        perp = (np.linspace(0, 1, steps) ** 2) * curve * 40.0\n        px = px + perp * np.sin(angle + np.pi / 2)\n        py = py + perp * np.cos(angle + np.pi / 2)\n        for x, y in zip(px, py):\n            xi, yi = int(round(x)), int(round(y))\n            if 0 <= xi < w and 0 <= yi < h:\n                mask[max(0, yi - thickness // 2):yi + thickness // 2 + 1,\n                     max(0, xi - thickness // 2):xi + thickness // 2 + 1] = 255\n    arr = np.asarray(img.convert(\"RGB\"), dtype=np.float64)\n    m = mask > 0\n    k = max(1, int(round(blur_sigma)))\n    if k > 1:\n        from scipy.ndimage import uniform_filter\n        soft = uniform_filter(mask.astype(np.float64), size=k)\n        m = soft > 0.05\n    darkness = rng.uniform(0.25, 0.55)\n    out = arr.copy()\n    out[m] = arr[m] * darkness\n    hairy = Image.fromarray(np.clip(out, 0, 255).astype(np.uint8))\n    return hairy, Image.fromarray(np.where(mask > 0, 255, 0).astype(np.uint8)), {\n        \"n_hairs\": n_hairs, \"blur_sigma\": blur_sigma, \"seed\": seed, \"source\": \"synthetic\",\n    }\n",
    "dm_ahr_vit/diffusion/__init__.py": "\"\"\"DM-AHR integration (pinned upstream SR3 implementation).\"\"\"\n\n",
    "dm_ahr_vit/diffusion/dmahr_adapter.py": "\"\"\"Adapter around the pinned upstream DM-AHR/SR3 implementation.\n\nThe upstream tree is vendored in `third_party/dmahr` at a pinned commit so the\naudit and the executable code cannot drift apart.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport re\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom PIL import Image\n\nfrom ..config import config_hash\nfrom ..paths import REPO_ROOT\n\nUPSTREAM_DIR = REPO_ROOT / \"third_party\" / \"dmahr\"\nCHECKPOINT_NAME = \"I725000_E1083_gen.pth\"\n\n\ndef _ensure_path() -> None:\n    p = str(UPSTREAM_DIR)\n    if p not in sys.path:\n        sys.path.insert(0, p)\n\n\ndef _strip_comments(text: str) -> str:\n    \"\"\"Make the upstream config JSON-parseable.\n\n    `config/*.json` in the pinned repo is JSON5: it has `//` and `/* */`\n    comments plus trailing commas, so plain `json.load` rejects it.\n    \"\"\"\n    out: list[str] = []\n    i, n = 0, len(text)\n    in_string = False\n    escaped = False\n    while i < n:\n        ch = text[i]\n        if in_string:\n            out.append(ch)\n            if escaped:\n                escaped = False\n            elif ch == \"\\\\\":\n                escaped = True\n            elif ch == '\"':\n                in_string = False\n            i += 1\n            continue\n        if ch == '\"':\n            in_string = True\n            out.append(ch)\n            i += 1\n            continue\n        if ch == \"/\" and i + 1 < n and text[i + 1] == \"/\":\n            j = text.find(\"\\n\", i)\n            i = n if j < 0 else j\n            continue\n        if ch == \"/\" and i + 1 < n and text[i + 1] == \"*\":\n            j = text.find(\"*/\", i + 2)\n            i = n if j < 0 else j + 2\n            continue\n        out.append(ch)\n        i += 1\n    return re.sub(r\",(\\s*[}\\]])\", r\"\\1\", \"\".join(out))\n\n\ndef load_upstream_config(name: str = \"akiec.json\") -> dict:\n    raw = (UPSTREAM_DIR / \"config\" / name).read_text()\n    return json.loads(_strip_comments(raw))\n\n\ndef build_opt(cfg: dict, upstream: dict | None = None) -> dict:\n    upstream = upstream or load_upstream_config()\n    keep = upstream.get(\"model\", {})\n    opt = {\n        \"name\": \"dmahr\",\n        \"phase\": \"val\",\n        \"gpu_ids\": [],\n        \"path\": {\"log\": \"/tmp\", \"tb_logger\": \"/tmp\", \"results\": \"/tmp\",\n                 \"checkpoint\": \"/tmp\", \"resume_state\": None},\n        \"datasets\": {},\n        \"model\": {\n            \"which_model_G\": keep.get(\"which_model_G\", \"sr3\"),\n            \"finetune_norm\": False,\n            \"unet\": dict(cfg[\"unet\"]),\n            \"beta_schedule\": {\n                \"train\": {**cfg[\"beta_schedule\"], \"n_timestep\": int(cfg[\"num_timesteps\"])},\n                \"val\": {**cfg[\"beta_schedule\"], \"n_timestep\": int(cfg[\"num_timesteps\"])},\n            },\n            \"diffusion\": {\n                \"image_size\": int(cfg[\"native_resolution\"]),\n                \"channels\": 3,\n                \"conditional\": True,\n            },\n        },\n        \"train\": {\n            \"n_iter\": int(cfg[\"train\"][\"iterations\"]),\n            \"val_freq\": 1e9,\n            \"save_checkpoint_freq\": 1e9,\n            \"print_freq\": 100,\n            \"optimizer\": {\"type\": cfg[\"train\"][\"optimizer\"], \"lr\": float(cfg[\"train\"][\"lr\"])},\n            \"ema_scheduler\": {},\n        },\n        \"wandb\": {},\n        \"distributed\": False,\n    }\n    return opt\n\n\ndef opt_hash(opt: dict) -> str:\n    return config_hash(opt)\n\n\ndef load_model(cfg: dict, checkpoint: Path, device: str = \"cuda\") -> torch.nn.Module:\n    _ensure_path()\n    import model.networks as networks  # noqa: PLC0415\n\n    opt = build_opt(cfg)\n    net = networks.define_G(opt)\n    # The upstream __init__ deliberately skips buffer registration (the call is\n    # commented out), so the diffusion buffers only exist after an explicit\n    # schedule call. Register them first, then load so strict=True can match.\n    net.set_new_noise_schedule(\n        {\"schedule\": cfg[\"beta_schedule\"][\"schedule\"],\n         \"linear_start\": cfg[\"beta_schedule\"][\"linear_start\"],\n         \"linear_end\": cfg[\"beta_schedule\"][\"linear_end\"],\n         \"n_timestep\": int(cfg[\"num_timesteps\"])},\n        device=torch.device(\"cpu\"),\n    )\n    state = torch.load(checkpoint, map_location=\"cpu\", weights_only=True)\n    if any(k.startswith(\"module.\") for k in state):\n        state = {k.replace(\"module.\", \"\", 1): v for k, v in state.items()}\n    net.load_state_dict(state, strict=True)\n    net.eval()\n    net.to(device)\n    net.num_timesteps = int(cfg[\"num_timesteps\"])\n    net.set_new_noise_schedule(\n        {\"schedule\": cfg[\"beta_schedule\"][\"schedule\"],\n         \"linear_start\": cfg[\"beta_schedule\"][\"linear_start\"],\n         \"linear_end\": cfg[\"beta_schedule\"][\"linear_end\"],\n         \"n_timestep\": int(cfg[\"num_timesteps\"])},\n        device=torch.device(device),\n    )\n    print(f\"loaded DM-AHR checkpoint {checkpoint} (strict)\", flush=True)\n    return net\n\n\ndef preprocess(images: list[Image.Image], size: int = 128) -> torch.Tensor:\n    arrs = []\n    for im in images:\n        a = np.asarray(im.convert(\"RGB\").resize((size, size), Image.Resampling.BICUBIC),\n                       dtype=np.float32)\n        arrs.append((a / 127.5) - 1.0)\n    return torch.from_numpy(np.stack(arrs)).permute(0, 3, 1, 2).contiguous()\n\n\ndef postprocess(tensor: torch.Tensor) -> list[Image.Image]:\n    arr = tensor.detach().float().cpu().numpy()\n    if arr.ndim == 3:\n        arr = arr[None]\n    arr = np.clip((arr + 1.0) * 127.5, 0, 255).round().astype(np.uint8)\n    return [Image.fromarray(a.transpose(1, 2, 0)) for a in arr]\n\n\n@torch.no_grad()\ndef clean_batch(\n    net: torch.nn.Module,\n    images: list[Image.Image],\n    cfg: dict,\n    device: str = \"cuda\",\n    seed: int | None = None,\n    num_timesteps: int | None = None,\n    use_bf16: bool = False,\n    respaced: bool = False,\n    schedule: str | None = None,\n    start_t: int | None = None,\n) -> tuple[list[Image.Image], dict]:\n    \"\"\"Run DM-AHR on a batch of images. Returns cleaned images and run stats.\"\"\"\n    size = int(cfg[\"native_resolution\"])\n    x = preprocess(images, size).to(device)\n    if respaced:\n        # Few-step DDIM over the trained 2,000-step schedule. Truncating the\n        # schedule instead (num_timesteps=25) barely adds noise and produces\n        # garbage; striding the trained schedule is what makes 25 steps work.\n        out, stats = respaced_super_resolution(\n            net, x, steps=int(num_timesteps or 25), seed=seed, use_bf16=use_bf16,\n            start_t=start_t,\n        )\n        stats.update({\"respaced\": True, \"num_timesteps\": int(net.num_timesteps),\n                      \"steps\": int(num_timesteps or 25), \"start_t\": start_t,\n                      \"bf16\": bool(use_bf16)})\n        return postprocess(out), stats\n    if num_timesteps is not None and int(num_timesteps) != int(net.num_timesteps):\n        net.num_timesteps = int(num_timesteps)\n        net.set_new_noise_schedule(\n            {\"schedule\": schedule or cfg[\"beta_schedule\"][\"schedule\"],\n             \"linear_start\": cfg[\"beta_schedule\"][\"linear_start\"],\n             \"linear_end\": cfg[\"beta_schedule\"][\"linear_end\"],\n             \"n_timestep\": int(num_timesteps)},\n            device=torch.device(device),\n        )\n    if seed is not None:\n        torch.manual_seed(int(seed))\n        torch.cuda.manual_seed_all(int(seed))\n    # Upstream's `continous=False` path returns `ret_img[-1]`, which silently\n    # collapses a batch to its last image. Ask for the full trajectory and take\n    # the final B rows, which is the last denoised set for the whole batch.\n    # bf16 autocast roughly halves the cost of the 2,000 sequential UNet calls\n    # on Ampere tensor cores; the sampler is compute bound, not memory bound.\n    with torch.autocast(device_type=\"cuda\", dtype=torch.bfloat16, enabled=use_bf16):\n        trajectory = net.super_resolution(x, continous=True)\n    out = trajectory[-x.shape[0]:]\n    finite = bool(torch.isfinite(out).all())\n    stats = {\n        \"num_timesteps\": int(net.num_timesteps),\n        \"finite\": finite,\n        \"shape\": list(out.shape),\n        \"min\": float(out.min()),\n        \"max\": float(out.max()),\n        \"seed\": seed,\n        \"bf16\": use_bf16,\n        \"batch_returned\": int(out.shape[0]),\n        \"batch_requested\": int(x.shape[0]),\n    }\n    if not finite:\n        raise FloatingPointError(\"DM-AHR produced non-finite values\")\n    if out.shape[0] != x.shape[0]:\n        raise RuntimeError(\"DM-AHR returned the wrong number of images\")\n    return postprocess(out), stats\n\n\n@torch.no_grad()\ndef respaced_super_resolution(net, cond: torch.Tensor, steps: int = 25,\n                              seed: int | None = None,\n                              use_bf16: bool = False,\n                              start_t: int | None = None) -> tuple[torch.Tensor, dict]:\n    \"\"\"Deterministic DDIM sampling over the trained schedule in `steps` jumps.\n\n    Uses the model's own 2,000-step alphas and visits `steps` of them evenly\n    spaced, so every denoiser call is made at a noise level the model was\n    trained on. This is what makes 25-step sampling usable.\n\n    `start_t` starts the chain at that schedule index from the noised\n    conditioning image instead of from pure noise. Hair is thin, so a chain\n    that begins at a middling noise level keeps the global tone of the input and\n    never visits the high-noise levels where a briefly trained model is weak.\n    \"\"\"\n    if seed is not None:\n        torch.manual_seed(int(seed))\n        torch.cuda.manual_seed_all(int(seed))\n    T = int(net.num_timesteps)\n    top = T - 1 if start_t is None else int(min(max(start_t, 1), T - 1))\n    seq = np.linspace(top, 0, steps).round().astype(int)\n    seq = np.unique(seq)[::-1]  # descending, no duplicates\n    noise = torch.randn(cond.shape, device=cond.device)\n    if start_t is None:\n        x = noise\n    else:\n        a = net.alphas_cumprod[top]\n        x = a.sqrt() * cond + (1.0 - a).sqrt() * noise\n    with torch.autocast(device_type=\"cuda\", dtype=torch.bfloat16, enabled=use_bf16):\n        for i, t in enumerate(seq):\n            level = torch.FloatTensor(\n                [net.sqrt_alphas_cumprod_prev[t + 1]]).repeat(cond.shape[0], 1).to(cond.device)\n            eps = net.denoise_fn(torch.cat([cond, x], dim=1), level)\n            x0 = (net.sqrt_recip_alphas_cumprod[t] * x\n                  - net.sqrt_recipm1_alphas_cumprod[t] * eps).clamp(-1.0, 1.0)\n            if i == len(seq) - 1:\n                x = x0\n                break\n            t_next = int(seq[i + 1])\n            ac_next = net.alphas_cumprod[t_next]\n            # Deterministic DDIM step: re-noise x0 to the next scheduled level.\n            x = ac_next.sqrt() * x0 + (1.0 - ac_next).sqrt() * eps\n    stats = {\"finite\": bool(torch.isfinite(x).all()),\n             \"shape\": list(x.shape), \"timesteps_visited\": seq.tolist()}\n    if not stats[\"finite\"]:\n        raise FloatingPointError(\"DM-AHR produced non-finite values\")\n    return x, stats\n\n",
    "dm_ahr_vit/diffusion/train.py": "\"\"\"Leakage-controlled DM-AHR training on train-split paired sources only.\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport hashlib\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom ..config import save_json\nfrom ..paths import data_root\nfrom ..utils import setup_logging\nfrom .dmahr_adapter import _ensure_path, build_opt, opt_hash\n\nlog = setup_logging()\n\n\ndef _resolve_manifest_path(path: str) -> Path:\n    \"\"\"Re-root a recorded modal-volume path onto the current DATA_ROOT.\n\n    Pair manifests were written inside a container, so they hold absolute\n    `/__modal/volumes/<id>/data/...` paths that are void on any other volume.\n    Splitting on the `/data/` marker keeps them usable.\n    \"\"\"\n    p = Path(path)\n    if p.exists():\n        return p\n    marker = \"/data/\"\n    if marker in str(p):\n        return data_root() / str(p).split(marker, 1)[1]\n    return p\n\n\ndef _manifest_hash(pairs: pd.DataFrame) -> str:\n    \"\"\"SHA-256 over the ordered pair ids, so a run records exactly what it saw.\"\"\"\n    ids = sorted(str(i) for i in pairs[\"pair_id\"])\n    return hashlib.sha256(\"\\n\".join(ids).encode()).hexdigest()\n\n\nclass PairedDataset(Dataset):\n    \"\"\"(hairy, hairless) pairs at native resolution, in [-1, 1].\"\"\"\n\n    def __init__(self, pairs: pd.DataFrame, size: int = 128):\n        self.pairs = pairs.reset_index(drop=True)\n        self.size = size\n\n    def __len__(self) -> int:\n        return len(self.pairs)\n\n    def __getitem__(self, idx: int):\n        r = self.pairs.iloc[idx]\n        hr = np.asarray(\n            Image.open(_resolve_manifest_path(r[\"reference_path\"])).convert(\"RGB\").resize(\n                (self.size, self.size), Image.Resampling.BICUBIC\n            ),\n            dtype=np.float32,\n        )\n        cond = np.asarray(\n            Image.open(_resolve_manifest_path(r[\"hairy_path\"])).convert(\"RGB\").resize(\n                (self.size, self.size), Image.Resampling.BICUBIC\n            ),\n            dtype=np.float32,\n        )\n        if idx % 2 == 1:\n            hr, cond = hr[:, ::-1].copy(), cond[:, ::-1].copy()\n        return {\n            \"HR\": torch.from_numpy(hr / 127.5 - 1.0).permute(2, 0, 1),\n            \"SR\": torch.from_numpy(cond / 127.5 - 1.0).permute(2, 0, 1),\n        }\n\n\ndef _infinite(loader):\n    while True:\n        for batch in loader:\n            yield batch\n\n\n@torch.no_grad()\ndef val_loss(net, loader, device: str, timesteps=(200, 800, 1400, 1900)) -> float:\n    net.eval()\n    losses = []\n    for batch in loader:\n        hr = batch[\"HR\"].to(device)\n        cond = batch[\"SR\"].to(device)\n        for t in timesteps:\n            # Upstream encodes the noise level as sqrt(alpha_cumprod_prev), not as\n            # t/T: `NoiseLevelEncoder` multiplies a frequency ramp by this value,\n            # and `p_sample` feeds the same quantity at inference. Using t/T here\n            # trains one encoding and samples with another.\n            alpha = net.sqrt_alphas_cumprod_prev[int(t) - 1]\n            level = torch.full((hr.shape[0], 1), float(alpha), device=device)\n            noise = torch.randn_like(hr)\n            noisy = alpha * hr + (1 - alpha**2) ** 0.5 * noise\n            pred = net.denoise_fn(torch.cat([cond, noisy], dim=1), level)\n            losses.append(float((pred - noise).abs().mean()))\n    net.train()\n    return float(np.mean(losses))\n\n\ndef training_loss(net, hr: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:\n    \"\"\"One training loss, equivalent to upstream `p_losses` / numel.\n\n    RNG draw order (numpy t, numpy levels, torch noise) matches upstream so a\n    seeded differential test can compare the two exactly.\n    \"\"\"\n    t = np.random.randint(1, net.num_timesteps + 1)\n    # Continuous noise level per sample, as upstream: uniform between\n    # sqrt(alpha_cumprod_prev) at t-1 and at t.\n    lo = float(net.sqrt_alphas_cumprod_prev[t - 1])\n    hi = float(net.sqrt_alphas_cumprod_prev[t])\n    alpha = torch.tensor(\n        np.random.uniform(lo, hi, size=hr.shape[0]), dtype=hr.dtype, device=hr.device\n    )\n    alpha_4d = alpha.view(-1, 1, 1, 1)\n    noise = torch.randn_like(hr)\n    x_noisy = alpha_4d * hr + (1 - alpha_4d**2) ** 0.5 * noise\n    pred = net.denoise_fn(torch.cat([cond, x_noisy], dim=1), alpha.view(-1, 1))\n    return (pred - noise).abs().mean()\n\n\ndef _describe(ds) -> dict:\n    \"\"\"What a dataset contains, for the training manifest (pair table or composite).\"\"\"\n    if hasattr(ds, \"manifest\"):\n        m = ds.manifest()\n        return {\"n\": m[\"n\"], \"ids\": m[\"target_ids\"] + m[\"source_ids\"],\n                \"splits\": m[\"splits\"], \"groups\": set(m[\"groups\"]),\n                \"hash\": hashlib.sha256(\"\\n\".join(sorted(m[\"target_ids\"] + m[\"source_ids\"]))\n                                       .encode()).hexdigest()}\n    p = ds.pairs\n    return {\"n\": int(len(ds)), \"ids\": p[\"pair_id\"].tolist(),\n            \"splits\": sorted(p[\"split\"].unique().tolist()),\n            \"groups\": set(p[\"group_id\"]), \"hash\": _manifest_hash(p)}\n\n\ndef train(\n    cfg: dict,\n    pairs: pd.DataFrame | None,\n    val_pairs: pd.DataFrame | None,\n    out_dir: Path,\n    device: str = \"cuda\",\n    overfit: bool = False,\n    init_from: Path | None = None,\n    max_iterations: int | None = None,\n    train_ds: Dataset | None = None,\n    val_ds: Dataset | None = None,\n    num_workers: int = 4,\n    grad_clip: float | None = 1.0,\n) -> dict:\n    _ensure_path()\n    import model.networks as networks  # noqa: PLC0415\n\n    out_dir.mkdir(parents=True, exist_ok=True)\n    opt = build_opt(cfg)\n    net = networks.define_G(opt).to(device)\n    # Upstream's define_G/__init__ never registers the diffusion buffers, so the\n    # noise schedule must be set explicitly before any q_sample/p_losses use.\n    net.set_new_noise_schedule(\n        {\"schedule\": cfg[\"beta_schedule\"][\"schedule\"],\n         \"linear_start\": cfg[\"beta_schedule\"][\"linear_start\"],\n         \"linear_end\": cfg[\"beta_schedule\"][\"linear_end\"],\n         \"n_timestep\": int(cfg[\"num_timesteps\"])},\n        device=torch.device(device),\n    )\n    opt_g = torch.optim.Adam(net.parameters(), lr=float(cfg[\"train\"][\"lr\"]))\n    iterations = int(cfg[\"train\"][\"iterations\"])\n    if overfit and pairs is not None:\n        pairs = pairs.head(int(cfg[\"train\"].get(\"overfit_pairs\", 12)))\n        iterations = 500\n    if max_iterations is not None:\n        iterations = int(max_iterations)\n    if train_ds is None:\n        train_ds = PairedDataset(pairs, int(cfg[\"native_resolution\"]))\n    if val_ds is None:\n        val_ds = PairedDataset(val_pairs, int(cfg[\"native_resolution\"]))\n    bs = int(cfg[\"train\"][\"batch_size\"])\n    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True, drop_last=True,\n                              num_workers=int(num_workers), pin_memory=True,\n                              persistent_workers=int(num_workers) > 0)\n    val_loader = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=2)\n\n    state_path = out_dir / (\"overfit_state.pt\" if overfit else \"train_state.pt\")\n    start = 0\n    best = float(\"inf\")\n    history = []\n    if init_from is not None:\n        # Warm start from an existing checkpoint. Used to check that the fixed\n        # trainer does not degrade weights it did not produce.\n        init_state = torch.load(init_from, map_location=\"cpu\", weights_only=True)\n        if any(k.startswith(\"module.\") for k in init_state):\n            init_state = {k.replace(\"module.\", \"\", 1): v for k, v in init_state.items()}\n        net.load_state_dict(init_state, strict=True)\n        log.info(\"warm-started from %s\", init_from)\n    if state_path.exists():\n        state = torch.load(state_path, map_location=\"cpu\", weights_only=False)\n        net.load_state_dict(state[\"model\"])\n        opt_g.load_state_dict(state[\"optimizer\"])\n        start = int(state[\"iteration\"])\n        best = float(state.get(\"best_val\", best))\n        history = state.get(\"history\", [])\n        log.info(\"resumed DM-AHR at iteration %d\", start)\n\n    d_tr, d_va = _describe(train_ds), _describe(val_ds)\n    save_json(\n        {\n            \"config\": cfg,\n            \"opt_hash\": opt_hash(opt),\n            \"train_pairs\": d_tr[\"n\"],\n            \"val_pairs\": d_va[\"n\"],\n            \"train_pair_ids\": d_tr[\"ids\"],\n            \"val_pair_ids\": d_va[\"ids\"],\n            \"train_sources_splits\": d_tr[\"splits\"],\n            \"val_sources_splits\": d_va[\"splits\"],\n            \"leakage_check\": bool(not d_tr[\"groups\"] & d_va[\"groups\"]),\n            \"overfit\": overfit,\n            # Provenance recorded at run time so a later audit does not have to\n            # infer which noise-level convention or which data this run used.\n            \"noise_level_convention\": \"sqrt_alphas_cumprod_prev per sample, \"\n                                      \"continuous in [t-1, t] (upstream p_losses)\",\n            \"num_timesteps\": int(net.num_timesteps),\n            \"ema\": \"none\",\n            \"grad_clip\": float(grad_clip) if grad_clip else None,\n            \"batch_size\": int(cfg[\"train\"][\"batch_size\"]),\n            \"init_from\": str(init_from) if init_from is not None else None,\n            \"train_manifest_sha256\": d_tr[\"hash\"],\n            \"dataset_kind\": type(train_ds).__name__,\n        },\n        out_dir / \"training_manifest.json\",\n    )\n\n    net.train()\n    it = _infinite(train_loader)\n    ckpt_every = int(cfg[\"train\"][\"checkpoint_every\"])\n    val_every = int(cfg[\"train\"][\"val_every\"])\n    t0 = time.time()\n    for step in range(start + 1, iterations + 1):\n        batch = next(it)\n        hr = batch[\"HR\"].to(device, non_blocking=True)\n        cond = batch[\"SR\"].to(device, non_blocking=True)\n        loss = training_loss(net, hr, cond)\n        opt_g.zero_grad(set_to_none=True)\n        loss.backward()\n        if grad_clip:\n            torch.nn.utils.clip_grad_norm_(net.parameters(), float(grad_clip))\n        opt_g.step()\n\n        if step % 100 == 0:\n            log.info(\"dmahr iter %d/%d loss %.5f (%.0fs)\", step, iterations,\n                     float(loss), time.time() - t0)\n        if step % val_every == 0 or step == iterations:\n            v = val_loss(net, val_loader, device)\n            history.append({\"iteration\": step, \"train_l1\": float(loss), \"val_l1\": v})\n            log.info(\"dmahr val iter %d: %.5f\", step, v)\n            if v < best:\n                best = v\n                torch.save(net.state_dict(), out_dir / \"best_gen.pth\")\n        if step % ckpt_every == 0 or step == iterations:\n            tmp = state_path.with_suffix(\".tmp\")\n            torch.save(\n                {\"iteration\": step, \"model\": net.state_dict(),\n                 \"optimizer\": opt_g.state_dict(), \"best_val\": best,\n                 \"history\": history},\n                tmp,\n            )\n            tmp.replace(state_path)\n        # Iteration-numbered weights at a coarser interval, so a learning curve\n        # can be measured afterwards without retraining.\n        snap_every = int(cfg[\"train\"].get(\"snapshot_every\", 5000))\n        if snap_every and (step % snap_every == 0 or step == iterations):\n            torch.save(net.state_dict(), out_dir / f\"gen_iter{step:06d}.pth\")\n\n    torch.save(net.state_dict(), out_dir / \"final_gen.pth\")\n    save_json({\"history\": history, \"best_val\": best, \"iterations\": iterations,\n               \"seconds\": round(time.time() - t0, 1)}, out_dir / \"train_log.json\")\n    return {\"iterations\": iterations, \"best_val\": best, \"seconds\": time.time() - t0}\n\n",
    "dm_ahr_vit/hair/__init__.py": "\"\"\"Hair screening and classical (DullRazor-style) removal.\"\"\"\n\n",
    "dm_ahr_vit/hair/dullrazor.py": "\"\"\"DullRazor-style morphology + Telea inpainting baseline (Arm B).\"\"\"\n\nfrom __future__ import annotations\n\nimport cv2\nimport numpy as np\n\n\ndef apply_dullrazor(image_rgb: np.ndarray, hair_mask: np.ndarray, cfg: dict) -> np.ndarray:\n    mask = (hair_mask > 0).astype(np.uint8) * 255\n    if cfg.get(\"mask_dilate_kernel\", 0):\n        k = cv2.getStructuringElement(\n            cv2.MORPH_ELLIPSE,\n            (int(cfg[\"mask_dilate_kernel\"]), int(cfg[\"mask_dilate_kernel\"])),\n        )\n        mask = cv2.dilate(mask, k, iterations=int(cfg.get(\"mask_dilate_iterations\", 1)))\n    method = cv2.INPAINT_TELEA if cfg.get(\"inpaint_method\", \"telea\") == \"telea\" else cv2.INPAINT_NS\n    bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)\n    out = cv2.inpaint(bgr, mask, float(cfg.get(\"inpaint_radius\", 3)), method)\n    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)\n\n",
    "dm_ahr_vit/hair/harvest.py": "\"\"\"Harvest real hair from dermoscopic images as thin-structure masks.\n\nThe frozen detector (`screening.py`) is built to flag *whether* hair is present and is\ndeliberately generous: its mask is dilated and picks up lesion texture. For compositing real\nhair onto clean images we need the opposite: only long, thin, dark strands.\n\nA dark thin component is kept as hair only when all of these hold:\n  * strong response (well above the screening threshold), so faint texture is excluded;\n  * long skeleton and small mean width (strands, not blobs or short dashes);\n  * few junctions per unit length (crossing strands yes, the dense mesh of pigment network no);\n  * not blue or purple (pen ink) and not hugging the image border (dermoscope frame edge).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport cv2\nimport numpy as np\nfrom skimage.morphology import skeletonize\n\nfrom .screening import hair_response\n\nHARVEST_THRESHOLD = 30\nMIN_PEAK_RESPONSE = 80\nMIN_COMPONENT_AREA = 40\nMIN_SKELETON_LENGTH = 40\nMAX_MEAN_WIDTH = 7.0\nMAX_BRANCH_DENSITY = 0.03\nMAX_BLUE_OVER_RED = 8.0\nMAX_BORDER_FRACTION = 0.5\nBORDER_PX = 4\nMAX_FRACTION = 0.25\n\n\ndef thin_hair_mask(image_rgb: np.ndarray, cfg: dict,\n                   min_skeleton: int = MIN_SKELETON_LENGTH,\n                   max_width: float = MAX_MEAN_WIDTH,\n                   threshold: int = HARVEST_THRESHOLD,\n                   min_peak: int = MIN_PEAK_RESPONSE,\n                   max_branch_density: float = MAX_BRANCH_DENSITY) -> tuple[np.ndarray, dict]:\n    \"\"\"Return a uint8 {0,1} mask of accepted strands and a stats dict.\"\"\"\n    h, w = image_rgb.shape[:2]\n    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)\n    dark, _ = hair_response(gray, {**cfg, \"light_hair\": False})\n    mask = (dark > int(threshold)).astype(np.uint8)\n    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)\n    skel = skeletonize(mask > 0)\n    skel_u8 = skel.astype(np.uint8)\n    skel_len = np.bincount(labels[skel], minlength=n).astype(np.float64)\n    nb = cv2.filter2D(skel_u8, -1, np.ones((3, 3), np.float32), borderType=cv2.BORDER_CONSTANT)\n    junction = (skel & ((nb - skel_u8) >= 3)).astype(np.uint8)\n    # Count branch *points* (clusters of junction pixels), not junction pixels.\n    n_j, jlab = cv2.connectedComponents(junction, connectivity=8)\n    branches = np.zeros(n, dtype=np.float64)\n    if n_j > 1:\n        ys, xs = np.nonzero(junction)\n        _, first = np.unique(jlab[ys, xs], return_index=True)\n        branches = np.bincount(labels[ys[first], xs[first]], minlength=n).astype(np.float64)\n    area = stats[:, cv2.CC_STAT_AREA].astype(np.float64)\n    peak = np.zeros(n, dtype=np.float64)\n    np.maximum.at(peak, labels.ravel(), dark.ravel().astype(np.float64))\n    flat = labels.ravel()\n    mean_rgb = [np.bincount(flat, weights=image_rgb[..., c].ravel().astype(np.float64),\n                            minlength=n) / np.maximum(area, 1) for c in range(3)]\n    border = np.zeros((h, w), bool)\n    border[:BORDER_PX], border[-BORDER_PX:] = True, True\n    border[:, :BORDER_PX], border[:, -BORDER_PX:] = True, True\n    border_frac = np.bincount(labels[border], minlength=n) / np.maximum(area, 1)\n\n    keep = np.zeros(n, dtype=bool)\n    reasons = {\"small\": 0, \"short\": 0, \"wide\": 0, \"weak\": 0, \"mesh\": 0, \"ink\": 0, \"border\": 0}\n    for i in range(1, n):\n        if area[i] < MIN_COMPONENT_AREA:\n            reasons[\"small\"] += 1\n        elif skel_len[i] < min_skeleton:\n            reasons[\"short\"] += 1\n        elif area[i] / skel_len[i] > max_width:\n            reasons[\"wide\"] += 1\n        elif peak[i] < min_peak:\n            reasons[\"weak\"] += 1\n        elif branches[i] / skel_len[i] > max_branch_density:\n            reasons[\"mesh\"] += 1\n        elif mean_rgb[2][i] - mean_rgb[0][i] > MAX_BLUE_OVER_RED:\n            reasons[\"ink\"] += 1\n        elif border_frac[i] > MAX_BORDER_FRACTION:\n            reasons[\"border\"] += 1\n        else:\n            keep[i] = True\n    out = keep[labels].astype(np.uint8)\n    fraction = float(out.mean())\n    info = {\"components_total\": int(n - 1), \"components_kept\": int(keep.sum()),\n            \"hair_pixels\": int(out.sum()), \"fraction\": fraction, \"rejected\": reasons}\n    if fraction > MAX_FRACTION:\n        out = np.zeros_like(out)\n        info.update(components_kept=0, hair_pixels=0, fraction=0.0)\n    return out, info\n",
    "dm_ahr_vit/hair/pasteback.py": "\"\"\"Full-resolution paste-back of a low-resolution diffusion fill.\n\nThe diffusion model works at 128 px and regenerates the whole image. Here its output is used\nonly where it removed dark hair: flagged pixels (fill brighter than input) are replaced, and\nevery other pixel stays the original, full-resolution value.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport cv2\nimport numpy as np\nfrom PIL import Image\n\n_GRAY = np.array([0.299, 0.587, 0.114], np.float32)\n\n\ndef hair_flag(in_small: np.ndarray, fill_small: np.ndarray, tau: float) -> np.ndarray:\n    \"\"\"True where the fill is brighter than the input by more than `tau` gray levels.\"\"\"\n    gin = in_small.astype(np.float32) @ _GRAY\n    gout = fill_small.astype(np.float32) @ _GRAY\n    return (gout - gin) > float(tau)\n\n\ndef pasteback(orig: np.ndarray, fill_small: np.ndarray, tau: float = 16.0, dilate: int = 1,\n              det_mask: np.ndarray | None = None, refine_dilate: int = 2\n              ) -> tuple[np.ndarray, dict]:\n    \"\"\"Blend `fill_small` (square, e.g. 128 px) into `orig` (H x W x 3 uint8).\n\n    `det_mask` is an optional full-resolution binary hair mask; when given, the replaced\n    region is limited to it (dilated by `refine_dilate` px).\n    \"\"\"\n    h, w = orig.shape[:2]\n    s = fill_small.shape[0]\n    in_small = np.asarray(Image.fromarray(orig).resize((s, s), Image.Resampling.BICUBIC))\n    flag = hair_flag(in_small, fill_small, tau).astype(np.uint8)\n    if dilate > 0:\n        flag = cv2.dilate(flag, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)),\n                          iterations=int(dilate))\n    m = cv2.resize(flag.astype(np.float32), (w, h), interpolation=cv2.INTER_LINEAR)\n    if det_mask is not None:\n        det = (det_mask > 0).astype(np.uint8)\n        if refine_dilate > 0:\n            k = 2 * int(refine_dilate) + 1\n            det = cv2.dilate(det, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))\n        m = m * det.astype(np.float32)\n    fill = np.asarray(Image.fromarray(fill_small).resize((w, h), Image.Resampling.BICUBIC),\n                      dtype=np.float32)\n    m3 = m[..., None]\n    out = orig.astype(np.float32) * (1.0 - m3) + fill * m3\n    out = np.clip(np.rint(out), 0, 255).astype(np.uint8)\n    return out, {\"mask_fraction\": float(m.mean()), \"changed_fraction\": float((m > 0).mean())}\n",
    "dm_ahr_vit/hair/screening.py": "\"\"\"Multi-orientation black-hat/top-hat hair detector.\n\nParameters are tuned on train/validation synthetic pairs only and frozen before\nany test image is processed.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport cv2\nimport numpy as np\n\n\ndef _oriented_kernel(length: int, angle_deg: float) -> np.ndarray:\n    kernel = np.zeros((length, length), dtype=np.uint8)\n    c = length // 2\n    rad = np.deg2rad(angle_deg)\n    dx, dy = int(round(c * np.cos(rad))), int(round(c * np.sin(rad)))\n    cv2.line(kernel, (c - dx, c - dy), (c + dx, c + dy), 1, 1)\n    return kernel\n\n\ndef hair_response(gray: np.ndarray, cfg: dict) -> tuple[np.ndarray, np.ndarray]:\n    dark = np.zeros_like(gray)\n    light = np.zeros_like(gray)\n    for i in range(int(cfg[\"orientations\"])):\n        angle = 180.0 * i / int(cfg[\"orientations\"])\n        k = _oriented_kernel(int(cfg[\"kernel_length\"]), angle)\n        dark = np.maximum(dark, cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, k))\n        if cfg.get(\"light_hair\", True):\n            light = np.maximum(light, cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, k))\n    return dark, light\n\n\ndef detect_hair(image_rgb: np.ndarray, cfg: dict) -> tuple[np.ndarray, dict]:\n    \"\"\"Return a binary hair mask and detection statistics.\"\"\"\n    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)\n    dark, light = hair_response(gray, cfg)\n    mask = (dark > int(cfg[\"blackhat_threshold\"])).astype(np.uint8)\n    if cfg.get(\"light_hair\", True):\n        mask |= (light > int(cfg[\"tophat_threshold\"])).astype(np.uint8)\n\n    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)\n    keep = np.zeros_like(mask)\n    for i in range(1, n):\n        if stats[i, cv2.CC_STAT_AREA] >= int(cfg[\"min_component_area\"]):\n            keep[labels == i] = 1\n    k = cv2.getStructuringElement(\n        cv2.MORPH_ELLIPSE, (int(cfg[\"dilate_kernel\"]), int(cfg[\"dilate_kernel\"]))\n    )\n    keep = cv2.dilate(keep, k, iterations=int(cfg[\"dilate_iterations\"]))\n    coverage = float(keep.mean())\n    stats_out = {\n        \"coverage\": coverage,\n        \"hair_present\": bool(coverage >= float(cfg[\"coverage_threshold\"])),\n        \"dark_max\": int(dark.max()),\n        \"light_max\": int(light.max()),\n    }\n    return keep, stats_out\n\n\ndef tune_coverage_threshold(\n    pairs: list[tuple[np.ndarray, int]], cfg: dict, grid: np.ndarray | None = None\n) -> dict:\n    \"\"\"Pick the coverage threshold maximizing balanced accuracy on labelled pairs.\n\n    `pairs` is [(image_rgb, is_hairy)] from train/validation synthetic pairs.\n    \"\"\"\n    grid = grid if grid is not None else np.round(np.arange(0.002, 0.081, 0.002), 4)\n    coverages = np.array([detect_hair(img, cfg)[1][\"coverage\"] for img, _ in pairs])\n    labels = np.array([lab for _, lab in pairs])\n    best = None\n    for thr in grid:\n        pred = (coverages >= thr).astype(int)\n        tpr = float((pred[labels == 1] == 1).mean()) if (labels == 1).any() else 0.0\n        tnr = float((pred[labels == 0] == 0).mean()) if (labels == 0).any() else 0.0\n        score = 0.5 * (tpr + tnr)\n        if best is None or score > best[\"balanced_accuracy\"]:\n            best = {\n                \"coverage_threshold\": float(thr),\n                \"balanced_accuracy\": score,\n                \"sensitivity\": tpr,\n                \"specificity\": tnr,\n            }\n    return best\n\n",
    "dm_ahr_vit/paths.py": "\"\"\"Filesystem layout. Large data lives on Modal Volumes, never in Git.\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\nREPO_ROOT = Path(__file__).resolve().parents[2]\n\n\ndef data_root() -> Path:\n    return Path(os.environ.get(\"DATA_ROOT\", REPO_ROOT / \"data\")).resolve()\n\n\ndef artifact_root() -> Path:\n    return Path(os.environ.get(\"ARTIFACT_ROOT\", REPO_ROOT / \"artifacts\")).resolve()\n\n\ndef raw_dir() -> Path:\n    return data_root() / \"raw\"\n\n\ndef isic_images_dir() -> Path:\n    return raw_dir() / \"isic_212\" / \"images\"\n\n\ndef ham_root() -> Path:\n    return raw_dir() / \"ham10000\"\n\n\ndef ham_masks_dir() -> Path:\n    return ham_root() / \"masks\"\n\n\ndef processed_dir(arm: str) -> Path:\n    return data_root() / \"processed\" / arm\n\n\ndef manifest_dir() -> Path:\n    return data_root() / \"manifests\"\n\n\ndef artifact_dir(*parts: str) -> Path:\n    p = artifact_root().joinpath(*parts)\n    p.mkdir(parents=True, exist_ok=True)\n    return p\n\n\ndef ensure_dirs() -> None:\n    for p in (\n        raw_dir(),\n        isic_images_dir(),\n        ham_root(),\n        ham_masks_dir(),\n        manifest_dir(),\n        artifact_dir(\"logs\"),\n        artifact_dir(\"checkpoints\"),\n        artifact_dir(\"predictions\"),\n        artifact_dir(\"figures\"),\n        artifact_dir(\"panels\"),\n        artifact_dir(\"xai_maps\"),\n    ):\n        p.mkdir(parents=True, exist_ok=True)\n\n",
    "dm_ahr_vit/reporting/__init__.py": "",
    "dm_ahr_vit/reporting/stats_text.py": "\"\"\"Per-image log-loss differences between arms (replaces the standardised effect size dz).\n\nFor two arms, the per-image log-loss difference (a minus b), averaged over the seeds both arms\nhave, with a lesion-group bootstrap interval; per-seed differences are returned alongside.\nNegative means arm a has the lower (better) loss.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom ..config import CLASSES\n\nFLOOR = 1e-12\n\n\ndef _loss(path: Path) -> pd.DataFrame:\n    d = pd.read_csv(path).sort_values(\"image_id\").reset_index(drop=True)\n    p = d[[f\"prob_{c}\" for c in CLASSES]].to_numpy()\n    true = d[\"label\"].map(CLASSES.index).to_numpy()\n    return pd.DataFrame({\"image_id\": d[\"image_id\"], \"group_id\": d[\"group_id\"],\n                         \"loss\": -np.log(np.maximum(p[np.arange(len(d)), true], FLOOR))})\n\n\ndef logloss_diff(pred_dir: Path, a: str, b: str, seeds=(42, 123, 2026), reps: int = 2000,\n                 rng_seed: int = 20261113, subset: str = \"full\") -> dict:\n    per_seed, diffs, groups = {}, [], None\n    for s in seeds:\n        fa, fb = Path(pred_dir) / f\"pred_{a}_seed{s}_{subset}.csv\", Path(pred_dir) / f\"pred_{b}_seed{s}_{subset}.csv\"\n        if not (fa.exists() and fb.exists()):\n            continue\n        la, lb = _loss(fa), _loss(fb)\n        assert (la[\"image_id\"].to_numpy() == lb[\"image_id\"].to_numpy()).all()\n        d = la[\"loss\"].to_numpy() - lb[\"loss\"].to_numpy()\n        per_seed[s] = float(d.mean())\n        diffs.append(d)\n        groups = la[\"group_id\"].to_numpy()\n    if not diffs:\n        return {\"difference\": float(\"nan\"), \"ci_low\": float(\"nan\"), \"ci_high\": float(\"nan\"), \"per_seed\": {}}\n    d = np.mean(diffs, axis=0)\n    uniq, gi = np.unique(groups, return_inverse=True)\n    sums = np.bincount(gi, weights=d, minlength=len(uniq))\n    n = np.bincount(gi, minlength=len(uniq)).astype(float)\n    rng = np.random.default_rng(rng_seed)\n    idx = rng.integers(0, len(uniq), size=(reps, len(uniq)))\n    boot = sums[idx].sum(axis=1) / n[idx].sum(axis=1)\n    lo, hi = np.percentile(boot, [2.5, 97.5])\n    return {\"difference\": float(d.mean()), \"ci_low\": float(lo), \"ci_high\": float(hi), \"per_seed\": per_seed}\n",
    "dm_ahr_vit/reporting/xai_text.py": "\"\"\"Generated wording for the corrected explanation results (single source for every report).\n\nReads the tables written by scripts/59_xai_stats.py and returns sentences and table rows, so\nno explanation number is typed by hand in any generator.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom ..xai.batched import metric_definitions\n\nARM_NAME = {\"A\": \"A (original)\", \"B\": \"B (DullRazor)\", \"Astar\": \"A* (128 px control)\",\n            \"Cv2\": \"Cv2 (rebuilt diffusion, exploratory)\", \"C\": \"C (first diffusion run, defective trainer)\",\n            \"Creleased\": \"Creleased (released checkpoint, exploratory)\"}\nPRIMARY = \"Grad-CAM, last-block ln_1 (primary)\"\nSENS = \"Grad-CAM, patch grid (sensitivity)\"\nROLL = \"attention rollout\"\nAS_IS, SHIFTED = \"as is\", \"off-centre (shifted up to 30%)\"\n\n\ndef load(logs: Path, split: str = \"test\") -> dict:\n    d = Path(logs) / \"xai_v2_analysis\"\n    return {\"geo\": pd.read_csv(d / f\"xai_geometry_by_arm_{split}.csv\"),\n            \"summary\": pd.read_csv(d / f\"xai_arm_summary_{split}.csv\"),\n            \"comp\": pd.read_csv(d / f\"xai_comparisons_{split}.csv\"),\n            \"agree\": pd.read_csv(d / f\"xai_seed_agreement_{split}.csv\")}\n\n\ndef _s(x, metric=None):\n    return f\"{x:.3f}\"\n\n\ndef geometry_sentence(x: dict) -> str:\n    g = x[\"geo\"].set_index(\"arm\")\n    full = g.loc[\"A\"]\n    dif = g.loc[\"Cv2\"] if \"Cv2\" in g.index else None\n    s = (f\"Carried into the 224-pixel frame the classifier sees, the lesion covers on average \"\n         f\"{full['mean_mask_area']:.3f} of the image for the arms stored at 600 x 450 (A, B)\")\n    if dif is not None:\n        s += (f\" and {dif['mean_mask_area']:.3f} for Cv2 and A*, where {dif['share_images_stored_128']:.1%} of \"\n              f\"images are stored at 128 x 128\")\n    s += \". With the top 20% of pixels, the best possible lesion coverage (recall ceiling) is \" \\\n         f\"{full['recall_ceiling']:.3f} for A and B\"\n    if dif is not None:\n        s += f\" and {dif['recall_ceiling']:.3f} for Cv2 and A*\"\n    s += (\", and the best possible share of salient pixels inside the lesion (precision ceiling) is \"\n          f\"{full['precision_ceiling']:.3f}\")\n    if dif is not None:\n        s += f\" and {dif['precision_ceiling']:.3f} respectively\"\n    return s + \".\"\n\n\ndef baseline_verdicts(x: dict, metric: str, frame: str = AS_IS) -> list[str]:\n    \"\"\"For each map: in how many arms the model is above, below, or not distinguishable from the\n    centred Gaussian (95% two-level bootstrap interval of the model-minus-Gaussian difference).\"\"\"\n    out = []\n    t = x[\"summary\"]\n    for method in (ROLL, PRIMARY, SENS):\n        q = t[(t[\"frame\"] == frame) & (t[\"metric\"] == metric) & (t[\"method\"] == method)]\n        if q.empty or \"minus_gauss\" not in q:\n            continue\n        above = [r.arm for r in q.itertuples() if r.minus_gauss_ci_low > 0]\n        below = [r.arm for r in q.itertuples() if r.minus_gauss_ci_high < 0]\n        mixed = [r.arm for r in q.itertuples() if r.minus_gauss_ci_low <= 0 <= r.minus_gauss_ci_high]\n        out.append(f\"{method}: above the centred Gaussian in {len(above)} of {len(q)} arms\"\n                   f\"{' (' + ', '.join(above) + ')' if above else ''}, below in {len(below)}\"\n                   f\"{' (' + ', '.join(below) + ')' if below else ''}, not distinguishable in {len(mixed)}\"\n                   f\"{' (' + ', '.join(mixed) + ')' if mixed else ''}.\")\n    return out\n\n\ndef comparison_rows(x: dict, metric: str, frame: str = AS_IS, methods=(ROLL, PRIMARY, SENS)) -> list[list[str]]:\n    c = x[\"comp\"]\n    rows = []\n    for r in c[(c[\"frame\"] == frame) & (c[\"metric\"] == metric) & c[\"method\"].isin(methods)].itertuples():\n        rows.append([r.family, f\"{r.a} - {r.b}\", r.method, f\"{r.difference:+.3f}\",\n                     f\"{r.ci_low:+.3f} to {r.ci_high:+.3f}\", f\"{r.p_holm:.3f}\", r.per_seed_differences,\n                     \"struck: seed spread exceeds the difference\" if r.struck else\n                     (\"seeds agree in sign\" if r.seeds_agree_in_sign else \"seeds disagree in sign\")])\n    return rows\n\n\ndef comparison_header() -> list[str]:\n    return [\"Family\", \"Comparison\", \"Map\", \"Difference\", \"95% CI (lesions and seeds)\", \"p (Holm)\",\n            \"Per-seed differences\", \"Reading\"]\n\n\ndef arm_table(x: dict, metric: str, frame: str = AS_IS) -> tuple[list[str], list[list[str]]]:\n    t = x[\"summary\"]\n    head = [\"Arm\", \"Map\", \"Seeds\", \"Mean (seed range)\", \"Gaussian\", \"Random pixels\", \"Random patches\",\n            \"Model - Gaussian (95% CI)\"]\n    rows = []\n    for r in t[(t[\"frame\"] == frame) & (t[\"metric\"] == metric)].itertuples():\n        rows.append([ARM_NAME.get(r.arm, r.arm), r.method, str(r.n_seeds),\n                     f\"{r.mean:.3f} ({r.seed_min:.3f} to {r.seed_max:.3f})\",\n                     f\"{r.baseline_gauss:.3f}\", f\"{getattr(r, 'baseline_rand_pixel', float('nan')):.3f}\",\n                     f\"{getattr(r, 'baseline_rand_patch', float('nan')):.3f}\",\n                     f\"{r.minus_gauss:+.3f} ({r.minus_gauss_ci_low:+.3f} to {r.minus_gauss_ci_high:+.3f})\"])\n    return head, rows\n\n\ndef agreement_sentence(x: dict) -> str:\n    a = x[\"agree\"]\n    parts = []\n    for m in (ROLL, PRIMARY, SENS):\n        q = a[a[\"method\"] == m]\n        parts.append(f\"{m}: {q['mean_pairwise_spearman'].min():.2f} to {q['mean_pairwise_spearman'].max():.2f}\")\n    return (\"Per-image lesion-coverage scores agree across independently trained seeds (mean pairwise Spearman \"\n            \"correlation, range over arms) as follows: \" + \"; \".join(parts) + \".\")\n\n\ndef definitions() -> str:\n    return metric_definitions(0.2)\n\n\ndef strike_summary(x: dict) -> str:\n    c = x[\"comp\"]\n    parts = []\n    for m in (ROLL, PRIMARY, SENS):\n        q = c[(c[\"method\"] == m) & (c[\"frame\"] == AS_IS) & (c[\"metric\"].isin([\"recall\", \"pointing\"]))]\n        parts.append(f\"{m}: {int(q['struck'].sum())} of {len(q)}\")\n    return (\"Rows struck by the pre-registered rule (seed spread larger than the between-arm difference), \"\n            \"recall and pointing, unshifted: \" + \"; \".join(parts) + \".\")\n",
    "dm_ahr_vit/stats/__init__.py": "\"\"\"Paired, lesion-cluster bootstrap statistics with Holm correction.\"\"\"\n\n",
    "dm_ahr_vit/stats/bootstrap.py": "\"\"\"Paired cluster bootstrap and Holm correction.\n\nAll resampling is done at the lesion-group level, and the same resample is\nshared across arms and seeds so comparisons stay paired.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\n\ndef holm(pvalues: list[float]) -> list[float]:\n    \"\"\"Holm step-down adjustment. NaN inputs are ignored in the family.\"\"\"\n    idx = [i for i, p in enumerate(pvalues) if p is not None and not np.isnan(p)]\n    m = len(idx)\n    out = [float(\"nan\")] * len(pvalues)\n    if m == 0:\n        return out\n    ordered = sorted(idx, key=lambda i: pvalues[i])\n    running = 0.0\n    for rank, i in enumerate(ordered):\n        adj = (m - rank) * float(pvalues[i])\n        running = max(running, adj)\n        out[i] = min(1.0, running)\n    return out\n\n\ndef paired_cluster_bootstrap(\n    values_a: np.ndarray,\n    values_b: np.ndarray,\n    groups: np.ndarray,\n    statistic=None,\n    replicates: int = 10000,\n    seed: int = 20260101,\n    alpha: float = 0.05,\n) -> dict:\n    \"\"\"Bootstrap the paired difference a - b, resampling lesion groups.\n\n    `values_a`/`values_b` hold one observation per row (same row alignment).\n    For image-level metric tables, aggregate to one row per group first.\n    \"\"\"\n    a = np.asarray(values_a, dtype=float)\n    b = np.asarray(values_b, dtype=float)\n    g = np.asarray(groups)\n    ok = np.isfinite(a) & np.isfinite(b)\n    a, b, g = a[ok], b[ok], g[ok]\n    if len(a) == 0:\n        return {\"n\": 0, \"note\": \"no finite paired observations\"}\n\n    stat = statistic or (lambda x: float(np.mean(x)))\n    uniq = np.unique(g)\n    index_by_group = {gu: np.nonzero(g == gu)[0] for gu in uniq}\n    observed = stat(a) - stat(b)\n    rng = np.random.default_rng(seed)\n    diffs = np.empty(replicates, dtype=float)\n    for r in range(replicates):\n        pick = rng.choice(uniq, size=len(uniq), replace=True)\n        rows = np.concatenate([index_by_group[gu] for gu in pick])\n        diffs[r] = stat(a[rows]) - stat(b[rows])\n    lo, hi = np.percentile(diffs, [100 * alpha / 2, 100 * (1 - alpha / 2)])\n    p_two = 2 * min(float((diffs <= 0).mean()), float((diffs >= 0).mean()))\n    sd = float(np.std(diffs, ddof=1)) if replicates > 1 else float(\"nan\")\n    return {\n        \"n\": int(len(a)),\n        \"n_groups\": int(len(uniq)),\n        \"stat_a\": float(stat(a)),\n        \"stat_b\": float(stat(b)),\n        \"difference\": float(observed),\n        \"ci_low\": float(lo),\n        \"ci_high\": float(hi),\n        \"p_raw\": float(min(1.0, p_two)),\n        \"effect_size_dz\": float(observed / sd) if sd and sd > 0 else float(\"nan\"),\n        \"replicates\": int(replicates),\n        \"seed\": int(seed),\n        \"alpha\": alpha,\n    }\n\n\ndef cohens_dz(differences: np.ndarray) -> float:\n    d = np.asarray(differences, dtype=float)\n    d = d[np.isfinite(d)]\n    if len(d) < 2 or d.std(ddof=1) == 0:\n        return float(\"nan\")\n    return float(d.mean() / d.std(ddof=1))\n\n\ndef wilcoxon_group_paired(df: pd.DataFrame, value: str, arm_col: str,\n                          group_col: str = \"group_id\") -> dict:\n    \"\"\"Group-aggregated Wilcoxon signed-rank between the two arms in `df`.\"\"\"\n    from scipy.stats import wilcoxon\n\n    wide = df.pivot_table(index=group_col, columns=arm_col, values=value, aggfunc=\"mean\")\n    wide = wide.dropna()\n    if wide.shape[1] != 2 or len(wide) < 6:\n        return {\"n_groups\": int(len(wide)), \"p_raw\": float(\"nan\"),\n                \"note\": \"insufficient paired groups\"}\n    arms = list(wide.columns)\n    try:\n        res = wilcoxon(wide[arms[0]], wide[arms[1]])\n        p = float(res.pvalue)\n        stat = float(res.statistic)\n    except Exception as exc:  # noqa: BLE001\n        return {\"n_groups\": int(len(wide)), \"p_raw\": float(\"nan\"), \"note\": repr(exc)}\n    return {\n        \"test\": \"wilcoxon_signed_rank_group_aggregated\",\n        \"arm_a\": arms[0],\n        \"arm_b\": arms[1],\n        \"n_groups\": int(len(wide)),\n        \"statistic\": stat,\n        \"p_raw\": p,\n        \"median_difference\": float((wide[arms[0]] - wide[arms[1]]).median()),\n    }\n\n\ndef wilcoxon_per_group(long_df: pd.DataFrame, arm_a: str, arm_b: str,\n                       value_col: str = \"value\",\n                       group_col: str = \"group_id\") -> dict:\n    \"\"\"Wilcoxon signed-rank on lesion-group-level paired values.\n\n    `long_df` carries one row per (arm, group). Collapsing to a single value per\n    lesion first keeps the many images of one lesion from being treated as\n    independent replicates.\n    \"\"\"\n    from scipy.stats import wilcoxon\n\n    out = {\"wilcoxon_n_groups\": 0, \"wilcoxon_statistic\": float(\"nan\"),\n           \"wilcoxon_p_raw\": float(\"nan\")}\n    if long_df is None or long_df.empty:\n        return out\n    sub = long_df[long_df[\"arm\"].isin([arm_a, arm_b])]\n    if sub.empty:\n        return out\n    wide = sub.pivot_table(index=group_col, columns=\"arm\",\n                           values=value_col, aggfunc=\"mean\").dropna()\n    out[\"wilcoxon_n_groups\"] = int(len(wide))\n    if wide.shape[1] != 2 or len(wide) < 6:\n        return out\n    a = wide[arm_a].to_numpy(dtype=float)\n    b = wide[arm_b].to_numpy(dtype=float)\n    if np.allclose(a - b, 0.0):\n        out[\"wilcoxon_statistic\"] = 0.0\n        out[\"wilcoxon_p_raw\"] = 1.0\n        return out\n    try:\n        res = wilcoxon(a, b)\n    except ValueError:\n        return out\n    out[\"wilcoxon_statistic\"] = float(res.statistic)\n    out[\"wilcoxon_p_raw\"] = float(res.pvalue)\n    return out\n\n",
    "dm_ahr_vit/stats/pooled.py": "\"\"\"Pooled-over-seeds statistics with a two-level bootstrap (lesion groups and training seeds).\n\nPure numpy/pandas so it runs and is tested without Modal. A scores array has shape\n(seeds, replicates): the score of one arm and seed on each lesion-group bootstrap replicate.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ..config import CLASSES\n\nLOSS_FLOOR = 1e-12\n\n\ndef macro_f1(tp: np.ndarray, fp: np.ndarray, fn: np.ndarray) -> np.ndarray:\n    denom = 2 * tp + fp + fn\n    per_class = np.divide(2 * tp, denom, out=np.zeros_like(tp, dtype=float), where=denom > 0)\n    return per_class.mean(axis=1)\n\n\ndef group_counts(df: pd.DataFrame, groups: list[str]) -> dict:\n    \"\"\"Per-lesion-group TP/FP/FN per class, as in scripts/09_stats.py.\"\"\"\n    gidx = {g: i for i, g in enumerate(groups)}\n    k = len(CLASSES)\n    tp, fp, fn = (np.zeros((len(groups), k)) for _ in range(3))\n    y = df[\"label\"].map(CLASSES.index).to_numpy()\n    p = df[\"pred\"].map(CLASSES.index).to_numpy()\n    g = df[\"group_id\"].map(gidx).to_numpy()\n    for gi, yi, pi in zip(g, y, p):\n        if yi == pi:\n            tp[gi, yi] += 1\n        else:\n            fp[gi, pi] += 1\n            fn[gi, yi] += 1\n    return {\"tp\": tp, \"fp\": fp, \"fn\": fn}\n\n\ndef group_loss(df: pd.DataFrame, groups: list[str]) -> tuple[np.ndarray, np.ndarray]:\n    \"\"\"Summed per-image log-loss and image count per lesion group.\"\"\"\n    probs = df[[f\"prob_{c}\" for c in CLASSES]].to_numpy()\n    true = df[\"label\"].map(CLASSES.index).to_numpy()\n    loss = -np.log(np.maximum(probs[np.arange(len(df)), true], LOSS_FLOOR))\n    gidx = {g: i for i, g in enumerate(groups)}\n    gi = df[\"group_id\"].map(gidx).to_numpy()\n    ls = np.bincount(gi, weights=loss, minlength=len(groups))\n    n = np.bincount(gi, minlength=len(groups)).astype(float)\n    return ls, n\n\n\ndef draw_weights(n_groups: int, reps: int, seed: int) -> np.ndarray:\n    rng = np.random.default_rng(seed)\n    idx = rng.integers(0, n_groups, size=(reps, n_groups))\n    w = np.zeros((reps, n_groups), dtype=np.float32)\n    for r in range(reps):\n        np.add.at(w[r], idx[r], 1.0)\n    return w\n\n\ndef f1_scores(counts_by_seed: list[dict], w: np.ndarray) -> np.ndarray:\n    return np.stack([macro_f1(w @ c[\"tp\"], w @ c[\"fp\"], w @ c[\"fn\"]) for c in counts_by_seed])\n\n\ndef loss_scores(loss_by_seed: list[tuple[np.ndarray, np.ndarray]], w: np.ndarray) -> np.ndarray:\n    return np.stack([(w @ ls) / np.maximum(w @ n, 1.0) for ls, n in loss_by_seed])\n\n\ndef two_level(scores: np.ndarray, rng: np.random.Generator) -> np.ndarray:\n    \"\"\"Resample seeds with replacement independently on every replicate, then average.\"\"\"\n    s, r = scores.shape\n    idx = rng.integers(0, s, size=(r, s))\n    return scores[idx, np.arange(r)[:, None]].mean(axis=1)\n\n\ndef one_level(scores: np.ndarray) -> np.ndarray:\n    \"\"\"Seeds held fixed (the earlier analysis): average the seeds on each replicate.\"\"\"\n    return scores.mean(axis=0)\n\n\ndef compare(dist_a: np.ndarray, dist_b: np.ndarray, obs_a: float, obs_b: float) -> dict:\n    diff = dist_a - dist_b\n    reps = diff.size\n    lo, hi = np.percentile(diff, [2.5, 97.5])\n    k_le, k_ge = int((diff <= 0).sum()), int((diff >= 0).sum())\n    p = min(1.0, 2 * min((k_le + 1) / (reps + 1), (k_ge + 1) / (reps + 1)))\n    se = float(diff.std(ddof=1))\n    return {\"difference\": float(obs_a - obs_b), \"ci_low\": float(lo), \"ci_high\": float(hi),\n            \"p_raw\": float(p), \"se_diff\": se, \"mdd_80pct\": 2.8 * se}\n",
    "dm_ahr_vit/utils.py": "\"\"\"Small shared helpers: logging, timing, run registry, image IO.\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport logging\nimport os\nimport platform\nimport socket\nimport sys\nimport time\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Iterator\n\nfrom .config import save_json\nfrom .paths import artifact_dir\n\n\ndef setup_logging(name: str = \"dm_ahr_vit\", level: int = logging.INFO) -> logging.Logger:\n    logger = logging.getLogger(name)\n    if not logger.handlers:\n        handler = logging.StreamHandler(sys.stdout)\n        handler.setFormatter(\n            logging.Formatter(\"%(asctime)s %(levelname)s %(name)s :: %(message)s\")\n        )\n        logger.addHandler(handler)\n        logger.setLevel(level)\n    return logger\n\n\ndef utc_now() -> str:\n    return datetime.now(timezone.utc).isoformat(timespec=\"seconds\")\n\n\n@contextmanager\ndef timed(label: str, logger: logging.Logger | None = None) -> Iterator[dict]:\n    start = time.time()\n    info: dict = {\"label\": label, \"started_at\": utc_now()}\n    try:\n        yield info\n    finally:\n        info[\"seconds\"] = round(time.time() - start, 3)\n        info[\"finished_at\"] = utc_now()\n        if logger is not None:\n            logger.info(\"%s finished in %.1fs\", label, info[\"seconds\"])\n\n\ndef run_registry_path() -> Path:\n    return artifact_dir(\"logs\") / \"run_registry.jsonl\"\n\n\ndef record_run(\n    job: str,\n    *,\n    gpu: str | None,\n    seconds: float,\n    status: str = \"ok\",\n    **extra,\n) -> None:\n    \"\"\"Append one JSON line per job. Never contains data or secrets.\"\"\"\n    entry = {\n        \"job\": job,\n        \"gpu\": gpu,\n        \"seconds\": round(float(seconds), 2),\n        \"status\": status,\n        \"host\": socket.gethostname(),\n        \"python\": platform.python_version(),\n        \"timestamp\": utc_now(),\n        \"pid\": os.getpid(),\n        **extra,\n    }\n    path = run_registry_path()\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\"a\", encoding=\"utf-8\") as fh:\n        fh.write(json.dumps(entry, sort_keys=True) + \"\\n\")\n\n\ndef write_json(path: str | Path, obj) -> Path:\n    return save_json(obj, path)\n\n\ndef read_json(path: str | Path):\n    return json.loads(Path(path).read_text())\n\n",
    "dm_ahr_vit/vit/__init__.py": "\"\"\"ViT-B/16 classifier training, evaluation, and explainability.\"\"\"\n\n",
    "dm_ahr_vit/vit/metrics.py": "\"\"\"Classification metrics with explicit handling of undefined values.\"\"\"\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import (\n    accuracy_score,\n    balanced_accuracy_score,\n    confusion_matrix,\n    f1_score,\n    precision_recall_fscore_support,\n    roc_auc_score,\n)\n\nfrom ..config import CLASSES\n\n\ndef classification_metrics(\n    y_true: np.ndarray,\n    y_pred: np.ndarray,\n    y_prob: np.ndarray | None = None,\n) -> dict:\n    labels = list(range(len(CLASSES)))\n    p, r, f1, support = precision_recall_fscore_support(\n        y_true, y_pred, labels=labels, zero_division=0\n    )\n    out = {\n        \"n\": int(len(y_true)),\n        \"accuracy\": float(accuracy_score(y_true, y_pred)),\n        \"balanced_accuracy\": float(balanced_accuracy_score(y_true, y_pred)),\n        \"macro_f1\": float(f1_score(y_true, y_pred, average=\"macro\", zero_division=0)),\n        \"weighted_f1\": float(f1_score(y_true, y_pred, average=\"weighted\", zero_division=0)),\n        \"per_class\": {},\n    }\n    cm = confusion_matrix(y_true, y_pred, labels=labels)\n    for i, c in enumerate(CLASSES):\n        tp = int(cm[i, i])\n        fn = int(cm[i, :].sum() - tp)\n        fp = int(cm[:, i].sum() - tp)\n        tn = int(cm.sum() - tp - fn - fp)\n        sensitivity = tp / (tp + fn) if (tp + fn) else float(\"nan\")\n        specificity = tn / (tn + fp) if (tn + fp) else float(\"nan\")\n        auc, auc_note = _auc_for(y_true, y_prob, i)\n        out[\"per_class\"][c] = {\n            \"support\": int(support[i]),\n            \"precision\": float(p[i]),\n            \"recall\": float(r[i]),\n            \"f1\": float(f1[i]),\n            \"sensitivity\": sensitivity,\n            \"specificity\": specificity,\n            \"auc\": auc,\n            \"auc_note\": auc_note,\n        }\n    if y_prob is not None:\n        out[\"auc_macro\"] = _macro_auc(y_true, y_prob)\n        out[\"auc_weighted\"] = _weighted_auc(y_true, y_prob)\n    else:\n        out[\"auc_macro\"] = float(\"nan\")\n        out[\"auc_weighted\"] = float(\"nan\")\n    out[\"confusion_matrix\"] = cm.tolist()\n    out[\"confusion_matrix_normalized\"] = (\n        (cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)).round(6).tolist()\n    )\n    return out\n\n\ndef _auc_for(y_true: np.ndarray, y_prob: np.ndarray | None, idx: int):\n    if y_prob is None:\n        return float(\"nan\"), \"no_probabilities\"\n    binary = (y_true == idx).astype(int)\n    if binary.sum() == 0 or binary.sum() == len(binary):\n        return float(\"nan\"), \"single_class_present\"\n    return float(roc_auc_score(binary, y_prob[:, idx])), \"\"\n\n\ndef _macro_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:\n    vals = [v for v in (_auc_for(y_true, y_prob, i)[0] for i in range(len(CLASSES)))\n            if not np.isnan(v)]\n    return float(np.mean(vals)) if vals else float(\"nan\")\n\n\ndef _weighted_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:\n    vals, weights = [], []\n    for i in range(len(CLASSES)):\n        v = _auc_for(y_true, y_prob, i)[0]\n        if not np.isnan(v):\n            vals.append(v)\n            weights.append(int((y_true == i).sum()))\n    if not vals:\n        return float(\"nan\")\n    return float(np.average(vals, weights=weights))\n\n\ndef metrics_table(metrics_by_key: dict[tuple, dict]) -> pd.DataFrame:\n    \"\"\"Flatten many metric dicts into one CSV-friendly table.\"\"\"\n    rows = []\n    for key, m in metrics_by_key.items():\n        arm, seed, subset = key\n        base = {\n            \"arm\": arm,\n            \"seed\": seed,\n            \"subset\": subset,\n            \"n\": m[\"n\"],\n            \"accuracy\": m[\"accuracy\"],\n            \"balanced_accuracy\": m[\"balanced_accuracy\"],\n            \"macro_f1\": m[\"macro_f1\"],\n            \"weighted_f1\": m[\"weighted_f1\"],\n            \"auc_macro\": m[\"auc_macro\"],\n            \"auc_weighted\": m[\"auc_weighted\"],\n        }\n        for c in CLASSES:\n            pc = m[\"per_class\"][c]\n            for k in (\"support\", \"precision\", \"recall\", \"f1\", \"sensitivity\",\n                      \"specificity\", \"auc\"):\n                base[f\"{c}_{k}\"] = pc[k]\n            base[f\"{c}_auc_note\"] = pc[\"auc_note\"]\n        rows.append(base)\n    return pd.DataFrame(rows)\n\n",
    "dm_ahr_vit/vit/model.py": "\"\"\"torchvision ViT-B/16 with a fixed seven-class head.\"\"\"\n\nfrom __future__ import annotations\n\nimport torch\nfrom torch import nn\nfrom torchvision import models\n\nfrom ..config import CLASSES\n\n\ndef build_model(pretrained: bool = True, num_classes: int = len(CLASSES)) -> nn.Module:\n    weights = models.ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None\n    model = models.vit_b_16(weights=weights)\n    model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)\n    return model\n\n\ndef save_checkpoint(\n    path,\n    model: nn.Module,\n    optimizer,\n    scheduler,\n    epoch: int,\n    best_f1: float,\n    meta: dict,\n) -> None:\n    tmp = str(path) + \".tmp\"\n    torch.save(\n        {\n            \"model\": model.state_dict(),\n            \"optimizer\": optimizer.state_dict(),\n            \"scheduler\": scheduler.state_dict() if scheduler is not None else None,\n            \"epoch\": epoch,\n            \"best_val_macro_f1\": best_f1,\n            \"meta\": meta,\n        },\n        tmp,\n    )\n    import os\n\n    os.replace(tmp, path)\n\n\ndef load_checkpoint(path, model: nn.Module, map_location=\"cpu\") -> dict:\n    ckpt = torch.load(path, map_location=map_location, weights_only=False)\n    model.load_state_dict(ckpt[\"model\"])\n    return ckpt\n\n\n@torch.no_grad()\ndef predict_proba(model: nn.Module, loader, device: str) -> tuple[torch.Tensor, torch.Tensor]:\n    model.eval()\n    probs, labels = [], []\n    for batch in loader:\n        x = batch[\"image\"].to(device, non_blocking=True)\n        with torch.autocast(device_type=\"cuda\", dtype=torch.bfloat16, enabled=device == \"cuda\"):\n            logits = model(x)\n        probs.append(torch.softmax(logits.float(), dim=1).cpu())\n        labels.append(batch[\"label\"])\n    return torch.cat(probs), torch.cat(labels)\n\n",
    "dm_ahr_vit/vit/train.py": "\"\"\"Matched ViT-B/16 training with resumable checkpoints and early stopping.\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport math\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom sklearn.metrics import f1_score\nfrom torch import nn\nfrom torch.utils.data import DataLoader\n\nfrom ..config import CLASS_TO_IDX, CLASSES, save_json\nfrom ..data.dataset import ArmDataset, build_transforms, collate\nfrom ..utils import setup_logging\nfrom .model import build_model, save_checkpoint\n\nlog = setup_logging()\n\n\ndef class_weights(splits_df, class_order=CLASSES) -> torch.Tensor:\n    \"\"\"Inverse-frequency weights computed from TRAIN labels only.\"\"\"\n    counts = splits_df[splits_df[\"split\"] == \"train\"][\"dx\"].value_counts()\n    total = counts.sum()\n    w = [total / (len(class_order) * max(counts.get(c, 0), 1)) for c in class_order]\n    return torch.tensor(w, dtype=torch.float32)\n\n\ndef cosine_with_warmup(optimizer, warmup_epochs: int, total_epochs: int, steps_per_epoch: int):\n    warmup = max(1, warmup_epochs * steps_per_epoch)\n    total = max(warmup + 1, total_epochs * steps_per_epoch)\n\n    def fn(step: int) -> float:\n        if step < warmup:\n            return (step + 1) / warmup\n        progress = (step - warmup) / max(1, total - warmup)\n        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))\n\n    return torch.optim.lr_scheduler.LambdaLR(optimizer, fn)\n\n\n@torch.no_grad()\ndef evaluate(model, loader, device: str, criterion=None) -> dict:\n    model.eval()\n    probs, labels = [], []\n    loss_sum, n = 0.0, 0\n    for batch in loader:\n        x = batch[\"image\"].to(device, non_blocking=True)\n        y = batch[\"label\"].to(device, non_blocking=True)\n        with torch.autocast(device_type=\"cuda\", dtype=torch.bfloat16, enabled=device == \"cuda\"):\n            logits = model(x)\n        if criterion is not None:\n            loss_sum += float(criterion(logits.float(), y)) * y.numel()\n            n += y.numel()\n        probs.append(torch.softmax(logits.float(), dim=1).cpu())\n        labels.append(y.cpu())\n    prob = torch.cat(probs)\n    label = torch.cat(labels)\n    pred = prob.argmax(dim=1)\n    return {\n        \"prob\": prob,\n        \"label\": label,\n        \"pred\": pred,\n        \"loss\": loss_sum / max(n, 1),\n        \"macro_f1\": float(f1_score(label, pred, average=\"macro\", zero_division=0)),\n        \"accuracy\": float((pred == label).float().mean()),\n    }\n\n\ndef train_arm(\n    arm: str,\n    seed: int,\n    cfg: dict,\n    splits_df,\n    raw_images_dir: Path,\n    processed_root: Path,\n    run_dir: Path,\n    device: str = \"cuda\",\n    max_epochs: int | None = None,\n) -> dict:\n    run_dir.mkdir(parents=True, exist_ok=True)\n    torch.manual_seed(seed)\n    np.random.seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\n    train_ds = ArmDataset(arm, \"train\", splits_df, raw_images_dir, processed_root,\n                          build_transforms(cfg, train=True))\n    val_ds = ArmDataset(arm, \"val\", splits_df, raw_images_dir, processed_root,\n                        build_transforms(cfg, train=False))\n    bs = int(cfg[\"batch_size\"])\n    nw = int(cfg[\"num_workers\"])\n    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True, drop_last=False,\n                              num_workers=nw, pin_memory=True, collate_fn=collate,\n                              persistent_workers=nw > 0)\n    val_loader = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=nw,\n                            pin_memory=True, collate_fn=collate, persistent_workers=nw > 0)\n\n    model = build_model(pretrained=True).to(device)\n    weights = class_weights(splits_df).to(device)\n    criterion = nn.CrossEntropyLoss(\n        weight=weights, label_smoothing=float(cfg.get(\"label_smoothing\", 0.0))\n    )\n    optimizer = torch.optim.AdamW(\n        model.parameters(), lr=float(cfg[\"lr\"]), weight_decay=float(cfg[\"weight_decay\"])\n    )\n    epochs = int(max_epochs or cfg[\"epochs\"])\n    scheduler = cosine_with_warmup(optimizer, int(cfg[\"warmup_epochs\"]), epochs, len(train_loader))\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=False)\n\n    meta = {\n        \"arm\": arm,\n        \"seed\": seed,\n        \"config\": cfg,\n        \"class_weights\": weights.detach().cpu().tolist(),\n        \"class_order\": list(CLASSES),\n        \"train_size\": len(train_ds),\n        \"val_size\": len(val_ds),\n    }\n    save_json(meta, run_dir / \"run_meta.json\")\n\n    last_path = run_dir / \"last.pt\"\n    best_path = run_dir / \"best.pt\"\n    start_epoch, best_f1, bad_epochs, history = 0, -1.0, 0, []\n    if last_path.exists():\n        ckpt = torch.load(last_path, map_location=\"cpu\", weights_only=False)\n        model.load_state_dict(ckpt[\"model\"])\n        optimizer.load_state_dict(ckpt[\"optimizer\"])\n        scheduler.load_state_dict(ckpt[\"scheduler\"])\n        start_epoch = ckpt[\"epoch\"] + 1\n        best_f1 = ckpt[\"best_val_macro_f1\"]\n        bad_epochs = ckpt[\"meta\"].get(\"bad_epochs\", 0)\n        history = ckpt[\"meta\"].get(\"history\", [])\n        log.info(\"resumed %s/%d at epoch %d\", arm, seed, start_epoch)\n\n    patience = int(cfg[\"early_stopping_patience\"])\n    t0 = time.time()\n    for epoch in range(start_epoch, epochs):\n        model.train()\n        run_loss, seen = 0.0, 0\n        for batch in train_loader:\n            x = batch[\"image\"].to(device, non_blocking=True)\n            y = batch[\"label\"].to(device, non_blocking=True)\n            with torch.autocast(device_type=\"cuda\", dtype=torch.bfloat16, enabled=device == \"cuda\"):\n                logits = model(x)\n                loss = criterion(logits.float(), y)\n            optimizer.zero_grad(set_to_none=True)\n            scaler.scale(loss).backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg[\"grad_clip\"]))\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n            run_loss += float(loss) * y.numel()\n            seen += y.numel()\n        val = evaluate(model, val_loader, device, criterion)\n        rec = {\n            \"epoch\": epoch,\n            \"train_loss\": run_loss / max(seen, 1),\n            \"val_loss\": val[\"loss\"],\n            \"val_macro_f1\": val[\"macro_f1\"],\n            \"val_accuracy\": val[\"accuracy\"],\n            \"lr\": scheduler.get_last_lr()[0],\n            \"seconds\": round(time.time() - t0, 1),\n        }\n        history.append(rec)\n        log.info(\"%s/%d ep %d: train %.4f val %.4f f1 %.4f\", arm, seed, epoch,\n                 rec[\"train_loss\"], rec[\"val_loss\"], rec[\"val_macro_f1\"])\n\n        if val[\"macro_f1\"] > best_f1:\n            best_f1 = val[\"macro_f1\"]\n            bad_epochs = 0\n            save_checkpoint(best_path, model, optimizer, scheduler, epoch, best_f1, meta)\n        else:\n            bad_epochs += 1\n        meta_epoch = {**meta, \"bad_epochs\": bad_epochs, \"history\": history}\n        save_checkpoint(last_path, model, optimizer, scheduler, epoch, best_f1, meta_epoch)\n        if bad_epochs >= patience:\n            log.info(\"%s/%d early stop at epoch %d (best %.4f)\", arm, seed, epoch, best_f1)\n            break\n\n    save_checkpoint(run_dir / \"final.pt\", model, optimizer, scheduler, epoch, best_f1,\n                    {**meta, \"bad_epochs\": bad_epochs, \"history\": history})\n    result = {\n        \"arm\": arm,\n        \"seed\": seed,\n        \"best_val_macro_f1\": best_f1,\n        \"epochs_run\": len(history),\n        \"seconds\": round(time.time() - t0, 1),\n        \"history\": history,\n    }\n    save_json(result, run_dir / \"train_result.json\")\n    return result\n\n",
    "dm_ahr_vit/xai/__init__.py": "\"\"\"Grad-CAM and attention rollout for ViT-B/16, scored against lesion masks.\"\"\"\n\n",
    "dm_ahr_vit/xai/batched.py": "\"\"\"Batched explanation maps and scoring used by the corrected localization analysis.\"\"\"\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\nfrom .vit_xai import (\n    _patch_grid,\n    capture_attention,\n    lesion_overlap,\n    pointing_game,\n    salient_in_lesion,\n    top_fraction_mask,\n    top_region_centroid_distance,\n)\n\n\ndef metric_definitions(fraction: float = 0.2) -> str:\n    \"\"\"The metric wording used in reports, generated from the function docstrings.\"\"\"\n    pct = f\"{int(round(fraction * 100))}%\"\n    fns = (lesion_overlap, salient_in_lesion, pointing_game, top_region_centroid_distance)\n    return \"\\n\".join(\"- \" + (fn.__doc__ or \"\").strip().replace(\"20%\", pct) for fn in fns)\n\n\nclass MultiLayerGradCAM:\n    \"\"\"Batched Grad-CAM at several layers from one backward pass.\n\n    ``last_ln1``: tokens after `ln_1` of the last encoder block, class token dropped, reshaped to\n    the patch grid. ``patch_grid``: output of the patch-embedding convolution (the layer used\n    by the recorded results, kept as a labelled sensitivity run).\n    \"\"\"\n\n    def __init__(self, model, names=(\"last_ln1\", \"patch_grid\")):\n        self.model = model\n        self.grid = _patch_grid(model)\n        avail = {\"last_ln1\": (model.encoder.layers[-1].ln_1, \"tokens\"),\n                 \"patch_grid\": (model.conv_proj, \"grid\")}\n        self.layers = {n: avail[n] for n in names}\n        self._handles: list = []\n        self._act: dict[str, torch.Tensor] = {}\n        self._grad: dict[str, torch.Tensor] = {}\n\n    def __enter__(self):\n        for name, (module, _kind) in self.layers.items():\n            def hook(_m, _i, out, name=name):\n                self._act[name] = out\n                if out.requires_grad:\n                    out.register_hook(lambda g, name=name: self._grad.__setitem__(name, g))\n            self._handles.append(module.register_forward_hook(hook))\n        return self\n\n    def __exit__(self, *exc):\n        for h in self._handles:\n            h.remove()\n        self._handles = []\n        return False\n\n    def _as_grid(self, t: torch.Tensor, kind: str) -> torch.Tensor:\n        if kind == \"tokens\":\n            b, _, c = t.shape\n            return t[:, 1:, :].transpose(1, 2).reshape(b, c, self.grid, self.grid)\n        return t\n\n    def __call__(self, x: torch.Tensor, class_idx: torch.Tensor | None = None):\n        \"\"\"Return ({layer: (B, g, g) maps in [0, 1]}, predicted class index per image).\"\"\"\n        self.model.eval()\n        self.model.zero_grad(set_to_none=True)\n        self._act, self._grad = {}, {}\n        x = x.detach().clone().requires_grad_(True)\n        with torch.enable_grad():\n            logits = self.model(x)\n            idx = logits.argmax(dim=1) if class_idx is None else class_idx\n            logits.gather(1, idx[:, None]).sum().backward()\n        maps = {}\n        for name, (_m, kind) in self.layers.items():\n            if name not in self._grad:\n                raise RuntimeError(f\"Grad-CAM captured no gradient at {name}\")\n            act = self._as_grid(self._act[name].detach(), kind)\n            grad = self._as_grid(self._grad[name].detach(), kind)\n            cam = F.relu((grad.mean(dim=(2, 3), keepdim=True) * act).sum(dim=1))\n            flat = cam.flatten(1)\n            flat = flat - flat.min(dim=1, keepdim=True).values\n            mx = flat.max(dim=1, keepdim=True).values\n            flat = torch.where(mx > 0, flat / mx.clamp_min(1e-12), flat)\n            maps[name] = flat.view_as(cam).cpu().numpy()\n        return maps, idx\n\n\n@torch.no_grad()\ndef attention_rollout_batch(model, x: torch.Tensor, discard_ratio: float = 0.9) -> np.ndarray:\n    \"\"\"`attention_rollout` for a batch: (B, g, g) maps.\"\"\"\n    model.eval()\n    with capture_attention(model) as captured:\n        model(x)\n    if not captured:\n        raise RuntimeError(\"no attention weights captured\")\n    b, tokens = captured[0].shape[0], captured[0].shape[-1]\n    eye = torch.eye(tokens, device=captured[0].device)\n    result = eye.expand(b, tokens, tokens).clone()\n    for attn in captured:\n        a = attn.float().clone()\n        if discard_ratio > 0:\n            k = int(tokens * discard_ratio)\n            if k > 0:\n                idx = a.topk(k, dim=-1, largest=False).indices\n                a.scatter_(-1, idx, 0.0)\n        a = a / a.sum(dim=-1, keepdim=True).clamp_min(1e-9)\n        result = (0.5 * a + 0.5 * eye) @ result\n    grid = _patch_grid(model)\n    m = result[:, 0, 1:].reshape(b, grid * grid)\n    m = m - m.min(dim=1, keepdim=True).values\n    mx = m.max(dim=1, keepdim=True).values\n    m = torch.where(mx > 0, m / mx.clamp_min(1e-12), m)\n    return m.view(b, grid, grid).cpu().numpy()\n\n\ndef localization_scores(saliency: np.ndarray, mask: np.ndarray, fraction: float = 0.2) -> dict:\n    \"\"\"recall, precision, pointing and centroid distance for one map, from one top-region pass.\"\"\"\n    m = mask.astype(bool)\n    if not m.any():\n        return {\"recall\": float(\"nan\"), \"precision\": float(\"nan\"),\n                \"pointing\": float(\"nan\"), \"centroid\": float(\"nan\")}\n    top = top_fraction_mask(saliency, fraction)\n    inter = float((top & m).sum())\n    ty, tx = np.nonzero(top)\n    ly, lx = np.nonzero(m)\n    r, c = divmod(int(np.argmax(saliency)), saliency.shape[1])\n    return {\"recall\": inter / float(m.sum()), \"precision\": inter / float(top.sum()),\n            \"pointing\": float(m[r, c]),\n            \"centroid\": float(np.hypot(tx.mean() - lx.mean(), ty.mean() - ly.mean())\n                              / saliency.shape[1])}\n\n\ndef random_map_baselines(mask: np.ndarray, seed: int, draws: int = 100, fraction: float = 0.2,\n                         grid: int = 14, device: str = \"cpu\") -> dict:\n    \"\"\"Mean recall, precision and pointing of random maps over `draws` draws: white-noise pixel\n    maps, and random patch-grid maps bilinearly upsampled the way the real maps are.\"\"\"\n    kinds = (\"pixel\", \"patch\")\n    keys = (\"recall\", \"precision\", \"pointing\")\n    m = torch.from_numpy(mask.astype(bool)).to(device).flatten()\n    if not bool(m.any()):\n        return {f\"rand_{k}_{q}\": float(\"nan\") for k in kinds for q in keys}\n    n, size = m.numel(), mask.shape[0]\n    k_top = max(1, int(round(n * fraction)))\n    gen = torch.Generator(device=\"cpu\").manual_seed(int(seed))\n    out = {}\n    for kind in kinds:\n        if kind == \"pixel\":\n            sal = torch.rand(draws, n, generator=gen).to(device)\n        else:\n            g = torch.rand(draws, 1, grid, grid, generator=gen).to(device)\n            sal = F.interpolate(g, size=(size, size), mode=\"bilinear\",\n                                align_corners=False).reshape(draws, n)\n        top = torch.zeros(draws, n, dtype=torch.bool, device=device)\n        top.scatter_(1, sal.topk(k_top, dim=1).indices, True)\n        inter = (top & m).sum(dim=1).float()\n        out[f\"rand_{kind}_recall\"] = float((inter / m.sum()).mean())\n        out[f\"rand_{kind}_precision\"] = float((inter / k_top).mean())\n        out[f\"rand_{kind}_pointing\"] = float(m[sal.argmax(dim=1)].float().mean())\n    return out\n",
    "dm_ahr_vit/xai/geometry.py": "\"\"\"The one place that defines the model's view of an image and its lesion mask.\n\nThe ViT never sees the stored photo: it sees the photo resized so the shorter side is\n``int(size * 1.14)`` and then centre-cropped to ``size`` x ``size``. Saliency maps live in that\ncropped frame, so a lesion mask has to be carried through the identical resize and crop before\nit is compared with a map. The classifier's evaluation transform (``data.dataset``) and every\nexplanation script call the functions below, so the two cannot drift apart.\n\nStored image sizes differ by arm AND by image: the original and DullRazor arms keep the 600 x 450\nphoto; in the A*, C, Creleased and Cv2 arms the hair-present images are stored at 128 x 128 (an\naspect-ratio squash) while hair-absent images pass through unchanged at 600 x 450 (see\n`scripts/02_hair_pipeline.py`). The geometry therefore reads each image's stored size from the file.\nMasks are stored at 600 x 450 and are squashed to that size first.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom torchvision import transforms\nfrom torchvision.transforms import InterpolationMode\n\nEVAL_SCALE = 1.14\nFULL_SIZE = (600, 450)\nSQUASHED_SIZE = (128, 128)\nARM_STORED_SIZES: dict[str, frozenset[tuple[int, int]]] = {\n    \"A\": frozenset({FULL_SIZE}), \"B\": frozenset({FULL_SIZE}),\n    \"Astar\": frozenset({FULL_SIZE, SQUASHED_SIZE}), \"C\": frozenset({FULL_SIZE, SQUASHED_SIZE}),\n    \"Creleased\": frozenset({FULL_SIZE, SQUASHED_SIZE}), \"Cv2\": frozenset({FULL_SIZE, SQUASHED_SIZE}),\n}\nFRAME_VERSION = \"model_frame_v2\"\n\n\ndef eval_resize_size(size: int = 224) -> int:\n    return int(size * EVAL_SCALE)\n\n\ndef eval_ops(size: int = 224, interpolation=InterpolationMode.BILINEAR) -> list:\n    \"\"\"The resize and centre-crop of the classifier's evaluation transform.\"\"\"\n    return [transforms.Resize(eval_resize_size(size), interpolation=interpolation),\n            transforms.CenterCrop(size)]\n\n\ndef to_model_frame(img: Image.Image, size: int = 224, is_mask: bool = False) -> Image.Image:\n    \"\"\"Carry a PIL image (or mask, with nearest-neighbour resampling) into the model's frame.\"\"\"\n    mode = InterpolationMode.NEAREST if is_mask else InterpolationMode.BILINEAR\n    for op in eval_ops(size, mode):\n        img = op(img)\n    return img\n\n\ndef mask_to_model_frame(mask: Image.Image, stored_size_wh: tuple[int, int] | None,\n                        size: int = 224) -> np.ndarray:\n    \"\"\"Binary {0,1} lesion mask in the model's frame, for an arm whose images are stored at\n    ``stored_size_wh``. ``None`` means the mask is already at the stored size.\"\"\"\n    m = mask.convert(\"L\")\n    if stored_size_wh is not None and m.size != tuple(stored_size_wh):\n        m = m.resize(tuple(int(v) for v in stored_size_wh), Image.Resampling.NEAREST)\n    m = to_model_frame(m, size, is_mask=True)\n    return (np.asarray(m) > 127).astype(np.uint8)\n\n\ndef shift_zero_fill(a, dx: int, dy: int):\n    \"\"\"Translate an (..., H, W) numpy array or tensor by (dx, dy) pixels, filling with zeros.\"\"\"\n    is_t = isinstance(a, torch.Tensor)\n    out = torch.zeros_like(a) if is_t else np.zeros_like(a)\n    h, w = a.shape[-2], a.shape[-1]\n    ys, yd = (slice(0, h - dy), slice(dy, h)) if dy >= 0 else (slice(-dy, h), slice(0, h + dy))\n    xs, xd = (slice(0, w - dx), slice(dx, w)) if dx >= 0 else (slice(-dx, w), slice(0, w + dx))\n    if abs(dx) < w and abs(dy) < h:\n        out[..., yd, xd] = a[..., ys, xs]\n    return out\n",
    "dm_ahr_vit/xai/vit_xai.py": "\"\"\"Explainability maps for torchvision ViT-B/16.\n\nGrad-CAM is taken on `conv_proj`, whose output is the (B, 768, 14, 14) patch\ngrid. Attention rollout multiplies identity-blended attention matrices captured\nfrom the real self-attention modules.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport contextlib\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\n\ndef _patch_grid(model) -> int:\n    size = model.image_size if isinstance(model.image_size, int) else model.image_size[0]\n    return size // model.patch_size\n\n\nclass GradCAM:\n    \"\"\"Grad-CAM on the patch-embedding grid (conv_proj).\"\"\"\n\n    def __init__(self, model, layer=None):\n        self.model = model\n        self.layer = layer if layer is not None else model.conv_proj\n        self.activations: torch.Tensor | None = None\n        self.gradients: torch.Tensor | None = None\n        self._handle = None\n\n    def __enter__(self):\n        def fwd_hook(_module, _inp, out):\n            self.activations = out\n            if out.requires_grad:\n                out.register_hook(self._save_grad)\n\n        self._handle = self.layer.register_forward_hook(fwd_hook)\n        return self\n\n    def __exit__(self, *exc):\n        if self._handle is not None:\n            self._handle.remove()\n            self._handle = None\n        return False\n\n    def _save_grad(self, grad):\n        self.gradients = grad\n\n    def __call__(self, x: torch.Tensor, class_idx: int | None = None) -> np.ndarray:\n        \"\"\"Return a normalized (grid, grid) CAM for one image tensor (1,3,H,W).\"\"\"\n        self.model.eval()\n        self.model.zero_grad(set_to_none=True)\n        # Grad-CAM needs a gradient path back to the patch grid, so the input\n        # must require grad even though the model is in eval mode.\n        x = x.detach().clone().requires_grad_(True)\n        with torch.enable_grad():\n            logits = self.model(x)\n            if class_idx is None:\n                class_idx = int(logits.argmax(dim=1))\n            score = logits[0, class_idx]\n            score.backward(retain_graph=False)\n        if self.gradients is None:\n            raise RuntimeError(\"Grad-CAM captured no gradients on conv_proj\")\n        act = self.activations.detach()[0]           # (768, g, g)\n        grad = self.gradients.detach()[0]            # (768, g, g)\n        weights = grad.mean(dim=(1, 2), keepdim=True)\n        cam = F.relu((weights * act).sum(dim=0))\n        cam = cam - cam.min()\n        denom = cam.max()\n        if float(denom) > 0:\n            cam = cam / denom\n        return cam.cpu().numpy()\n\n\n@contextlib.contextmanager\ndef capture_attention(model):\n    \"\"\"Capture averaged attention weights from every block's self_attn module.\"\"\"\n    captured: list[torch.Tensor] = []\n    originals = []\n\n    for block in model.encoder.layers:\n        # torchvision names this module `self_attention`; timm-style code uses\n        # `self_attn`. Accept either so the hook does not depend on the fork.\n        attn = getattr(block, \"self_attention\", None) or getattr(block, \"self_attn\", None)\n        if attn is None:\n            raise AttributeError(\"encoder block exposes no self-attention module\")\n        originals.append((attn, attn.forward))\n\n        def make(original_forward, store):\n            def forward(query, key, value, *args, **kwargs):\n                kwargs[\"need_weights\"] = True\n                kwargs[\"average_attn_weights\"] = True\n                out, weights = original_forward(query, key, value, *args, **kwargs)\n                store.append(weights.detach())\n                # torchvision unpacks `x, _ = self_attention(...)`; keep that shape.\n                return out, None\n\n            return forward\n\n        attn.forward = make(attn.forward, captured)\n\n    try:\n        yield captured\n    finally:\n        for module, original in originals:\n            module.forward = original\n\n\n@torch.no_grad()\ndef attention_rollout(model, x: torch.Tensor, discard_ratio: float = 0.9) -> np.ndarray:\n    \"\"\"Identity-blended attention rollout, returned as a (grid, grid) map.\"\"\"\n    model.eval()\n    with capture_attention(model) as captured:\n        model(x)\n    if not captured:\n        raise RuntimeError(\"no attention weights captured\")\n\n    tokens = captured[0].shape[-1]\n    result = torch.eye(tokens)\n    for attn in captured:\n        a = attn[0].float().cpu()\n        if discard_ratio > 0:\n            flat = a.flatten(1)\n            k = int(flat.shape[1] * discard_ratio)\n            if k > 0:\n                idx = flat.topk(k, dim=1, largest=False).indices\n                flat.scatter_(1, idx, 0.0)\n                a = flat.view_as(a)\n        a = a / a.sum(dim=-1, keepdim=True).clamp_min(1e-9)\n        blended = 0.5 * a + 0.5 * torch.eye(tokens)\n        result = blended @ result\n\n    cls_attention = result[0, 1:]\n    grid = _patch_grid(model)\n    mask = cls_attention.reshape(grid, grid).numpy()\n    mask = mask - mask.min()\n    denom = mask.max()\n    if denom > 0:\n        mask = mask / denom\n    return mask\n\n\ndef upsample_map(cam: np.ndarray, size: int = 224) -> np.ndarray:\n    t = torch.from_numpy(cam).float()[None, None]\n    return F.interpolate(t, size=(size, size), mode=\"bilinear\",\n                         align_corners=False)[0, 0].numpy()\n\n\ndef top_fraction_mask(saliency: np.ndarray, fraction: float = 0.2) -> np.ndarray:\n    flat = saliency.ravel()\n    k = max(1, int(round(flat.size * fraction)))\n    idx = np.argpartition(-flat, k - 1)[:k]\n    out = np.zeros(flat.size, dtype=bool)\n    out[idx] = True\n    return out.reshape(saliency.shape)\n\n\ndef lesion_overlap(saliency: np.ndarray, mask: np.ndarray, fraction: float = 0.2) -> float:\n    \"\"\"Overlap (mask recall): the share of the lesion covered by the top-20% most salient pixels.\"\"\"\n    top = top_fraction_mask(saliency, fraction)\n    lesion_area = float(mask.sum())\n    if lesion_area == 0:\n        return float(\"nan\")\n    return float((top & mask.astype(bool)).sum() / lesion_area)\n\n\ndef salient_in_lesion(saliency: np.ndarray, mask: np.ndarray, fraction: float = 0.2) -> float:\n    \"\"\"Precision: the share of the top-20% most salient pixels that fall inside the lesion.\"\"\"\n    if mask.sum() == 0:\n        return float(\"nan\")\n    top = top_fraction_mask(saliency, fraction)\n    return float((top & mask.astype(bool)).sum() / top.sum())\n\n\ndef top_region_centroid_distance(saliency: np.ndarray, mask: np.ndarray, fraction: float = 0.2) -> float:\n    \"\"\"Centroid distance: between the centre of the top-20% region and the lesion centre, in image widths.\"\"\"\n    if mask.sum() == 0:\n        return float(\"nan\")\n    top = top_fraction_mask(saliency, fraction)\n    ty, tx = np.nonzero(top)\n    ly, lx = np.nonzero(mask.astype(bool))\n    d = np.hypot(tx.mean() - lx.mean(), ty.mean() - ly.mean())\n    return float(d / saliency.shape[1])\n\n\ndef pointing_game(saliency: np.ndarray, mask: np.ndarray, tolerance: int = 0) -> float:\n    \"\"\"Pointing: 1 if the single most salient pixel falls inside the lesion, else 0.\"\"\"\n    m = mask.astype(bool)\n    if tolerance > 0:\n        k = 2 * tolerance + 1\n        m = (\n            F.max_pool2d(\n                torch.from_numpy(m.astype(np.float32))[None, None], k, stride=1, padding=tolerance\n            )[0, 0].numpy()\n            > 0\n        )\n    if m.sum() == 0:\n        return float(\"nan\")\n    idx = int(np.argmax(saliency))\n    r, c = divmod(idx, saliency.shape[1])\n    return float(bool(m[r, c]))\n\n\ndef uniform_chance_overlap(mask: np.ndarray, fraction: float = 0.2, seed: int = 0,\n                           n: int = 1) -> float:\n    rng = np.random.default_rng(seed)\n    vals = []\n    for _ in range(n):\n        sal = rng.random(mask.shape)\n        vals.append(lesion_overlap(sal, mask, fraction))\n    return float(np.nanmean(vals))\n\n\ndef gaussian_chance_map(size: int = 224, sigma_frac: float = 0.25) -> np.ndarray:\n    yy, xx = np.mgrid[0:size, 0:size]\n    c = (size - 1) / 2.0\n    sigma = size * sigma_frac\n    g = np.exp(-(((xx - c) ** 2 + (yy - c) ** 2) / (2 * sigma**2)))\n    return g / g.max()\n\n\ndef overlay_heatmap(image: np.ndarray, cam: np.ndarray, alpha: float = 0.45) -> np.ndarray:\n    import matplotlib.cm as cm\n\n    heat = cm.jet(cam)[..., :3]\n    base = image.astype(np.float32) / 255.0\n    return np.clip((1 - alpha) * base + alpha * heat, 0, 1).astype(np.float32)\n\n",
    "third_party/dmahr/model/__init__.py": "import logging\nlogger = logging.getLogger('base')\n\n\ndef create_model(opt):\n    from .model import DDPM as M\n    m = M(opt)\n    logger.info('Model [{:s}] is created.'.format(m.__class__.__name__))\n    return m\n",
    "third_party/dmahr/model/base_model.py": "import os\nimport torch\nimport torch.nn as nn\n\n\nclass BaseModel():\n    def __init__(self, opt):\n        self.opt = opt\n        self.device = torch.device(\n            'cuda' if opt['gpu_ids'] is not None else 'cpu')\n        self.begin_step = 0\n        self.begin_epoch = 0\n\n    def feed_data(self, data):\n        pass\n\n    def optimize_parameters(self):\n        pass\n\n    def get_current_visuals(self):\n        pass\n\n    def get_current_losses(self):\n        pass\n\n    def print_network(self):\n        pass\n\n    def set_device(self, x):\n        if isinstance(x, dict):\n            for key, item in x.items():\n                if item is not None:\n                    x[key] = item.to(self.device)\n        elif isinstance(x, list):\n            for item in x:\n                if item is not None:\n                    item = item.to(self.device)\n        else:\n            x = x.to(self.device)\n        return x\n\n    def get_network_description(self, network):\n        '''Get the string and total parameters of the network'''\n        if isinstance(network, nn.DataParallel):\n            network = network.module\n        s = str(network)\n        n = sum(map(lambda x: x.numel(), network.parameters()))\n        return s, n\n",
    "third_party/dmahr/model/ddpm_modules/diffusion.py": "import math\nimport torch\nfrom torch import nn, einsum\nimport torch.nn.functional as F\nfrom inspect import isfunction\nfrom functools import partial\nimport numpy as np\nfrom tqdm import tqdm\n\n\ndef _warmup_beta(linear_start, linear_end, n_timestep, warmup_frac):\n    betas = linear_end * np.ones(n_timestep, dtype=np.float64)\n    warmup_time = int(n_timestep * warmup_frac)\n    betas[:warmup_time] = np.linspace(\n        linear_start, linear_end, warmup_time, dtype=np.float64)\n    return betas\n\n\ndef make_beta_schedule(schedule, n_timestep, linear_start=1e-4, linear_end=2e-2, cosine_s=8e-3):\n    if schedule == 'quad':\n        betas = np.linspace(linear_start ** 0.5, linear_end ** 0.5,\n                            n_timestep, dtype=np.float64) ** 2\n    elif schedule == 'linear':\n        betas = np.linspace(linear_start, linear_end,\n                            n_timestep, dtype=np.float64)\n    elif schedule == 'warmup10':\n        betas = _warmup_beta(linear_start, linear_end,\n                             n_timestep, 0.1)\n    elif schedule == 'warmup50':\n        betas = _warmup_beta(linear_start, linear_end,\n                             n_timestep, 0.5)\n    elif schedule == 'const':\n        betas = linear_end * np.ones(n_timestep, dtype=np.float64)\n    elif schedule == 'jsd':  # 1/T, 1/(T-1), 1/(T-2), ..., 1\n        betas = 1. / np.linspace(n_timestep,\n                                 1, n_timestep, dtype=np.float64)\n    elif schedule == \"cosine\":\n        timesteps = (\n            torch.arange(n_timestep + 1, dtype=torch.float64) /\n            n_timestep + cosine_s\n        )\n        alphas = timesteps / (1 + cosine_s) * math.pi / 2\n        alphas = torch.cos(alphas).pow(2)\n        alphas = alphas / alphas[0]\n        betas = 1 - alphas[1:] / alphas[:-1]\n        betas = betas.clamp(max=0.999)\n    else:\n        raise NotImplementedError(schedule)\n    return betas\n\n\n# gaussian diffusion trainer class\n\ndef exists(x):\n    return x is not None\n\n\ndef default(val, d):\n    if exists(val):\n        return val\n    return d() if isfunction(d) else d\n\n\ndef extract(a, t, x_shape):\n    b, *_ = t.shape\n    out = a.gather(-1, t)\n    return out.reshape(b, *((1,) * (len(x_shape) - 1)))\n\n\ndef noise_like(shape, device, repeat=False):\n    def repeat_noise(): return torch.randn(\n        (1, *shape[1:]), device=device).repeat(shape[0], *((1,) * (len(shape) - 1)))\n\n    def noise(): return torch.randn(shape, device=device)\n    return repeat_noise() if repeat else noise()\n\n\nclass GaussianDiffusion(nn.Module):\n    def __init__(\n        self,\n        denoise_fn,\n        image_size,\n        channels=3,\n        loss_type='l1',\n        conditional=True,\n        schedule_opt=None\n    ):\n        super().__init__()\n        self.channels = channels\n        self.image_size = image_size\n        self.denoise_fn = denoise_fn\n        self.conditional = conditional\n        self.loss_type = loss_type\n        if schedule_opt is not None:\n            pass\n            # self.set_new_noise_schedule(schedule_opt)\n\n    def set_loss(self, device):\n        if self.loss_type == 'l1':\n            self.loss_func = nn.L1Loss(reduction='sum').to(device)\n        elif self.loss_type == 'l2':\n            self.loss_func = nn.MSELoss(reduction='sum').to(device)\n        else:\n            raise NotImplementedError()\n\n    def set_new_noise_schedule(self, schedule_opt, device):\n        to_torch = partial(torch.tensor, dtype=torch.float32, device=device)\n        betas = make_beta_schedule(\n            schedule=schedule_opt['schedule'],\n            n_timestep=schedule_opt['n_timestep'],\n            linear_start=schedule_opt['linear_start'],\n            linear_end=schedule_opt['linear_end'])\n        betas = betas.detach().cpu().numpy() if isinstance(betas, torch.Tensor) else betas\n        alphas = 1. - betas\n        alphas_cumprod = np.cumprod(alphas, axis=0)\n        alphas_cumprod_prev = np.append(1., alphas_cumprod[:-1])\n\n        timesteps, = betas.shape\n        self.num_timesteps = int(timesteps)\n        self.register_buffer('betas', to_torch(betas))\n        self.register_buffer('alphas_cumprod', to_torch(alphas_cumprod))\n        self.register_buffer('alphas_cumprod_prev',\n                             to_torch(alphas_cumprod_prev))\n\n        # calculations for diffusion q(x_t | x_{t-1}) and others\n        self.register_buffer('sqrt_alphas_cumprod',\n                             to_torch(np.sqrt(alphas_cumprod)))\n        self.register_buffer('sqrt_one_minus_alphas_cumprod',\n                             to_torch(np.sqrt(1. - alphas_cumprod)))\n        self.register_buffer('log_one_minus_alphas_cumprod',\n                             to_torch(np.log(1. - alphas_cumprod)))\n        self.register_buffer('sqrt_recip_alphas_cumprod',\n                             to_torch(np.sqrt(1. / alphas_cumprod)))\n        self.register_buffer('sqrt_recipm1_alphas_cumprod',\n                             to_torch(np.sqrt(1. / alphas_cumprod - 1)))\n\n        # calculations for posterior q(x_{t-1} | x_t, x_0)\n        posterior_variance = betas * \\\n            (1. - alphas_cumprod_prev) / (1. - alphas_cumprod)\n        # above: equal to 1. / (1. / (1. - alpha_cumprod_tm1) + alpha_t / beta_t)\n        self.register_buffer('posterior_variance',\n                             to_torch(posterior_variance))\n        # below: log calculation clipped because the posterior variance is 0 at the beginning of the diffusion chain\n        self.register_buffer('posterior_log_variance_clipped', to_torch(\n            np.log(np.maximum(posterior_variance, 1e-20))))\n        self.register_buffer('posterior_mean_coef1', to_torch(\n            betas * np.sqrt(alphas_cumprod_prev) / (1. - alphas_cumprod)))\n        self.register_buffer('posterior_mean_coef2', to_torch(\n            (1. - alphas_cumprod_prev) * np.sqrt(alphas) / (1. - alphas_cumprod)))\n\n    def q_mean_variance(self, x_start, t):\n        mean = extract(self.sqrt_alphas_cumprod, t, x_start.shape) * x_start\n        variance = extract(1. - self.alphas_cumprod, t, x_start.shape)\n        log_variance = extract(\n            self.log_one_minus_alphas_cumprod, t, x_start.shape)\n        return mean, variance, log_variance\n\n    def predict_start_from_noise(self, x_t, t, noise):\n        return (\n            extract(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t -\n            extract(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape) * noise\n        )\n\n    def q_posterior(self, x_start, x_t, t):\n        posterior_mean = (\n            extract(self.posterior_mean_coef1, t, x_t.shape) * x_start +\n            extract(self.posterior_mean_coef2, t, x_t.shape) * x_t\n        )\n        posterior_variance = extract(self.posterior_variance, t, x_t.shape)\n        posterior_log_variance_clipped = extract(\n            self.posterior_log_variance_clipped, t, x_t.shape)\n        return posterior_mean, posterior_variance, posterior_log_variance_clipped\n\n    def p_mean_variance(self, x, t, clip_denoised: bool, condition_x=None):\n        if condition_x is not None:\n            x_recon = self.predict_start_from_noise(\n                x, t=t, noise=self.denoise_fn(torch.cat([condition_x, x], dim=1), t))\n        else:\n            x_recon = self.predict_start_from_noise(\n                x, t=t, noise=self.denoise_fn(x, t))\n\n        if clip_denoised:\n            x_recon.clamp_(-1., 1.)\n\n        model_mean, posterior_variance, posterior_log_variance = self.q_posterior(\n            x_start=x_recon, x_t=x, t=t)\n        return model_mean, posterior_variance, posterior_log_variance\n\n    @torch.no_grad()\n    def p_sample(self, x, t, clip_denoised=True, repeat_noise=False, condition_x=None):\n        b, *_, device = *x.shape, x.device\n        model_mean, _, model_log_variance = self.p_mean_variance(\n            x=x, t=t, clip_denoised=clip_denoised, condition_x=condition_x)\n        noise = noise_like(x.shape, device, repeat_noise)\n        # no noise when t == 0\n        nonzero_mask = (1 - (t == 0).float()).reshape(b,\n                                                      *((1,) * (len(x.shape) - 1)))\n        return model_mean + nonzero_mask * (0.5 * model_log_variance).exp() * noise\n\n    @torch.no_grad()\n    def p_sample_loop(self, x_in, continous=False):\n        device = self.betas.device\n        sample_inter = (1 | (self.num_timesteps//10))\n\n        if not self.conditional:\n            shape = x_in\n            b = shape[0]\n            img = torch.randn(shape, device=device)\n            ret_img = img\n            for i in tqdm(reversed(range(0, self.num_timesteps)), desc='sampling loop time step', total=self.num_timesteps):\n                img = self.p_sample(img, torch.full(\n                    (b,), i, device=device, dtype=torch.long))\n                if i % sample_inter == 0:\n                    ret_img = torch.cat([ret_img, img], dim=0)\n            return img\n        else:\n            x = x_in\n            shape = x.shape\n            b = shape[0]\n            img = torch.randn(shape, device=device)\n            ret_img = x\n            for i in tqdm(reversed(range(0, self.num_timesteps)), desc='sampling loop time step', total=self.num_timesteps):\n                img = self.p_sample(img, torch.full(\n                    (b,), i, device=device, dtype=torch.long), condition_x=x)\n                if i % sample_inter == 0:\n                    ret_img = torch.cat([ret_img, img], dim=0)\n        if continous:\n            return ret_img\n        else:\n            return ret_img[-1]\n\n    @torch.no_grad()\n    def sample(self, batch_size=1, continous=False):\n        image_size = self.image_size\n        channels = self.channels\n        return self.p_sample_loop((batch_size, channels, image_size, image_size), continous)\n\n    @torch.no_grad()\n    def super_resolution(self, x_in, continous=False):\n        return self.p_sample_loop(x_in, continous)\n\n    @torch.no_grad()\n    def interpolate(self, x1, x2, t=None, lam=0.5):\n        b, *_, device = *x1.shape, x1.device\n        t = default(t, self.num_timesteps - 1)\n\n        assert x1.shape == x2.shape\n\n        t_batched = torch.stack([torch.tensor(t, device=device)] * b)\n        xt1, xt2 = map(lambda x: self.q_sample(x, t=t_batched), (x1, x2))\n\n        img = (1 - lam) * xt1 + lam * xt2\n        for i in tqdm(reversed(range(0, t)), desc='interpolation sample time step', total=t):\n            img = self.p_sample(img, torch.full(\n                (b,), i, device=device, dtype=torch.long))\n\n        return img\n\n    def q_sample(self, x_start, t, noise=None):\n        noise = default(noise, lambda: torch.randn_like(x_start))\n\n        # fix gama\n        return (\n            extract(self.sqrt_alphas_cumprod, t, x_start.shape) * x_start +\n            extract(self.sqrt_one_minus_alphas_cumprod,\n                    t, x_start.shape) * noise\n        )\n        # random gama\n        # x_shape = x_start.shape\n        # l = self.alphas_cumprod .gather(-1, t)\n        # r = self.alphas_cumprod .gather(-1, t+1)\n        # gama = (r - l) * torch.rand(0, 1) + l\n        # gama = gama.reshape(t.shape[0], *((1,) * (len(x_shape) - 1)))\n        # return (\n        #     nq.sqrt(gama) * x_start + nq.sqrt(1-gama)* noise\n        # )\n\n    def p_losses(self, x_in, noise=None):\n        x_start = x_in['HR']\n        [b, c, h, w] = x_start.shape\n        t = torch.randint(0, self.num_timesteps, (b,),\n                          device=x_start.device).long()\n\n        noise = default(noise, lambda: torch.randn_like(x_start))\n        x_noisy = self.q_sample(x_start=x_start, t=t, noise=noise)\n\n        if not self.conditional:\n            x_recon = self.denoise_fn(x_noisy, t)\n        else:\n            x_recon = self.denoise_fn(\n                torch.cat([x_in['SR'], x_noisy], dim=1), t)\n        loss = self.loss_func(noise, x_recon)\n\n        return loss\n\n    def forward(self, x, *args, **kwargs):\n        return self.p_losses(x, *args, **kwargs)\n",
    "third_party/dmahr/model/ddpm_modules/unet.py": "import math\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom inspect import isfunction\n\ndef exists(x):\n    return x is not None\n\n\ndef default(val, d):\n    if exists(val):\n        return val\n    return d() if isfunction(d) else d\n\n# model\n\n\nclass TimeEmbedding(nn.Module):\n    def __init__(self, dim):\n        super().__init__()\n        self.dim = dim\n        inv_freq = torch.exp(\n            torch.arange(0, dim, 2, dtype=torch.float32) *\n            (-math.log(10000) / dim)\n        )\n        self.register_buffer(\"inv_freq\", inv_freq)\n\n    def forward(self, input):\n        shape = input.shape\n        sinusoid_in = torch.ger(input.view(-1).float(), self.inv_freq)\n        pos_emb = torch.cat([sinusoid_in.sin(), sinusoid_in.cos()], dim=-1)\n        pos_emb = pos_emb.view(*shape, self.dim)\n        return pos_emb\n\n\nclass Swish(nn.Module):\n    def forward(self, x):\n        return x * torch.sigmoid(x)\n\n\nclass Upsample(nn.Module):\n    def __init__(self, dim):\n        super().__init__()\n        self.up = nn.Upsample(scale_factor=2, mode=\"nearest\")\n        self.conv = nn.Conv2d(dim, dim, 3, padding=1)\n\n    def forward(self, x):\n        return self.conv(self.up(x))\n\n\nclass Downsample(nn.Module):\n    def __init__(self, dim):\n        super().__init__()\n        self.conv = nn.Conv2d(dim, dim, 3, 2, 1)\n\n    def forward(self, x):\n        return self.conv(x)\n\n\n# building block modules\n\n\nclass Block(nn.Module):\n    def __init__(self, dim, dim_out, groups=32, dropout=0):\n        super().__init__()\n        self.block = nn.Sequential(\n            nn.GroupNorm(groups, dim),\n            Swish(),\n            nn.Dropout(dropout) if dropout != 0 else nn.Identity(),\n            nn.Conv2d(dim, dim_out, 3, padding=1)\n        )\n\n    def forward(self, x):\n        return self.block(x)\n\n\nclass ResnetBlock(nn.Module):\n    def __init__(self, dim, dim_out, time_emb_dim=None, dropout=0, norm_groups=32):\n        super().__init__()\n        self.mlp = nn.Sequential(\n            Swish(),\n            nn.Linear(time_emb_dim, dim_out)\n        ) if exists(time_emb_dim) else None\n\n        self.block1 = Block(dim, dim_out, groups=norm_groups)\n        self.block2 = Block(dim_out, dim_out, groups=norm_groups, dropout=dropout)\n        self.res_conv = nn.Conv2d(\n            dim, dim_out, 1) if dim != dim_out else nn.Identity()\n\n    def forward(self, x, time_emb):\n        h = self.block1(x)\n        if exists(self.mlp):\n            h += self.mlp(time_emb)[:, :, None, None]\n        h = self.block2(h)\n        return h + self.res_conv(x)\n\n\nclass SelfAttention(nn.Module):\n    def __init__(self, in_channel, n_head=1, norm_groups=32):\n        super().__init__()\n\n        self.n_head = n_head\n\n        self.norm = nn.GroupNorm(norm_groups, in_channel)\n        self.qkv = nn.Conv2d(in_channel, in_channel * 3, 1, bias=False)\n        self.out = nn.Conv2d(in_channel, in_channel, 1)\n\n    def forward(self, input):\n        batch, channel, height, width = input.shape\n        n_head = self.n_head\n        head_dim = channel // n_head\n\n        norm = self.norm(input)\n        qkv = self.qkv(norm).view(batch, n_head, head_dim * 3, height, width)\n        query, key, value = qkv.chunk(3, dim=2)  # bhdyx\n\n        attn = torch.einsum(\n            \"bnchw, bncyx -> bnhwyx\", query, key\n        ).contiguous() / math.sqrt(channel)\n        attn = attn.view(batch, n_head, height, width, -1)\n        attn = torch.softmax(attn, -1)\n        attn = attn.view(batch, n_head, height, width, height, width)\n\n        out = torch.einsum(\"bnhwyx, bncyx -> bnchw\", attn, value).contiguous()\n        out = self.out(out.view(batch, channel, height, width))\n\n        return out + input\n\n\nclass ResnetBlocWithAttn(nn.Module):\n    def __init__(self, dim, dim_out, *, time_emb_dim=None, norm_groups=32, dropout=0, with_attn=False):\n        super().__init__()\n        self.with_attn = with_attn\n        self.res_block = ResnetBlock(\n            dim, dim_out, time_emb_dim, norm_groups=norm_groups, dropout=dropout)\n        if with_attn:\n            self.attn = SelfAttention(dim_out, norm_groups=norm_groups)\n\n    def forward(self, x, time_emb):\n        x = self.res_block(x, time_emb)\n        if(self.with_attn):\n            x = self.attn(x)\n        return x\n\n\nclass UNet(nn.Module):\n    def __init__(\n        self,\n        in_channel=6,\n        out_channel=3,\n        inner_channel=32,\n        norm_groups=32,\n        channel_mults=(1, 2, 4, 8, 8),\n        attn_res=(8),\n        res_blocks=3,\n        dropout=0,\n        with_time_emb=True,\n        image_size=128\n    ):\n        super().__init__()\n\n        if with_time_emb:\n            time_dim = inner_channel\n            self.time_mlp = nn.Sequential(\n                TimeEmbedding(inner_channel),\n                nn.Linear(inner_channel, inner_channel * 4),\n                Swish(),\n                nn.Linear(inner_channel * 4, inner_channel)\n            )\n        else:\n            time_dim = None\n            self.time_mlp = None\n\n        num_mults = len(channel_mults)\n        pre_channel = inner_channel\n        feat_channels = [pre_channel]\n        now_res = image_size\n        downs = [nn.Conv2d(in_channel, inner_channel,\n                           kernel_size=3, padding=1)]\n        for ind in range(num_mults):\n            is_last = (ind == num_mults - 1)\n            use_attn = (now_res in attn_res)\n            channel_mult = inner_channel * channel_mults[ind]\n            for _ in range(0, res_blocks):\n                downs.append(ResnetBlocWithAttn(\n                    pre_channel, channel_mult, time_emb_dim=time_dim, norm_groups=norm_groups, dropout=dropout, with_attn=use_attn))\n                feat_channels.append(channel_mult)\n                pre_channel = channel_mult\n            if not is_last:\n                downs.append(Downsample(pre_channel))\n                feat_channels.append(pre_channel)\n                now_res = now_res//2\n        self.downs = nn.ModuleList(downs)\n\n        self.mid = nn.ModuleList([\n            ResnetBlocWithAttn(pre_channel, pre_channel, time_emb_dim=time_dim, norm_groups=norm_groups,\n                               dropout=dropout, with_attn=True),\n            ResnetBlocWithAttn(pre_channel, pre_channel, time_emb_dim=time_dim, norm_groups=norm_groups, \n                                dropout=dropout, with_attn=False)\n        ])\n\n        ups = []\n        for ind in reversed(range(num_mults)):\n            is_last = (ind < 1)\n            use_attn = (now_res in attn_res)\n            channel_mult = inner_channel * channel_mults[ind]\n            for _ in range(0, res_blocks+1):\n                ups.append(ResnetBlocWithAttn(\n                    pre_channel+feat_channels.pop(), channel_mult, time_emb_dim=time_dim, dropout=dropout, norm_groups=norm_groups, with_attn=use_attn))\n                pre_channel = channel_mult\n            if not is_last:\n                ups.append(Upsample(pre_channel))\n                now_res = now_res*2\n\n        self.ups = nn.ModuleList(ups)\n\n        self.final_conv = Block(pre_channel, default(out_channel, in_channel), groups=norm_groups)\n\n    def forward(self, x, time):\n        t = self.time_mlp(time) if exists(self.time_mlp) else None\n\n        feats = []\n        for layer in self.downs:\n            if isinstance(layer, ResnetBlocWithAttn):\n                x = layer(x, t)\n            else:\n                x = layer(x)\n            feats.append(x)\n\n        for layer in self.mid:\n            if isinstance(layer, ResnetBlocWithAttn):\n                x = layer(x, t)\n            else:\n                x = layer(x)\n\n        for layer in self.ups:\n            if isinstance(layer, ResnetBlocWithAttn):\n                x = layer(torch.cat((x, feats.pop()), dim=1), t)\n            else:\n                x = layer(x)\n\n        return self.final_conv(x)\n",
    "third_party/dmahr/model/model.py": "import logging\nfrom collections import OrderedDict\n\nimport torch\nimport torch.nn as nn\nimport os\nimport model.networks as networks\nfrom .base_model import BaseModel\nlogger = logging.getLogger('base')\n\n\nclass DDPM(BaseModel):\n    def __init__(self, opt):\n        super(DDPM, self).__init__(opt)\n        # define network and load pretrained models\n        self.netG = self.set_device(networks.define_G(opt))\n        self.schedule_phase = None\n\n        # set loss and load resume state\n        self.set_loss()\n        self.set_new_noise_schedule(\n            opt['model']['beta_schedule']['train'], schedule_phase='train')\n        if self.opt['phase'] == 'train':\n            self.netG.train()\n            # find the parameters to optimize\n            if opt['model']['finetune_norm']:\n                optim_params = []\n                for k, v in self.netG.named_parameters():\n                    v.requires_grad = False\n                    if k.find('transformer') >= 0:\n                        v.requires_grad = True\n                        v.data.zero_()\n                        optim_params.append(v)\n                        logger.info(\n                            'Params [{:s}] initialized to 0 and will optimize.'.format(k))\n            else:\n                optim_params = list(self.netG.parameters())\n\n            self.optG = torch.optim.Adam(\n                optim_params, lr=opt['train'][\"optimizer\"][\"lr\"])\n            self.log_dict = OrderedDict()\n        self.load_network()\n        self.print_network()\n\n    def feed_data(self, data):\n        self.data = self.set_device(data)\n\n    def optimize_parameters(self):\n        self.optG.zero_grad()\n        l_pix = self.netG(self.data)\n        # need to average in multi-gpu\n        b, c, h, w = self.data['HR'].shape\n        l_pix = l_pix.sum()/int(b*c*h*w)\n        l_pix.backward()\n        self.optG.step()\n\n        # set log\n        self.log_dict['l_pix'] = l_pix.item()\n\n    def test(self, continous=False):\n        self.netG.eval()\n        with torch.no_grad():\n            if isinstance(self.netG, nn.DataParallel):\n                self.SR = self.netG.module.super_resolution(\n                    self.data['SR'], continous)\n            else:\n                self.SR = self.netG.super_resolution(\n                    self.data['SR'], continous)\n        self.netG.train()\n\n    def sample(self, batch_size=1, continous=False):\n        self.netG.eval()\n        with torch.no_grad():\n            if isinstance(self.netG, nn.DataParallel):\n                self.SR = self.netG.module.sample(batch_size, continous)\n            else:\n                self.SR = self.netG.sample(batch_size, continous)\n        self.netG.train()\n\n    def set_loss(self):\n        if isinstance(self.netG, nn.DataParallel):\n            self.netG.module.set_loss(self.device)\n        else:\n            self.netG.set_loss(self.device)\n\n    def set_new_noise_schedule(self, schedule_opt, schedule_phase='train'):\n        if self.schedule_phase is None or self.schedule_phase != schedule_phase:\n            self.schedule_phase = schedule_phase\n            if isinstance(self.netG, nn.DataParallel):\n                self.netG.module.set_new_noise_schedule(\n                    schedule_opt, self.device)\n            else:\n                self.netG.set_new_noise_schedule(schedule_opt, self.device)\n\n    def get_current_log(self):\n        return self.log_dict\n\n    def get_current_visuals(self, need_LR=True, sample=False):\n        out_dict = OrderedDict()\n        if sample:\n            out_dict['SAM'] = self.SR.detach().float().cpu()\n        else:\n            out_dict['SR'] = self.SR.detach().float().cpu()\n            out_dict['INF'] = self.data['SR'].detach().float().cpu()\n            out_dict['HR'] = self.data['HR'].detach().float().cpu()\n            if need_LR and 'LR' in self.data:\n                out_dict['LR'] = self.data['LR'].detach().float().cpu()\n            else:\n                out_dict['LR'] = out_dict['INF']\n        return out_dict\n\n    def print_network(self):\n        s, n = self.get_network_description(self.netG)\n        if isinstance(self.netG, nn.DataParallel):\n            net_struc_str = '{} - {}'.format(self.netG.__class__.__name__,\n                                             self.netG.module.__class__.__name__)\n        else:\n            net_struc_str = '{}'.format(self.netG.__class__.__name__)\n\n        logger.info(\n            'Network G structure: {}, with parameters: {:,d}'.format(net_struc_str, n))\n        logger.info(s)\n\n    def save_network(self, epoch, iter_step):\n        gen_path = os.path.join(\n            self.opt['path']['checkpoint'], 'I{}_E{}_gen.pth'.format(iter_step, epoch))\n        opt_path = os.path.join(\n            self.opt['path']['checkpoint'], 'I{}_E{}_opt.pth'.format(iter_step, epoch))\n        # gen\n        network = self.netG\n        if isinstance(self.netG, nn.DataParallel):\n            network = network.module\n        state_dict = network.state_dict()\n        for key, param in state_dict.items():\n            state_dict[key] = param.cpu()\n        torch.save(state_dict, gen_path)\n        # opt\n        opt_state = {'epoch': epoch, 'iter': iter_step,\n                     'scheduler': None, 'optimizer': None}\n        opt_state['optimizer'] = self.optG.state_dict()\n        torch.save(opt_state, opt_path)\n\n        logger.info(\n            'Saved model in [{:s}] ...'.format(gen_path))\n\n    def load_network(self):\n        load_path = self.opt['path']['resume_state']\n        if load_path is not None:\n            logger.info(\n                'Loading pretrained model for G [{:s}] ...'.format(load_path))\n            gen_path = '{}_gen.pth'.format(load_path)\n            opt_path = '{}_opt.pth'.format(load_path)\n            # gen\n            network = self.netG\n            if isinstance(self.netG, nn.DataParallel):\n                network = network.module\n            network.load_state_dict(torch.load(\n                gen_path), strict=(not self.opt['model']['finetune_norm']))\n            # network.load_state_dict(torch.load(\n            #     gen_path), strict=False)\n            if self.opt['phase'] == 'train':\n                # optimizer\n                opt = torch.load(opt_path)\n                self.optG.load_state_dict(opt['optimizer'])\n                self.begin_step = opt['iter']\n                self.begin_epoch = opt['epoch']\n",
    "third_party/dmahr/model/networks.py": "import functools\nimport logging\nimport torch\nimport torch.nn as nn\nfrom torch.nn import init\nfrom torch.nn import modules\nlogger = logging.getLogger('base')\n####################\n# initialize\n####################\n\n\ndef weights_init_normal(m, std=0.02):\n    classname = m.__class__.__name__\n    if classname.find('Conv') != -1:\n        init.normal_(m.weight.data, 0.0, std)\n        if m.bias is not None:\n            m.bias.data.zero_()\n    elif classname.find('Linear') != -1:\n        init.normal_(m.weight.data, 0.0, std)\n        if m.bias is not None:\n            m.bias.data.zero_()\n    elif classname.find('BatchNorm2d') != -1:\n        init.normal_(m.weight.data, 1.0, std)  # BN also uses norm\n        init.constant_(m.bias.data, 0.0)\n\n\ndef weights_init_kaiming(m, scale=1):\n    classname = m.__class__.__name__\n    if classname.find('Conv2d') != -1:\n        init.kaiming_normal_(m.weight.data, a=0, mode='fan_in')\n        m.weight.data *= scale\n        if m.bias is not None:\n            m.bias.data.zero_()\n    elif classname.find('Linear') != -1:\n        init.kaiming_normal_(m.weight.data, a=0, mode='fan_in')\n        m.weight.data *= scale\n        if m.bias is not None:\n            m.bias.data.zero_()\n    elif classname.find('BatchNorm2d') != -1:\n        init.constant_(m.weight.data, 1.0)\n        init.constant_(m.bias.data, 0.0)\n\n\ndef weights_init_orthogonal(m):\n    classname = m.__class__.__name__\n    if classname.find('Conv') != -1:\n        init.orthogonal_(m.weight.data, gain=1)\n        if m.bias is not None:\n            m.bias.data.zero_()\n    elif classname.find('Linear') != -1:\n        init.orthogonal_(m.weight.data, gain=1)\n        if m.bias is not None:\n            m.bias.data.zero_()\n    elif classname.find('BatchNorm2d') != -1:\n        init.constant_(m.weight.data, 1.0)\n        init.constant_(m.bias.data, 0.0)\n\n\ndef init_weights(net, init_type='kaiming', scale=1, std=0.02):\n    # scale for 'kaiming', std for 'normal'.\n    logger.info('Initialization method [{:s}]'.format(init_type))\n    if init_type == 'normal':\n        weights_init_normal_ = functools.partial(weights_init_normal, std=std)\n        net.apply(weights_init_normal_)\n    elif init_type == 'kaiming':\n        weights_init_kaiming_ = functools.partial(\n            weights_init_kaiming, scale=scale)\n        net.apply(weights_init_kaiming_)\n    elif init_type == 'orthogonal':\n        net.apply(weights_init_orthogonal)\n    else:\n        raise NotImplementedError(\n            'initialization method [{:s}] not implemented'.format(init_type))\n\n\n####################\n# define network\n####################\n\n\n# Generator\ndef define_G(opt):\n    model_opt = opt['model']\n    if model_opt['which_model_G'] == 'ddpm':\n        from .ddpm_modules import diffusion, unet\n    elif model_opt['which_model_G'] == 'sr3':\n        from .sr3_modules import diffusion, unet\n    if ('norm_groups' not in model_opt['unet']) or model_opt['unet']['norm_groups'] is None:\n        model_opt['unet']['norm_groups']=32\n    model = unet.UNet(\n        in_channel=model_opt['unet']['in_channel'],\n        out_channel=model_opt['unet']['out_channel'],\n        norm_groups=model_opt['unet']['norm_groups'],\n        inner_channel=model_opt['unet']['inner_channel'],\n        channel_mults=model_opt['unet']['channel_multiplier'],\n        attn_res=model_opt['unet']['attn_res'],\n        res_blocks=model_opt['unet']['res_blocks'],\n        dropout=model_opt['unet']['dropout'],\n        image_size=model_opt['diffusion']['image_size']\n    )\n    netG = diffusion.GaussianDiffusion(\n        model,\n        image_size=model_opt['diffusion']['image_size'],\n        channels=model_opt['diffusion']['channels'],\n        loss_type='l1',    # L1 or L2\n        conditional=model_opt['diffusion']['conditional'],\n        schedule_opt=model_opt['beta_schedule']['train']\n    )\n    if opt['phase'] == 'train':\n        # init_weights(netG, init_type='kaiming', scale=0.1)\n        init_weights(netG, init_type='orthogonal')\n    if opt['gpu_ids'] and opt['distributed']:\n        assert torch.cuda.is_available()\n        netG = nn.DataParallel(netG)\n    return netG\n",
    "third_party/dmahr/model/sr3_modules/diffusion.py": "import math\nimport torch\nfrom torch import device, nn, einsum\nimport torch.nn.functional as F\nfrom inspect import isfunction\nfrom functools import partial\nimport numpy as np\nfrom tqdm import tqdm\n\n\ndef _warmup_beta(linear_start, linear_end, n_timestep, warmup_frac):\n    betas = linear_end * np.ones(n_timestep, dtype=np.float64)\n    warmup_time = int(n_timestep * warmup_frac)\n    betas[:warmup_time] = np.linspace(\n        linear_start, linear_end, warmup_time, dtype=np.float64)\n    return betas\n\n\ndef make_beta_schedule(schedule, n_timestep, linear_start=1e-4, linear_end=2e-2, cosine_s=8e-3):\n    if schedule == 'quad':\n        betas = np.linspace(linear_start ** 0.5, linear_end ** 0.5,\n                            n_timestep, dtype=np.float64) ** 2\n    elif schedule == 'linear':\n        betas = np.linspace(linear_start, linear_end,\n                            n_timestep, dtype=np.float64)\n    elif schedule == 'warmup10':\n        betas = _warmup_beta(linear_start, linear_end,\n                             n_timestep, 0.1)\n    elif schedule == 'warmup50':\n        betas = _warmup_beta(linear_start, linear_end,\n                             n_timestep, 0.5)\n    elif schedule == 'const':\n        betas = linear_end * np.ones(n_timestep, dtype=np.float64)\n    elif schedule == 'jsd':  # 1/T, 1/(T-1), 1/(T-2), ..., 1\n        betas = 1. / np.linspace(n_timestep,\n                                 1, n_timestep, dtype=np.float64)\n    elif schedule == \"cosine\":\n        timesteps = (\n            torch.arange(n_timestep + 1, dtype=torch.float64) /\n            n_timestep + cosine_s\n        )\n        alphas = timesteps / (1 + cosine_s) * math.pi / 2\n        alphas = torch.cos(alphas).pow(2)\n        alphas = alphas / alphas[0]\n        betas = 1 - alphas[1:] / alphas[:-1]\n        betas = betas.clamp(max=0.999)\n    else:\n        raise NotImplementedError(schedule)\n    return betas\n\n\n# gaussian diffusion trainer class\n\ndef exists(x):\n    return x is not None\n\n\ndef default(val, d):\n    if exists(val):\n        return val\n    return d() if isfunction(d) else d\n\n\nclass GaussianDiffusion(nn.Module):\n    def __init__(\n        self,\n        denoise_fn,\n        image_size,\n        channels=3,\n        loss_type='l1',\n        conditional=True,\n        schedule_opt=None\n    ):\n        super().__init__()\n        self.channels = channels\n        self.image_size = image_size\n        self.denoise_fn = denoise_fn\n        self.loss_type = loss_type\n        self.conditional = conditional\n        if schedule_opt is not None:\n            pass\n            # self.set_new_noise_schedule(schedule_opt)\n\n    def set_loss(self, device):\n        if self.loss_type == 'l1':\n            self.loss_func = nn.L1Loss(reduction='sum').to(device)\n        elif self.loss_type == 'l2':\n            self.loss_func = nn.MSELoss(reduction='sum').to(device)\n        else:\n            raise NotImplementedError()\n\n    def set_new_noise_schedule(self, schedule_opt, device):\n        to_torch = partial(torch.tensor, dtype=torch.float32, device=device)\n\n        betas = make_beta_schedule(\n            schedule=schedule_opt['schedule'],\n            n_timestep=schedule_opt['n_timestep'],\n            linear_start=schedule_opt['linear_start'],\n            linear_end=schedule_opt['linear_end'])\n        betas = betas.detach().cpu().numpy() if isinstance(\n            betas, torch.Tensor) else betas\n        alphas = 1. - betas\n        alphas_cumprod = np.cumprod(alphas, axis=0)\n        alphas_cumprod_prev = np.append(1., alphas_cumprod[:-1])\n        self.sqrt_alphas_cumprod_prev = np.sqrt(\n            np.append(1., alphas_cumprod))\n\n        timesteps, = betas.shape\n        self.num_timesteps = int(timesteps)\n        self.register_buffer('betas', to_torch(betas))\n        self.register_buffer('alphas_cumprod', to_torch(alphas_cumprod))\n        self.register_buffer('alphas_cumprod_prev',\n                             to_torch(alphas_cumprod_prev))\n\n        # calculations for diffusion q(x_t | x_{t-1}) and others\n        self.register_buffer('sqrt_alphas_cumprod',\n                             to_torch(np.sqrt(alphas_cumprod)))\n        self.register_buffer('sqrt_one_minus_alphas_cumprod',\n                             to_torch(np.sqrt(1. - alphas_cumprod)))\n        self.register_buffer('log_one_minus_alphas_cumprod',\n                             to_torch(np.log(1. - alphas_cumprod)))\n        self.register_buffer('sqrt_recip_alphas_cumprod',\n                             to_torch(np.sqrt(1. / alphas_cumprod)))\n        self.register_buffer('sqrt_recipm1_alphas_cumprod',\n                             to_torch(np.sqrt(1. / alphas_cumprod - 1)))\n\n        # calculations for posterior q(x_{t-1} | x_t, x_0)\n        posterior_variance = betas * \\\n            (1. - alphas_cumprod_prev) / (1. - alphas_cumprod)\n        # above: equal to 1. / (1. / (1. - alpha_cumprod_tm1) + alpha_t / beta_t)\n        self.register_buffer('posterior_variance',\n                             to_torch(posterior_variance))\n        # below: log calculation clipped because the posterior variance is 0 at the beginning of the diffusion chain\n        self.register_buffer('posterior_log_variance_clipped', to_torch(\n            np.log(np.maximum(posterior_variance, 1e-20))))\n        self.register_buffer('posterior_mean_coef1', to_torch(\n            betas * np.sqrt(alphas_cumprod_prev) / (1. - alphas_cumprod)))\n        self.register_buffer('posterior_mean_coef2', to_torch(\n            (1. - alphas_cumprod_prev) * np.sqrt(alphas) / (1. - alphas_cumprod)))\n\n    def predict_start_from_noise(self, x_t, t, noise):\n        return self.sqrt_recip_alphas_cumprod[t] * x_t - \\\n            self.sqrt_recipm1_alphas_cumprod[t] * noise\n\n    def q_posterior(self, x_start, x_t, t):\n        posterior_mean = self.posterior_mean_coef1[t] * \\\n            x_start + self.posterior_mean_coef2[t] * x_t\n        posterior_log_variance_clipped = self.posterior_log_variance_clipped[t]\n        return posterior_mean, posterior_log_variance_clipped\n\n    def p_mean_variance(self, x, t, clip_denoised: bool, condition_x=None):\n        batch_size = x.shape[0]\n        noise_level = torch.FloatTensor(\n            [self.sqrt_alphas_cumprod_prev[t+1]]).repeat(batch_size, 1).to(x.device)\n        if condition_x is not None:\n            x_recon = self.predict_start_from_noise(\n                x, t=t, noise=self.denoise_fn(torch.cat([condition_x, x], dim=1), noise_level))\n        else:\n            x_recon = self.predict_start_from_noise(\n                x, t=t, noise=self.denoise_fn(x, noise_level))\n\n        if clip_denoised:\n            x_recon.clamp_(-1., 1.)\n\n        model_mean, posterior_log_variance = self.q_posterior(\n            x_start=x_recon, x_t=x, t=t)\n        return model_mean, posterior_log_variance\n\n    @torch.no_grad()\n    def p_sample(self, x, t, clip_denoised=True, condition_x=None):\n        model_mean, model_log_variance = self.p_mean_variance(\n            x=x, t=t, clip_denoised=clip_denoised, condition_x=condition_x)\n        noise = torch.randn_like(x) if t > 0 else torch.zeros_like(x)\n        return model_mean + noise * (0.5 * model_log_variance).exp()\n\n    @torch.no_grad()\n    def p_sample_loop(self, x_in, continous=False):\n        device = self.betas.device\n        sample_inter = (1 | (self.num_timesteps//10))\n        if not self.conditional:\n            shape = x_in\n            img = torch.randn(shape, device=device)\n            ret_img = img\n            for i in tqdm(reversed(range(0, self.num_timesteps)), desc='sampling loop time step', total=self.num_timesteps):\n                img = self.p_sample(img, i)\n                if i % sample_inter == 0:\n                    ret_img = torch.cat([ret_img, img], dim=0)\n        else:\n            x = x_in\n            shape = x.shape\n            img = torch.randn(shape, device=device)\n            ret_img = x\n            for i in tqdm(reversed(range(0, self.num_timesteps)), desc='sampling loop time step', total=self.num_timesteps):\n                img = self.p_sample(img, i, condition_x=x)\n                if i % sample_inter == 0:\n                    ret_img = torch.cat([ret_img, img], dim=0)\n        if continous:\n            return ret_img\n        else:\n            return ret_img[-1]\n\n    @torch.no_grad()\n    def sample(self, batch_size=1, continous=False):\n        image_size = self.image_size\n        channels = self.channels\n        return self.p_sample_loop((batch_size, channels, image_size, image_size), continous)\n\n    @torch.no_grad()\n    def super_resolution(self, x_in, continous=False):\n        return self.p_sample_loop(x_in, continous)\n\n    def q_sample(self, x_start, continuous_sqrt_alpha_cumprod, noise=None):\n        noise = default(noise, lambda: torch.randn_like(x_start))\n\n        # random gama\n        return (\n            continuous_sqrt_alpha_cumprod * x_start +\n            (1 - continuous_sqrt_alpha_cumprod**2).sqrt() * noise\n        )\n\n    def p_losses(self, x_in, noise=None):\n        x_start = x_in['HR']\n        [b, c, h, w] = x_start.shape\n        t = np.random.randint(1, self.num_timesteps + 1)\n        continuous_sqrt_alpha_cumprod = torch.FloatTensor(\n            np.random.uniform(\n                self.sqrt_alphas_cumprod_prev[t-1],\n                self.sqrt_alphas_cumprod_prev[t],\n                size=b\n            )\n        ).to(x_start.device)\n        continuous_sqrt_alpha_cumprod = continuous_sqrt_alpha_cumprod.view(\n            b, -1)\n\n        noise = default(noise, lambda: torch.randn_like(x_start))\n        x_noisy = self.q_sample(\n            x_start=x_start, continuous_sqrt_alpha_cumprod=continuous_sqrt_alpha_cumprod.view(-1, 1, 1, 1), noise=noise)\n\n        if not self.conditional:\n            x_recon = self.denoise_fn(x_noisy, continuous_sqrt_alpha_cumprod)\n        else:\n            x_recon = self.denoise_fn(\n                torch.cat([x_in['SR'], x_noisy], dim=1), continuous_sqrt_alpha_cumprod)\n\n        loss = self.loss_func(noise, x_recon)\n        return loss\n\n    def forward(self, x, *args, **kwargs):\n        return self.p_losses(x, *args, **kwargs)\n",
    "third_party/dmahr/model/sr3_modules/unet.py": "import math\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom inspect import isfunction\n\n\ndef exists(x):\n    return x is not None\n\n\ndef default(val, d):\n    if exists(val):\n        return val\n    return d() if isfunction(d) else d\n\n# PositionalEncoding Source\uff1a https://github.com/lmnt-com/wavegrad/blob/master/src/wavegrad/model.py\nclass PositionalEncoding(nn.Module):\n    def __init__(self, dim):\n        super().__init__()\n        self.dim = dim\n\n    def forward(self, noise_level):\n        count = self.dim // 2\n        step = torch.arange(count, dtype=noise_level.dtype,\n                            device=noise_level.device) / count\n        encoding = noise_level.unsqueeze(\n            1) * torch.exp(-math.log(1e4) * step.unsqueeze(0))\n        encoding = torch.cat(\n            [torch.sin(encoding), torch.cos(encoding)], dim=-1)\n        return encoding\n\n\nclass FeatureWiseAffine(nn.Module):\n    def __init__(self, in_channels, out_channels, use_affine_level=False):\n        super(FeatureWiseAffine, self).__init__()\n        self.use_affine_level = use_affine_level\n        self.noise_func = nn.Sequential(\n            nn.Linear(in_channels, out_channels*(1+self.use_affine_level))\n        )\n\n    def forward(self, x, noise_embed):\n        batch = x.shape[0]\n        if self.use_affine_level:\n            gamma, beta = self.noise_func(noise_embed).view(\n                batch, -1, 1, 1).chunk(2, dim=1)\n            x = (1 + gamma) * x + beta\n        else:\n            x = x + self.noise_func(noise_embed).view(batch, -1, 1, 1)\n        return x\n\n\nclass Swish(nn.Module):\n    def forward(self, x):\n        return x * torch.sigmoid(x)\n\n\nclass Upsample(nn.Module):\n    def __init__(self, dim):\n        super().__init__()\n        self.up = nn.Upsample(scale_factor=2, mode=\"nearest\")\n        self.conv = nn.Conv2d(dim, dim, 3, padding=1)\n\n    def forward(self, x):\n        return self.conv(self.up(x))\n\n\nclass Downsample(nn.Module):\n    def __init__(self, dim):\n        super().__init__()\n        self.conv = nn.Conv2d(dim, dim, 3, 2, 1)\n\n    def forward(self, x):\n        return self.conv(x)\n\n\n# building block modules\n\n\nclass Block(nn.Module):\n    def __init__(self, dim, dim_out, groups=32, dropout=0):\n        super().__init__()\n        self.block = nn.Sequential(\n            nn.GroupNorm(groups, dim),\n            Swish(),\n            nn.Dropout(dropout) if dropout != 0 else nn.Identity(),\n            nn.Conv2d(dim, dim_out, 3, padding=1)\n        )\n\n    def forward(self, x):\n        return self.block(x)\n\n\nclass ResnetBlock(nn.Module):\n    def __init__(self, dim, dim_out, noise_level_emb_dim=None, dropout=0, use_affine_level=False, norm_groups=32):\n        super().__init__()\n        self.noise_func = FeatureWiseAffine(\n            noise_level_emb_dim, dim_out, use_affine_level)\n\n        self.block1 = Block(dim, dim_out, groups=norm_groups)\n        self.block2 = Block(dim_out, dim_out, groups=norm_groups, dropout=dropout)\n        self.res_conv = nn.Conv2d(\n            dim, dim_out, 1) if dim != dim_out else nn.Identity()\n\n    def forward(self, x, time_emb):\n        b, c, h, w = x.shape\n        h = self.block1(x)\n        h = self.noise_func(h, time_emb)\n        h = self.block2(h)\n        return h + self.res_conv(x)\n\n\nclass SelfAttention(nn.Module):\n    def __init__(self, in_channel, n_head=1, norm_groups=32):\n        super().__init__()\n\n        self.n_head = n_head\n\n        self.norm = nn.GroupNorm(norm_groups, in_channel)\n        self.qkv = nn.Conv2d(in_channel, in_channel * 3, 1, bias=False)\n        self.out = nn.Conv2d(in_channel, in_channel, 1)\n\n    def forward(self, input):\n        batch, channel, height, width = input.shape\n        n_head = self.n_head\n        head_dim = channel // n_head\n\n        norm = self.norm(input)\n        qkv = self.qkv(norm).view(batch, n_head, head_dim * 3, height, width)\n        query, key, value = qkv.chunk(3, dim=2)  # bhdyx\n\n        attn = torch.einsum(\n            \"bnchw, bncyx -> bnhwyx\", query, key\n        ).contiguous() / math.sqrt(channel)\n        attn = attn.view(batch, n_head, height, width, -1)\n        attn = torch.softmax(attn, -1)\n        attn = attn.view(batch, n_head, height, width, height, width)\n\n        out = torch.einsum(\"bnhwyx, bncyx -> bnchw\", attn, value).contiguous()\n        out = self.out(out.view(batch, channel, height, width))\n\n        return out + input\n\n\nclass ResnetBlocWithAttn(nn.Module):\n    def __init__(self, dim, dim_out, *, noise_level_emb_dim=None, norm_groups=32, dropout=0, with_attn=False):\n        super().__init__()\n        self.with_attn = with_attn\n        self.res_block = ResnetBlock(\n            dim, dim_out, noise_level_emb_dim, norm_groups=norm_groups, dropout=dropout)\n        if with_attn:\n            self.attn = SelfAttention(dim_out, norm_groups=norm_groups)\n\n    def forward(self, x, time_emb):\n        x = self.res_block(x, time_emb)\n        if(self.with_attn):\n            x = self.attn(x)\n        return x\n\n\nclass UNet(nn.Module):\n    def __init__(\n        self,\n        in_channel=6,\n        out_channel=3,\n        inner_channel=32,\n        norm_groups=32,\n        channel_mults=(1, 2, 4, 8, 8),\n        attn_res=(8),\n        res_blocks=3,\n        dropout=0,\n        with_noise_level_emb=True,\n        image_size=128\n    ):\n        super().__init__()\n\n        if with_noise_level_emb:\n            noise_level_channel = inner_channel\n            self.noise_level_mlp = nn.Sequential(\n                PositionalEncoding(inner_channel),\n                nn.Linear(inner_channel, inner_channel * 4),\n                Swish(),\n                nn.Linear(inner_channel * 4, inner_channel)\n            )\n        else:\n            noise_level_channel = None\n            self.noise_level_mlp = None\n\n        num_mults = len(channel_mults)\n        pre_channel = inner_channel\n        feat_channels = [pre_channel]\n        now_res = image_size\n        downs = [nn.Conv2d(in_channel, inner_channel,\n                           kernel_size=3, padding=1)]\n        for ind in range(num_mults):\n            is_last = (ind == num_mults - 1)\n            use_attn = (now_res in attn_res)\n            channel_mult = inner_channel * channel_mults[ind]\n            for _ in range(0, res_blocks):\n                downs.append(ResnetBlocWithAttn(\n                    pre_channel, channel_mult, noise_level_emb_dim=noise_level_channel, norm_groups=norm_groups, dropout=dropout, with_attn=use_attn))\n                feat_channels.append(channel_mult)\n                pre_channel = channel_mult\n            if not is_last:\n                downs.append(Downsample(pre_channel))\n                feat_channels.append(pre_channel)\n                now_res = now_res//2\n        self.downs = nn.ModuleList(downs)\n\n        self.mid = nn.ModuleList([\n            ResnetBlocWithAttn(pre_channel, pre_channel, noise_level_emb_dim=noise_level_channel, norm_groups=norm_groups,\n                               dropout=dropout, with_attn=True),\n            ResnetBlocWithAttn(pre_channel, pre_channel, noise_level_emb_dim=noise_level_channel, norm_groups=norm_groups,\n                               dropout=dropout, with_attn=False)\n        ])\n\n        ups = []\n        for ind in reversed(range(num_mults)):\n            is_last = (ind < 1)\n            use_attn = (now_res in attn_res)\n            channel_mult = inner_channel * channel_mults[ind]\n            for _ in range(0, res_blocks+1):\n                ups.append(ResnetBlocWithAttn(\n                    pre_channel+feat_channels.pop(), channel_mult, noise_level_emb_dim=noise_level_channel, norm_groups=norm_groups,\n                        dropout=dropout, with_attn=use_attn))\n                pre_channel = channel_mult\n            if not is_last:\n                ups.append(Upsample(pre_channel))\n                now_res = now_res*2\n\n        self.ups = nn.ModuleList(ups)\n\n        self.final_conv = Block(pre_channel, default(out_channel, in_channel), groups=norm_groups)\n\n    def forward(self, x, time):\n        t = self.noise_level_mlp(time) if exists(\n            self.noise_level_mlp) else None\n\n        feats = []\n        for layer in self.downs:\n            if isinstance(layer, ResnetBlocWithAttn):\n                x = layer(x, t)\n            else:\n                x = layer(x)\n            feats.append(x)\n\n        for layer in self.mid:\n            if isinstance(layer, ResnetBlocWithAttn):\n                x = layer(x, t)\n            else:\n                x = layer(x)\n\n        for layer in self.ups:\n            if isinstance(layer, ResnetBlocWithAttn):\n                x = layer(torch.cat((x, feats.pop()), dim=1), t)\n            else:\n                x = layer(x)\n\n        return self.final_conv(x)\n",
    "third_party/dmahr/model/sr3_modules/__init__.py": "",
    "configs/dmahr.json": "{\n  \"version\": \"1.0.0\",\n  \"upstream_repo\": \"https://github.com/AnasHXH/DM-AHR\",\n  \"upstream_commit\": \"0aee1bbabf788a59be5e1eecddc20f533b5006d8\",\n  \"native_resolution\": 128,\n  \"condition_resolution\": 128,\n  \"num_timesteps\": 2000,\n  \"beta_schedule\": {\"schedule\": \"linear\", \"linear_start\": 1e-06, \"linear_end\": 0.01},\n  \"unet\": {\n    \"in_channel\": 6,\n    \"out_channel\": 3,\n    \"inner_channel\": 64,\n    \"channel_multiplier\": [1, 2, 4, 8, 8],\n    \"attn_res\": [16],\n    \"res_blocks\": 2,\n    \"dropout\": 0.2\n  },\n  \"train\": {\n    \"iterations\": 20000,\n    \"batch_size\": 4,\n    \"lr\": 0.0001,\n    \"optimizer\": \"adam\",\n    \"checkpoint_every\": 1000,\n    \"val_every\": 1000,\n    \"snapshot_every\": 5000,\n    \"overfit_pairs\": 12\n  },\n  \"sampling\": {\n    \"batch_size\": 32,\n    \"num_timesteps\": 2000,\n    \"reduced_steps_ablation\": null\n  }\n}\n\n",
    "configs/dullrazor.json": "{\n  \"config_hash\": \"0af616f50f77e1c57c4fff1e1f6bb8c2ab99100052c703d937bccbbf109a9d70\",\n  \"frozen\": true,\n  \"inpaint_method\": \"telea\",\n  \"inpaint_radius\": 3,\n  \"mask_dilate_iterations\": 1,\n  \"mask_dilate_kernel\": 3,\n  \"version\": \"1.0.0\"\n}\n",
    "configs/hair_screen.json": "{\n  \"blackhat_threshold\": 18,\n  \"config_hash\": \"e232efe4705133a175ec50d5147954a6545456d469b6353fb42d348387704273\",\n  \"coverage_threshold\": 0.01,\n  \"dilate_iterations\": 2,\n  \"dilate_kernel\": 5,\n  \"frozen\": true,\n  \"kernel_length\": 17,\n  \"light_hair\": true,\n  \"min_component_area\": 40,\n  \"orientations\": 12,\n  \"tophat_threshold\": 22,\n  \"version\": \"1.0.0\"\n}\n",
    "configs/protocol.json": "{\n  \"study\": \"DM-AHR-ViT\",\n  \"version\": \"1.0.0\",\n  \"max_spend_usd\": 250.0,\n  \"seeds\": [42, 123, 2026],\n  \"split_seed\": 42,\n  \"split_fractions\": {\"train\": 0.70, \"val\": 0.15, \"test\": 0.15},\n  \"classes\": [\"akiec\", \"bcc\", \"bkl\", \"df\", \"mel\", \"nv\", \"vasc\"],\n  \"image_size\": 224,\n  \"isic_collection_id\": 212,\n  \"expected_c0_total\": 10015,\n  \"expected_collection_total\": 11720,\n  \"near_duplicate_hamming\": 6,\n  \"arms\": [\"A\", \"B\", \"C\", \"Astar\"],\n  \"primary_endpoints\": [\n    \"test_macro_f1\",\n    \"paired_psnr\",\n    \"paired_ssim\",\n    \"x1_gradcam_overlap20\",\n    \"x1_rollout_overlap20\"\n  ],\n  \"planned_comparisons\": [[\"A\", \"B\"], [\"A\", \"C\"], [\"B\", \"C\"]],\n  \"bootstrap_replicates\": 10000,\n  \"bootstrap_seed\": 20260101,\n  \"research_only_statement\": \"Research prototype only. Not clinically validated and not intended for diagnosis or patient care.\"\n}\n\n",
    "configs/sampling.json": "{\n  \"version\": \"1.0.0\",\n  \"psnr_data_range\": 1.0,\n  \"ssim_data_range\": 1.0,\n  \"ssim_win_size\": 7,\n  \"xai_top_fraction\": 0.2,\n  \"xai_pointing_tolerance_px\": 0,\n  \"chance_baselines\": [\"uniform\", \"centered_gaussian\"]\n}\n\n",
    "configs/stale_phrases.json": "{\n  \"comment\": \"Phrases that must not appear in the current deliverables. Each pattern is a case-insensitive regex. History belongs in decision_log.md, which is not scanned.\",\n  \"files\": [\n    \"reports/client_summary.md\",\n    \"reports/final_research_report.md\",\n    \"reports/stageE_results.md\",\n    \"reports/stageC_results.md\",\n    \"reports/xai_corrected_results.md\"\n  ],\n  \"docx_dir\": \"artifacts/reports\",\n  \"patterns\": [\n    \"no arm beats a centre prior\",\n    \"below the centred-Gaussian baseline\",\n    \"centred[- ]Gaussian baseline[^.\\\\n]{0,20}0\\\\.64\",\n    \"share of the top 20% attention inside the lesion\",\n    \"top 20% most salient pixels that fall inside the mask\",\n    \"\\\\bd_?z\\\\b\",\n    \"effect_size_dz\",\n    \"over-?smooth(ed|ing)? (output|images|diffusion)\",\n    \"Grad-CAM[^.\\\\n]{0,60}(visual )?faithfulness has been validated\",\n    \"attributable to the hair-removal condition alone\",\n    \"classical removal hurt it\",\n    \"drives both its quality scores and its classification deficit\",\n    \"only manipulated variable\",\n    \"larger share of the 128 px\"\n  ],\n  \"paper_files\": [\"artifacts/reports/Research_Paper.docx\"],\n  \"paper_patterns\": [\n    \"\\\\bColab\\\\b\",\n    \"\\\\bthe client\\\\b\",\n    \"expected results\",\n    \"(is|are) expected to (achieve|show|produce|preserve)\",\n    \"rotations? of up to [\u00b1+]/?-?\\\\s?90\",\n    \"improves not only classification\",\n    \"(?<!knowledge, )no prior study has\",\n    \"to be agreed by\"\n  ]\n}\n",
    "configs/vit.json": "{\n  \"version\": \"1.0.0\",\n  \"model\": \"vit_b_16\",\n  \"weights\": \"IMAGENET1K_V1\",\n  \"image_size\": 224,\n  \"batch_size\": 32,\n  \"grad_accum_steps\": 1,\n  \"epochs\": 100,\n  \"lr\": 3e-05,\n  \"weight_decay\": 0.05,\n  \"warmup_epochs\": 5,\n  \"scheduler\": \"cosine\",\n  \"label_smoothing\": 0.0,\n  \"class_weights\": \"train_inverse_frequency\",\n  \"early_stopping_patience\": 10,\n  \"early_stopping_metric\": \"val_macro_f1\",\n  \"grad_clip\": 1.0,\n  \"precision\": \"bf16\",\n  \"num_workers\": 8,\n  \"augmentation\": {\n    \"random_resized_crop_scale\": [0.75, 1.0],\n    \"random_resized_crop_ratio\": [0.9, 1.1111111111111112],\n    \"horizontal_flip\": 0.5,\n    \"vertical_flip\": 0.5,\n    \"rotation_degrees\": 15,\n    \"color_jitter\": [0.15, 0.15, 0.1, 0.02]\n  }\n}\n\n",
    "third_party/dmahr/config/akiec.json": "{\n    \"name\": \"sr_ffhq\",\n    \"phase\": \"train\", // train or val\n    \"gpu_ids\": [\n        0\n    ],\n    \"path\": { //set the path\n        \"log\": \"logs\",\n        \"tb_logger\": \"tb_logger\",\n        \"results\": \"results\",\n        \"checkpoint\": \"checkpoint\",\n        \"resume_state\": \"experiments/sr_ffhq_231220_121203/checkpoint/I725000_E1083\"\n        // \"resume_state\": \"experiments/sr_ffhq_210806_204158/checkpoint/I640000_E37\" //pretrain model or training state\n    },\n    \"datasets\": {\n        \"train\": {\n            \"name\": \"FFHQ\",\n            \"mode\": \"HR\", // whether need LR img\n            \"dataroot\": \"hair_removal/akiec_16_128\",\n            \"datatype\": \"img\", //lmdb or img, path of img files\n            \"l_resolution\": 16, // low resolution need to super_resolution\n            \"r_resolution\": 128, // high resolution\n            \"batch_size\": 4,\n            \"num_workers\": 1,\n            \"use_shuffle\": true,\n            \"data_len\": -1 // -1 represents all data used in train\n        },\n        \"val\": {\n            \"name\": \"CelebaHQ\",\n            \"mode\": \"LRHR\",\n            \"dataroot\": \"hair_removal/akiec_16_128\",\n            \"datatype\": \"img\", //lmdb or img, path of img files\n            \"l_resolution\": 16,\n            \"r_resolution\": 128,\n            \"data_len\": -1 // data length in validation \n        }\n    },\n    \"model\": {\n        \"which_model_G\": \"sr3\", // use the ddpm or sr3 network structure\n        \"finetune_norm\": false,\n        \"unet\": {\n            \"in_channel\": 6,\n            \"out_channel\": 3,\n            \"inner_channel\": 64,\n            \"channel_multiplier\": [\n                1,\n                2,\n                4,\n                8,\n                8\n            ],\n            \"attn_res\": [\n                16\n            ],\n            \"res_blocks\": 2,\n            \"dropout\": 0.2\n        },\n        \"beta_schedule\": { // use munual beta_schedule for acceleration\n            \"train\": {\n                \"schedule\": \"linear\",\n                \"n_timestep\": 2000,\n                \"linear_start\": 1e-6,\n                \"linear_end\": 1e-2\n            },\n            \"val\": {\n                \"schedule\": \"linear\",\n                \"n_timestep\": 2000,\n                \"linear_start\": 1e-6,\n                \"linear_end\": 1e-2\n            }\n        },\n        \"diffusion\": {\n            \"image_size\": 128,\n            \"channels\": 3, //sample channel\n            \"conditional\": true // unconditional generation or unconditional generation(super_resolution)\n        }\n    },\n    \"train\": {\n        \"n_iter\": 660000,\n        \"val_freq\": 1e4,\n        \"save_checkpoint_freq\": 1e4,\n        \"print_freq\": 200,\n        \"optimizer\": {\n            \"type\": \"adam\",\n            \"lr\": 1e-4\n        },\n        \"ema_scheduler\": { // not used now\n            \"step_start_ema\": 5000,\n            \"update_ema_every\": 1,\n            \"ema_decay\": 0.9999\n        }\n    },\n    \"wandb\": {\n        \"project\": \"sr_ffhq\"\n    }\n}\n"
}
INLINE = BASE / "inline"
for _rel, _src in EMBEDDED.items():
    # dm_ahr_vit derives REPO_ROOT from its own depth (parents[2] of paths.py),
    # so the package lives in INLINE/ while configs and third_party sit at BASE/.
    _p = (BASE if _rel.startswith(("configs/", "third_party/")) else INLINE) / _rel
    _p.parent.mkdir(parents=True, exist_ok=True)
    if not _p.exists() or _p.read_text(encoding="utf-8") != _src:
        _p.write_text(_src, encoding="utf-8")
sys.path.insert(0, str(INLINE))
import dm_ahr_vit
from dm_ahr_vit.config import load_config, load_frozen, config_hash, config_hash_file, save_json
from dm_ahr_vit.paths import (ensure_dirs, data_root, artifact_dir, raw_dir, isic_images_dir,
                              manifest_dir, ham_root, ham_masks_dir, processed_dir)
print("embedded files:", len(EMBEDDED), "| package:", dm_ahr_vit.__file__)


## 3. Dependencies and GPU


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch==2.5.1", "torchvision==0.20.1", "numpy==1.26.4", "pandas==2.2.3",
                "pillow==11.0.0", "opencv-python-headless==4.10.0.84",
                "scikit-image==0.24.0", "scipy==1.14.1", "tqdm==4.67.1",
                "requests==2.32.3", "PyYAML==6.0.2"], check=True)
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
print("torch", torch.__version__, "| device", name)
if "T4" in name:
    print("WARNING: T4 is supported but 3-5x slower than the A100 used for the recorded run.")
assert torch.cuda.is_available(), "No GPU. Colab: Runtime -> Change runtime type -> GPU."


## 4. Phase 1 — download ISIC collection 212 and HAM10000 metadata/masks

Public data, no account. Cached on Drive, so a re-run does not download again.


In [ ]:
from dm_ahr_vit.data import isic, ham10000
ensure_dirs()

def fetch(url, dest):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return {"path": str(dest), "sha256": hashlib.sha256(dest.read_bytes()).hexdigest(), "cached": True}
    dest.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        tmp = dest.with_suffix(dest.suffix + ".part")
        with tmp.open("wb") as fh:
            for chunk in r.iter_content(1 << 20):
                fh.write(chunk)
        tmp.replace(dest)
    return {"path": str(dest), "sha256": hashlib.sha256(dest.read_bytes()).hexdigest(), "cached": False}

stats = isic.download_collection(212, isic_images_dir(),
                                 raw_dir() / "isic_212" / "records.jsonl", workers=16)
meta = fetch(ham10000.DATAVERSE_ACCESS.format(fid=ham10000.METADATA_FILE_ID),
             ham_root() / "HAM10000_metadata.tab")
masks_zip = fetch(ham10000.DATAVERSE_ACCESS.format(fid=ham10000.MASKS_FILE_ID),
                  ham_root() / "HAM10000_segmentations_lesion_tschandl.zip")
extracted = ham10000.extract_masks(Path(masks_zip["path"]), ham_masks_dir())
save_json({"isic": stats, "metadata": meta, "masks": masks_zip,
           "masks_extracted": extracted},
          artifact_dir("logs") / "download_report.json")
print("ISIC:", stats)


## 5. Phase 2 — audit, deduplicate and freeze the lesion-grouped split

Every count is recomputed from files. Confirmed near-duplicate links are merged into lesion
groups before the 70/15/15 split, and the leakage check must pass.


In [ ]:
import numpy as np, pandas as pd
from dm_ahr_vit.config import CLASSES, EXPECTED_C0_COUNTS
from dm_ahr_vit.data import manifest as manifest_mod, masks as masks_mod, splits as splits_mod

protocol = load_config("configs/protocol.json")
protocol_hash = config_hash_file("configs/protocol.json")
records_path = raw_dir() / "isic_212" / "records.jsonl"
metadata = ham10000.read_metadata(ham_root() / "HAM10000_metadata.tab")
comparison = ham10000.expected_comparison(metadata)

out_manifest = manifest_dir() / "data_manifest.csv"
n_records = len([ln for ln in records_path.read_text().splitlines() if ln.strip()])
if out_manifest.exists():
    man = pd.read_csv(out_manifest)
    if not (len(man) == n_records and man["sha256"].notna().all() and man["dhash"].notna().all()):
        man = None
else:
    man = None
if man is None:
    man, _, _ = manifest_mod.build_manifest(records_path, isic_images_dir(), metadata, manifest_dir())

c0 = man[(man["cohort"] == "C0") & (man["decode_status"] == "ok")].copy()
c0["dx"] = c0["dx"].astype(str)
c0["lesion_id"] = c0["lesion_id"].astype(str)
cand_path = manifest_dir() / "near_duplicates.csv"
if cand_path.exists():
    candidates = pd.read_csv(cand_path)
else:
    hashes = c0["dhash"].astype(np.uint64).to_numpy()
    candidates = pd.DataFrame(manifest_mod.hamming_pairs(hashes, 6, c0["image_id"].tolist()),
                              columns=["id_a", "id_b", "distance"])
    candidates.to_csv(cand_path, index=False)
conf_path = manifest_dir() / "near_duplicates_confirmed.csv"
if conf_path.exists():
    confirmed = pd.read_csv(conf_path)
else:
    confirmed = manifest_mod.confirm_near_duplicates(candidates, man,
                                                      max_mean_abs_diff=4.0, min_correlation=0.985)
    if len(confirmed):
        confirmed.to_csv(conf_path, index=False)
kept = confirmed[confirmed["confirmed"]] if len(confirmed) else confirmed
grouped = splits_mod.build_groups(c0, kept if len(kept) else candidates.iloc[0:0])
summary = splits_mod.freeze_splits(grouped, seed=int(protocol["split_seed"]),
                                   fractions=protocol["split_fractions"],
                                   out_dir=manifest_dir(), protocol_hash=protocol_hash)
split_df = pd.read_csv(manifest_dir() / "splits.csv")
verification = splits_mod.verify_splits(split_df)
x1 = masks_mod.build_x1_manifest(ham_masks_dir(), split_df, man,
                                 manifest_dir() / "mask_manifest_x1.csv")
audit = {"collection_total": int(len(man)), "c0_decoded": int(len(c0)),
         "frozen_splits": summary, "split_verification": verification,
         "near_duplicates_confirmed": int(len(kept))}
save_json(audit, artifact_dir("logs") / "data_audit.json")
print("split hash:", summary["splits_sha256"])
print("verification:", verification)
assert verification["ok"], "leakage check failed"
assert comparison["all_match"], "class counts differ from expected"


## 6. Phase 3 — hair screening, DullRazor (Arm B), A* and synthetic pairs

Thresholds are tuned on train/val only and frozen before any test use.


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
import cv2
from dm_ahr_vit.hair.screening import detect_hair, tune_coverage_threshold
from dm_ahr_vit.hair.dullrazor import apply_dullrazor
from dm_ahr_vit.data.synthetic_hair import render_hair

SYNTH_SEED = 7000
SYNTH_LIMITS = {"train": 3000, "val": 800, "test": 186}

def read_rgb(path):
    with Image.open(path) as im:
        return np.array(im.convert("RGB"))

def write_png(arr, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(arr).save(path, compress_level=3)

def tune_screen(screen_cfg, splits):
    tune_rows = splits[splits["split"].isin(["train", "val"])]
    sample = tune_rows.sample(n=min(400, len(tune_rows)), random_state=42)
    pairs = []
    for _, r in sample.iterrows():
        img = read_rgb(isic_images_dir() / f"{r['image_id']}.jpg")
        clean_cov = detect_hair(img, {**screen_cfg, "coverage_threshold": 10.0})[1]["coverage"]
        if clean_cov > 0.01:
            continue
        hairy, _, _ = render_hair(Image.fromarray(img), r["image_id"], SYNTH_SEED + 1)
        pairs.append((img, 0))
        pairs.append((np.array(hairy), 1))
    return tune_coverage_threshold(pairs, screen_cfg), {"tuning_pairs": len(pairs)}

screen_cfg = load_frozen("hair_screen", "configs/hair_screen.json")
dull_cfg = load_frozen("dullrazor", "configs/dullrazor.json")
splits = pd.read_csv(manifest_dir() / "splits.csv")
if not screen_cfg.get("frozen"):
    best, tune_info = tune_screen(screen_cfg, splits)
    screen_cfg = {**screen_cfg, "coverage_threshold": best["coverage_threshold"], "frozen": True}
    screen_cfg["config_hash"] = config_hash({k: v for k, v in screen_cfg.items() if k != "config_hash"})
    dull_cfg = {**dull_cfg, "frozen": True}
    dull_cfg["config_hash"] = config_hash({k: v for k, v in dull_cfg.items() if k != "config_hash"})
    save_json(screen_cfg, "configs/hair_screen.json")
    save_json(dull_cfg, "configs/dullrazor.json")
    print("froze hair screen:", best)
print("screen config hash:", screen_cfg["config_hash"])


In [ ]:
def process_one(row):
    iid = row["image_id"]
    out = {"image_id": iid, "split": row["split"], "dx": row["dx"],
           "group_id": row["group_id"], "screen_config_hash": screen_cfg["config_hash"]}
    try:
        img = read_rgb(isic_images_dir() / f"{iid}.jpg")
    except Exception as exc:
        out.update({"status": "fail_decode", "error": repr(exc),
                    "hair_present": None, "hair_coverage": None})
        return out
    mask, stats = detect_hair(img, screen_cfg)
    out.update({"width": img.shape[1], "height": img.shape[0],
                "hair_coverage": round(stats["coverage"], 6),
                "hair_present": stats["hair_present"], "status": "ok"})
    try:
        write_png(mask * 255, processed_dir("hair_masks") / f"{iid}.png")
        if stats["hair_present"]:
            write_png(apply_dullrazor(img, mask, dull_cfg), processed_dir("B") / f"{iid}.png")
            write_png(np.array(Image.fromarray(img).resize((128, 128), Image.Resampling.BICUBIC)),
                      processed_dir("Astar") / f"{iid}.png")
            out["arm_b_status"], out["arm_astar"] = "dullrazor", "resized_128"
        else:
            write_png(img, processed_dir("B") / f"{iid}.png")
            write_png(img, processed_dir("Astar") / f"{iid}.png")
            out["arm_b_status"] = out["arm_astar"] = "hair_absent_passthrough"
    except Exception as exc:
        out.update({"status": "fail_write", "error": repr(exc)})
    return out

records = [r for _, r in splits.iterrows()]
with ThreadPoolExecutor(max_workers=min(48, (os.cpu_count() or 4) * 2)) as ex:
    rows = list(ex.map(process_one, records))
df = pd.DataFrame(rows)
df.to_csv(manifest_dir() / "hair_screening.csv", index=False)
print("screened:", len(df), "| hair present:", int(df["hair_present"].sum()))


In [ ]:
def synth_one(row, split):
    iid = row["image_id"]
    img = read_rgb(isic_images_dir() / f"{iid}.jpg")
    hairy, hmask, params = render_hair(Image.fromarray(img), iid, SYNTH_SEED)
    base = processed_dir("synth") / split
    write_png(np.array(hairy), base / "hairy" / f"{iid}.png")
    write_png(img, base / "reference" / f"{iid}.png")
    write_png(np.array(hmask), base / "hair_mask" / f"{iid}.png")
    return {"pair_id": f"synth_{split}_{iid}", "image_id": iid, "split": split,
            "dx": row["dx"], "group_id": row["group_id"],
            "hairy_path": str(base / "hairy" / f"{iid}.png"),
            "reference_path": str(base / "reference" / f"{iid}.png"),
            "hair_mask_path": str(base / "hair_mask" / f"{iid}.png"),
            "seed": SYNTH_SEED, "n_hairs": params["n_hairs"],
            "blur_sigma": params["blur"], "source": "synthetic"}

synthetic_rows = []
hair_absent = df[~df["hair_present"]]
for split, limit in SYNTH_LIMITS.items():
    sub = [r for _, r in hair_absent[hair_absent["split"] == split].head(limit).iterrows()]
    with ThreadPoolExecutor(max_workers=min(32, (os.cpu_count() or 4) * 2)) as ex:
        synthetic_rows += list(ex.map(lambda r, s=split: synth_one(r, s), sub))
    print("synthetic", split, len(sub))
pd.DataFrame(synthetic_rows).to_csv(manifest_dir() / "synthetic_pairs.csv", index=False)
print("total synthetic pairs:", len(synthetic_rows))


## 7. Phase 4 — harvest pasted real strands (CPU, train/val only)


In [ ]:
from dm_ahr_vit.config import load_hair_screen_config
from dm_ahr_vit.data.hair_composite import HAIR_MIN_COVERAGE
from dm_ahr_vit.hair.harvest import thin_hair_mask

hcfg = load_hair_screen_config()
scr = pd.read_csv(manifest_dir() / "hair_screening.csv")
scr = scr[scr["split"].isin(["train", "val"]) & (scr["hair_coverage"] >= HAIR_MIN_COVERAGE)]
assert not (scr["split"] == "test").any(), "test split must never be harvested"
layers_dir = data_root() / "hair_layers"
layers_dir.mkdir(parents=True, exist_ok=True)
harvest_rows = []
for iid in sorted(scr["image_id"]):
    rgb = read_rgb(isic_images_dir() / f"{iid}.jpg")
    m, info = thin_hair_mask(rgb, hcfg)
    if info["hair_pixels"] > 0:
        write_png((m * 255).astype(np.uint8), layers_dir / f"{iid}.png")
    harvest_rows.append({"image_id": iid, **info})
pd.DataFrame(harvest_rows).to_csv(layers_dir / "summary.csv", index=False)
print("hair layers:", len(list(layers_dir.glob("*.png"))))


## 8. Phase 5 — train the repaired base diffusion generator (50k)

Checkpoints are written during training; an interrupted run resumes from `train_state.pt`.


In [ ]:
from dm_ahr_vit.diffusion.train import train as train_diffusion

cfg_dm = load_config("configs/dmahr.json")
pairs = pd.read_csv(manifest_dir() / "synthetic_pairs.csv")
train_pairs, val_pairs = pairs[pairs["split"] == "train"], pairs[pairs["split"] == "val"]
assert set(train_pairs["group_id"]).isdisjoint(set(val_pairs["group_id"])), "group leakage"
result_base = train_diffusion(cfg_dm, train_pairs, val_pairs,
                              artifact_dir("checkpoints", "dmahr"),
                              max_iterations=DIFF_BASE_ITERS)
save_json(result_base, artifact_dir("logs") / "dmahr_train.json")
base_ckpts = sorted((artifact_dir("checkpoints", "dmahr")).glob("gen_iter*.pth"),
                    key=lambda p: int(p.stem.split("_")[-1]))
base_ckpt = base_ckpts[-1]
print("base checkpoint:", base_ckpt)


## 9. Phase 6 — fine-tune for 20k iterations on pasted real strands


In [ ]:
from dm_ahr_vit.data.hair_composite import build_datasets

train_ds, val_ds = build_datasets(manifest_dir() / "hair_screening.csv",
                                  isic_images_dir(), layers_dir,
                                  size=int(cfg_dm["native_resolution"]))
result_ft = train_diffusion(cfg_dm, None, None,
                            artifact_dir("checkpoints", "dmahr_v2"),
                            init_from=base_ckpt, max_iterations=DIFF_FINETUNE_ITERS,
                            train_ds=train_ds, val_ds=val_ds, num_workers=8)
save_json(result_ft, artifact_dir("logs") / "dmahr_v2_train.json")
ft_ckpt = artifact_dir("checkpoints", "dmahr_v2") / "final_gen.pth"
assert ft_ckpt.exists(), f"missing {ft_ckpt}"
print("fine-tuned checkpoint:", ft_ckpt)


## 10. Phase 7 — generate Cv2 images (128 px, 50 strided DDIM steps from t0=600, FP32)

Shards skip images already written, so a disconnected session resumes.


In [ ]:
from PIL import Image as PILImage
from dm_ahr_vit.diffusion.dmahr_adapter import load_model, clean_batch, _ensure_path
from dm_ahr_vit.diffusion.train import _resolve_manifest_path

net = load_model(cfg_dm, ft_ckpt, device="cuda")

def sample_shard(shard, n_shards, pairs_split=None):
    if pairs_split:
        p = pd.read_csv(manifest_dir() / "synthetic_pairs.csv")
        p = p[p["split"] == pairs_split].sort_values("image_id").iloc[shard::n_shards]
        todo = pd.DataFrame({"image_id": p["image_id"].values,
                             "source_path": [str(_resolve_manifest_path(q)) for q in p["hairy_path"].values]})
        out_dir = processed_dir("synth") / pairs_split / "cleaned_Cv2"
    else:
        s = pd.read_csv(manifest_dir() / "hair_screening.csv")
        s = s[s["hair_present"]].sort_values("image_id").reset_index(drop=True).iloc[shard::n_shards]
        todo = pd.DataFrame({"image_id": s["image_id"].values,
                             "source_path": [str(isic_images_dir() / f"{i}.jpg") for i in s["image_id"]]})
        out_dir = processed_dir("Cv2")
    out_dir.mkdir(parents=True, exist_ok=True)
    todo = todo[~todo["image_id"].map(lambda i: (out_dir / f"{i}.png").exists())]
    bs = int(cfg_dm["sampling"]["batch_size"])
    done = 0
    for start in range(0, len(todo), bs):
        chunk = todo.iloc[start:start + bs]
        images = [PILImage.open(p).convert("RGB") for p in chunk["source_path"]]
        seed = int(hashlib.sha256(f"{shard}:{start}".encode()).hexdigest()[:8], 16)
        outs, stats = clean_batch(net, images, cfg_dm, device="cuda", seed=seed,
                                  num_timesteps=CV2_STEPS, use_bf16=False,
                                  respaced=True, start_t=CV2_START_T)
        assert stats["finite"]
        for iid, out in zip(chunk["image_id"], outs):
            tmp = out_dir / f"{iid}.png.tmp"
            out.save(tmp, format="PNG")
            tmp.replace(out_dir / f"{iid}.png")
            done += 1
    return done

total = 0
for shard in range(CV2_SHARDS):
    n = sample_shard(shard, CV2_SHARDS)
    total += n
    print("shard", shard, "processed", n)
print("Cv2 hair-present total:", len(list((data_root() / "processed" / "Cv2").glob("*.png"))))


In [ ]:
n_pairs = sample_shard(0, 1, pairs_split="test")
print("held-out pair outputs:", n_pairs)

# hair-absent images are identical in every arm
passthrough = pd.read_csv(manifest_dir() / "hair_screening.csv")
passthrough = passthrough[~passthrough["hair_present"]]
cv2_dir = processed_dir("Cv2")
for iid in passthrough["image_id"]:
    dest = cv2_dir / f"{iid}.png"
    if not dest.exists():
        shutil.copy2(isic_images_dir() / f"{iid}.jpg", dest)
print("Cv2 complete:", len(list(cv2_dir.glob("*.png"))))


## 11. Phase 8 — train the ViT-B/16 classifiers (last training phase)


In [ ]:
from dm_ahr_vit.vit.train import train_arm

cfg_vit = load_config("configs/vit.json")
splits_df = pd.read_csv(manifest_dir() / "splits.csv")
for arm in ARMS:
    for seed in SEEDS:
        run_dir = artifact_dir("checkpoints", "vit", f"{arm}_seed{seed}")
        res = train_arm(arm=arm, seed=seed, cfg=cfg_vit, splits_df=splits_df,
                        raw_images_dir=isic_images_dir(),
                        processed_root=data_root() / "processed",
                        run_dir=run_dir, device="cuda")
        print(arm, seed, "best val macro-F1", round(res["best_val_macro_f1"], 4),
              "epochs", res["epochs_run"], flush=True)


## 12. Models produced — that is the whole job


In [ ]:
ck = ARTIFACT_ROOT / "checkpoints"
diffusion = sorted((ck / "dmahr").glob("gen_iter*.pth")) + sorted((ck / "dmahr_v2").glob("*.pth"))
vit = sorted(ck.glob("vit/*/best.pt"))
print("diffusion generators:", len(diffusion))
for p in diffusion:
    print(" ", p.relative_to(ck), f"{p.stat().st_size/1e6:.1f} MB")
print("ViT checkpoints:", len(vit))
for p in vit:
    print(" ", p.relative_to(ck), f"{p.stat().st_size/1e6:.1f} MB")
print("\nAll models under:", ck)
